# PastTheEyeTest player ratings

Builds skater and goalie ratings from the files written by `nhl_xg_scraper.ipynb` and `nhl_xg_modeling.ipynb`:

- **5v5 Offense and 5v5 Defense:** each skater's isolated impact on expected goals.
- **Finishing:** goals scored above or below what the player's shots were expected to produce, together with his raw goal total, each split by game state.
- **Playmaking:** primary and secondary assists, split by game state.
- **Production:** the earlier combined score (goals and assists together), kept for reference.
- **Teammates and Competition:** the average quality of the skaters a player shares the ice with and plays against at 5v5.
- **Penalties:** minor penalties drawn minus taken, per 60 minutes.
- **Power Play and Penalty Kill:** each skater's isolated impact on expected goals with the man advantage and while shorthanded.
- **Shooting and Setup results:** goals above expected on a player's own shots, and on his teammates' shots while he is on the ice. This is what lets Impact and WAR reflect goals that were actually scored.
- **Goalies:** goals saved above average on every shot faced, adjusted for the team in front of the goalie; split by 5v5, against the power play, and low, medium and high danger; rebound control; win probability added.
- **Individualized Impact and Wins Above Replacement,** for skaters and goalies: every component converted to goals, put on the scale of real goal differences, and added up, as a rate (Impact) and as a season total against a replacement player (WAR).

**Predictive and descriptive.** The expected-goals ratings are tuned to predict next season. The results components are tuned to balance two aims at once: predicting next season, and describing the goals that were actually scored this season. `DESCRIPTIVE_WEIGHT` in section 1 sets the balance (0.5 = both equally).

**5v5 method.** A ridge regression over every 5v5 stint. Each stint contributes two rows (home attacking, away attacking). The target is the attacking team's xG per 60, weighted by the stint's length. The model estimates, at the same time, an offense and a defense effect for every player, the effect of deployment (offensive, neutral or defensive zone start), the score and home ice. Because teammates and opponents are in the same model, each player's number is what is left after accounting for who he played with and against and where his shifts started.

**Scoring method.** For Finishing, Playmaking and Production, section 8 measures from your own data how well each ingredient this season predicts the same player next season, and uses those as the weights.

**Two versions of every rating:**

- **1-year:** that season alone.
- **3-year:** that season plus the two before it, with the older seasons weighted less. The season weights are chosen by how well they predict the *following* season, so the 3-year number is the best available gauge of current value.

Run the cells top to bottom. The whole notebook takes roughly 45–70 minutes.

| Section | What it does |
|---|---|
| 1–2 | Setup, load stints, attach xG, count goals, assists and ice time by situation |
| 3 | Measure the deployment (zone start) effect |
| 4 | Set up the 5v5 regression |
| 5 | Choose the shrinkage and the season weights |
| 6 | Fit the 1-year and 3-year 5v5 ratings |
| 7 | Raw and zone-adjusted on-ice rates, percentiles |
| 8 | Finishing, Playmaking and Production: find the weights, build the 1-year and 3-year scores |
| 9 | Teammates, Competition and Penalties |
| 10 | Power Play and Penalty Kill |
| 11 | Results: goals above expected on own shots and on teammates' shots |
| 12 | Skaters: goals above average by component |
| 13 | Goalies: goals saved above expected, team-adjusted, by situation and danger; rebound control; win probability added |
| 14 | Calibration to real goals, Individualized Impact and WAR, for skaters and goalies |
| 15 | Checks: leaders and year-to-year repeatability |
| 16 | Save `player_ratings.csv`, `goalie_ratings.csv`, `team_war.csv` |
| 17 | Optional: one team's report |
| 18 | Optional: team WAR against results chart |
| 19 | Optional: WAR to next season's WAR, and Cronbach's alpha |
| 20 | Optional: impact tiers (line, pair, goalie role) and impact by season |
| 21 | Player dashboard (Streamlit) |

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import glob, time
from pathlib import Path
import numpy as np, pandas as pd
import scipy.sparse as sp
from scipy.linalg import solve
from scipy.optimize import nnls
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold

DATA_DIR = '/content/drive/MyDrive'
XG_COL = 'xg_flurry'          # flurry-adjusted xG is the right one for adding up
ZONE_WINDOW = 20              # a stint starting within this many seconds of a faceoff counts as that zone start
LAMBDAS = [1e4, 3e4, 1e5, 3e5, 1e6, 3e6]      # shrinkage strengths tried in section 5
# (weight on last season, weight on the season before that); the current season always has weight 1
WEIGHT_GRID = [(0, 0), (1, 1), (0.9, 0.7), (0.8, 0.6), (0.7, 0.5), (0.6, 0.4), (0.6, 0.3),
               (0.5, 0.25), (0.4, 0.2), (0.3, 0.1)]
MIN_MIN_1Y, MIN_MIN_3Y = 300, 600             # 5v5 minutes needed to be ranked (3-year minutes are weighted)

# Finishing, Playmaking and Production (section 8)
PROD_TARGET = 'ev'            # what Playmaking and Production are tuned to predict next season: 'ev' = the
                              # even-strength rate, 'all' = the rate in all situations
MIN_ALL_1Y, MIN_ALL_3Y = 500, 1000            # all-situations minutes needed to be ranked for these three
MIN_PROD_FIT = 400                            # minutes in both seasons for a player to be used to find the weights

# Special teams, results, Impact and WAR (sections 10-14)
LAMBDAS_ST = [3e3, 1e4, 3e4, 1e5, 3e5, 1e6]   # shrinkage strengths tried for the power-play / penalty-kill model
MIN_PP_1Y, MIN_PP_3Y = 50, 100                # power-play (or penalty-kill) minutes needed to be ranked there
DESCRIPTIVE_WEIGHT = 0.5      # how the results component is chosen in section 12: 0 = only by how well team
                              # totals predict next season, 1 = only by how well they describe this season,
                              # 0.5 = both equally
RESULT_LAMBDAS = [100, 300, 1000, 3000]       # shrinkage strengths offered for the results component
RESULTS_FORCE = None          # None = let section 12 choose; or the exact name of a row in its table to force it,
                              # for example 'shot model, lambda 300'
SHOT_LAMBDAS = [3, 10, 30, 100, 300, 1000, 3000, 10000]      # shrinkage strengths tried in section 11
SHOT_WEIGHTS = [(0, 0), (1, 1), (0.8, 0.6), (0.6, 0.4), (0.4, 0.2)]
SETUP_RATIOS = [1, 3, 10, 30, None]           # how much harder to shrink setup than shooting; None = no setup effect
MIN_CURRENT_5V5 = 200         # 5v5 minutes this season needed to be ranked in the 3-year percentiles
REPLACEMENT_F, REPLACEMENT_D = 13, 7          # a team's regulars by ice time; everyone below is replacement level

# Goalies and calibration (sections 13-14)
G_LAMBDAS = [100, 300, 1000, 3000, 10000, 30000]     # shrinkage strengths tried for the goalie model
G_TEAM_RATIOS = [0.3, 1, 3, 10]               # how hard the team effect is shrunk relative to the goalie effect
ADJUST_FOR_SHOOTERS = True    # credit goalies for facing better finishers (uses the shooting ratings of section 11)
GOALIE_1Y = 'shooter-adjusted GSAx'          # what 1-year goalie WAR is built on: the exact name of a row in section 13's
                              # table, or None for the shrunk team-adjusted model
GOALIE_3Y = None              # the same for 3-year goalie WAR; None = the team-adjusted model, tuned to predict
REPLACEMENT_G = 2             # a team's regular goalies by shots faced; everyone below is replacement level
MIN_G_1Y, MIN_G_3Y, MIN_G_CURRENT = 600, 1200, 300   # unblocked shots faced needed for a goalie to be ranked
LATEST_SEASON_SHARE = None    # how much of the latest season has been played. None = work it out from the games
                              # (1 = finished); or set a share yourself, such as 0.25
CALIBRATE = True              # stretch player values so team totals match real goal differences (section 14)
REBOUNDS_IN_WAR = False       # True = add rebound control to goalie Impact and WAR (it is always shown as its own row)

SEASONS = sorted(int(Path(f).stem.split('_')[1]) for f in glob.glob(f'{DATA_DIR}/stints_*.csv'))
assert SEASONS, f"No stints_*.csv files found in {DATA_DIR}. Is Drive mounted?"
print("seasons:", SEASONS)

## 2. Load 5v5 stints and attach xG

For each season: reads the stints, labels each one with its zone start, adds up the xG of the shots that fell inside it (home and away separately), and keeps the 5v5 stints.

A shot belongs to the stint with `start_gs < game_seconds <= end_gs`. Shots with no xG (blocked shots and the few the xG model filters out) count as zero.

The same pass also collects what the scoring rows need: every goal with its scorer, both assists and the situation it was scored in; each shooter's own expected goals by situation; and each player's ice time by situation. Situations are from the player's own side: `EV` (5v5, 4v4, 3v3), `PP`, `SH`, and `OTH` (either net empty).

In [ ]:
xg = pd.read_csv(f'{DATA_DIR}/xg_final.csv', usecols=['game_id', 'event_id', 'xg_final', 'xg_flurry'])
ZONES = {'fly': 0, 'O': 1, 'N': 2, 'D': 3}
SIT_HOME = {'5v5': 'EV', 'ev_other': 'EV', 'home_pp': 'PP', 'away_pp': 'SH', 'other': 'OTH'}
SIT_AWAY = {'5v5': 'EV', 'ev_other': 'EV', 'home_pp': 'SH', 'away_pp': 'PP', 'other': 'OTH'}
FLIP = np.array([0, 3, 2, 1])            # the other team's view of the same zone

def load_season(y):
    st = pd.read_csv(f'{DATA_DIR}/stints_{y}.csv').sort_values(['game_id', 'start_gs']).reset_index(drop=True)
    st['stint'] = st.index

    # zone start, carried forward to stints that begin within ZONE_WINDOW seconds of the faceoff
    fo = st.zone_start_home != 'fly'
    st['fo_gs'], st['fo_zone'] = st.start_gs.where(fo), st.zone_start_home.where(fo)
    st[['fo_gs', 'fo_zone']] = st.groupby(['game_id', 'period'])[['fo_gs', 'fo_zone']].ffill()
    st['zone_home'] = np.where((st.start_gs - st.fo_gs) <= ZONE_WINDOW, st.fo_zone, 'fly')

    # match every shot to its stint
    sh = pd.read_csv(f'{DATA_DIR}/shots_{y}.csv', usecols=['game_id', 'event_id', 'game_seconds', 'is_home', 'is_goal',
                                                          'shooter_id', 'assist1_id', 'assist2_id', 'home_team', 'away_team', 'on_ice_for'])
    # final score of every game (regulation and overtime; shootouts are not in the shot data)
    gl = sh[sh.is_goal == 1].groupby(['game_id', 'is_home']).size().unstack(fill_value=0).reindex(columns=[0, 1], fill_value=0)
    games = sh.groupby('game_id')[['home_team', 'away_team']].first()
    games['home_goals'] = gl[1].reindex(games.index).fillna(0).values
    games['away_goals'] = gl[0].reindex(games.index).fillna(0).values
    sh = sh.dropna(subset=['game_seconds'])
    sh['game_seconds'] = sh.game_seconds.astype('int64')
    m = pd.merge_asof(sh.sort_values('game_seconds'),
                      st[['game_id', 'start_gs', 'end_gs', 'stint', 'state']].sort_values('end_gs'),
                      left_on='game_seconds', right_on='end_gs', by='game_id', direction='forward')
    m = m[m.stint.notna() & (m.start_gs < m.game_seconds)]

    # xG per stint
    mx = m.merge(xg, on=['game_id', 'event_id']).dropna(subset=[XG_COL])
    st['xg_home'] = mx[mx.is_home == 1].groupby('stint')[XG_COL].sum().reindex(st.stint).fillna(0).values
    st['xg_away'] = mx[mx.is_home == 0].groupby('stint')[XG_COL].sum().reindex(st.stint).fillna(0).values

    # goals with scorer, assists and situation (from the scoring team's side)
    g = m[m.is_goal == 1]
    goals = g[['shooter_id', 'assist1_id', 'assist2_id']].assign(
        sit=np.where(g.is_home == 1, g.state.map(SIT_HOME), g.state.map(SIT_AWAY)))

    # each shooter's own xG and goals by situation, on the shots the xG model scored. This uses the
    # unadjusted xG: the flurry adjustment is for team totals and would flatter rebound shooters here.
    mi = mx[mx.shooter_id.notna()].assign(sit=np.where(mx.is_home == 1, mx.state.map(SIT_HOME), mx.state.map(SIT_AWAY))[mx.shooter_id.notna()])
    mi = mi[mi.sit.isin(['EV', 'PP', 'SH'])]
    ind = mi.groupby(['shooter_id', 'sit']).agg(ixg=('xg_final', 'sum'), gx=('is_goal', 'sum')).unstack(fill_value=0)
    ind.columns = [f'{a}_{b}' for a, b in ind.columns]
    ind.index = ind.index.astype('int64')
    shotlvl = mi[mi.on_ice_for.notna()][['shooter_id', 'on_ice_for', 'sit', 'is_goal', 'xg_final']]

    # ice time by situation for every skater, in seconds
    parts = []
    for col, mp in (('home_skaters', SIT_HOME), ('away_skaters', SIT_AWAY)):
        e = pd.DataFrame({'player_id': st[col].str.split('|'), 'sit': st.state.map(mp), 'sec': st.seconds})
        e = e.explode('player_id').dropna(subset=['player_id'])
        e = e[e.player_id != '']
        parts.append(e.groupby(['player_id', 'sit']).sec.sum())
    toi_sit = pd.concat(parts).groupby(level=[0, 1]).sum().unstack(fill_value=0)
    toi_sit.index = toi_sit.index.astype('int64')

    # power-play stints, from the side with the man advantage
    pp = st[st.state.isin(['home_pp', 'away_pp']) & (st.seconds > 0)]
    hp = (pp.state == 'home_pp').values
    special = pd.DataFrame({'sec': pp.seconds.values.astype(float), 'xg': np.where(hp, pp.xg_home, pp.xg_away),
                            'att': np.where(hp, pp.home_skaters, pp.away_skaters),
                            'dfn': np.where(hp, pp.away_skaters, pp.home_skaters),
                            'n_att': np.where(hp, pp.n_home, pp.n_away), 'n_def': np.where(hp, pp.n_away, pp.n_home)})

    s5 = st[(st.state == '5v5') & (st.seconds > 0)]
    ids = lambda col: s5[col].str.split('|', expand=True).astype('int64').values
    return dict(game_id=s5.game_id.values, sec=s5.seconds.values.astype(float),
                H=ids('home_skaters'), A=ids('away_skaters'),
                xg_h=s5.xg_home.values, xg_a=s5.xg_away.values,
                zone_h=s5.zone_home.map(ZONES).values,
                sdiff_h=(s5.home_score - s5.away_score).clip(-3, 3).values,
                matched=len(m) / max(len(sh), 1), goals=goals, toi_sit=toi_sit, ind=ind,
                games=games, special=special, shotlvl=shotlvl)

data = {}
for y in SEASONS:
    d = data[y] = load_season(y)
    games = len(np.unique(d['game_id']))
    print(f"{y}: {len(d['sec']):,} 5v5 stints, {d['sec'].sum() / 60 / games:.1f} min and "
          f"{(d['xg_h'].sum() + d['xg_a'].sum()) / games:.2f} xG per game at 5v5, "
          f"{d['matched']:.1%} of shots matched to a stint, {len(d['goals']):,} goals")

# ---------- is the latest season still being played?
SEASON_DONE = {y: 1.0 for y in SEASONS}                      # share of each season played; only the latest can be under 1
LATEST = SEASONS[-1]
g_ = data[LATEST]['games']
share = len(g_) / (len(set(g_.home_team) | set(g_.away_team)) * 82 / 2)
if LATEST_SEASON_SHARE is not None:
    share = float(LATEST_SEASON_SHARE)
if share < 0.97:
    SEASON_DONE[LATEST] = max(share, 0.01)
season_share = lambda d, tag: d.season.map(SEASON_DONE) if tag == '1y' else 1.0
if SEASON_DONE[LATEST] < 1:
    print()
    print(f"{LATEST}-{str(LATEST + 1)[2:]} is in progress: {SEASON_DONE[LATEST]:.0%} of the season played ({len(g_):,} games).")
    print("Its 1-year cutoffs are scaled to match, and the 3-year versions blend it in with the seasons before it.")

## 3. How much does deployment matter?

xG per 60 for the attacking team, by how the stint began from that team's side. This is the raw size of the deployment effect in your data, before the regression. `fly` covers changes on the fly and anything more than `ZONE_WINDOW` seconds after a faceoff.

In [ ]:
rows = []
for y in SEASONS:
    d = data[y]
    zone = np.concatenate([d['zone_h'], FLIP[d['zone_h']]])
    rows.append(pd.DataFrame({'zone': zone, 'sec': np.tile(d['sec'], 2),
                              'xg': np.concatenate([d['xg_h'], d['xg_a']])}))
z = pd.concat(rows).groupby('zone').agg(sec=('sec', 'sum'), xg=('xg', 'sum'))
z.index = z.index.map({v: k for k, v in ZONES.items()}).map(
    {'fly': 'on the fly / open play', 'O': 'offensive zone start', 'N': 'neutral zone start', 'D': 'defensive zone start'})
z['share_of_time'] = (z.sec / z.sec.sum()).round(3)
z['xGF_per_60'] = (z.xg / z.sec * 3600).round(2)
print(z[['share_of_time', 'xGF_per_60']])
print(f"\nall 5v5: {z.xg.sum() / z.sec.sum() * 3600:.2f} xG per 60 per team")

## 4. Regression setup

Builds the design matrix for each season and reduces it to the totals the ridge solution needs. Any weighted combination of seasons can then be fitted in about a second, which is what makes the search in section 5 cheap.

Player effects are shrunk toward zero (league average). The deployment, score and home-ice controls are not shrunk.

**A season in progress.** When the latest season is still being played, the 3-year versions do not treat it as a full season of its own. On its first day the seasons before it keep the weights they had a year earlier (1 on last season, then the two older weights on the two seasons before that, so four seasons are in play). As the new season fills in, those weights slide to the usual ones, and the oldest season fades out by the final game. The new season's games always count in full; there are just fewer of them. This is used everywhere a 3-year number is built: the regressions, Finishing and Playmaking, penalties, special teams, shooting results and goalies. The 1-year versions are that season alone, with the minutes and shots cutoffs scaled to how much of it has been played.

In [ ]:
players = np.unique(np.concatenate([np.concatenate([data[y]['H'].ravel(), data[y]['A'].ravel()]) for y in SEASONS]))
P = len(players)
CONTROLS = ['zone_O', 'zone_N', 'zone_D', 'trail_3', 'trail_2', 'trail_1', 'lead_1', 'lead_2', 'lead_3',
            'home', 'intercept']
K = 2 * P + len(CONTROLS)
print(f"{P:,} skaters, {K:,} columns")

def design(d):
    n = len(d['sec'])
    off = np.searchsorted(players, np.vstack([d['H'], d['A']]))            # attackers
    dfn = np.searchsorted(players, np.vstack([d['A'], d['H']])) + P        # defenders
    zone = np.concatenate([d['zone_h'], FLIP[d['zone_h']]])
    sdiff = np.concatenate([d['sdiff_h'], -d['sdiff_h']])
    home = np.concatenate([np.ones(n), np.zeros(n)])
    r = np.arange(2 * n)
    rr, cc = [np.repeat(r, 5), np.repeat(r, 5)], [off.ravel(), dfn.ravel()]
    for code_, col in ((1, 0), (2, 1), (3, 2)):                            # zone dummies
        k = r[zone == code_]; rr.append(k); cc.append(np.full(len(k), 2 * P + col))
    for val, col in ((-3, 3), (-2, 4), (-1, 5), (1, 6), (2, 7), (3, 8)):   # score dummies
        k = r[sdiff == val]; rr.append(k); cc.append(np.full(len(k), 2 * P + col))
    k = r[home == 1]; rr.append(k); cc.append(np.full(len(k), 2 * P + 9))
    rr.append(r); cc.append(np.full(2 * n, 2 * P + 10))                    # intercept
    rr, cc = np.concatenate(rr), np.concatenate(cc)
    X = sp.csr_matrix((np.ones(len(rr)), (rr, cc)), shape=(2 * n, K))
    return X, off, dfn, zone

stats = {}
for y in SEASONS:
    t0 = time.time()
    d = data[y]
    X, off, dfn, zone = design(d)
    sec, xg_row = np.tile(d['sec'], 2), np.concatenate([d['xg_h'], d['xg_a']])
    G = (X.T @ sp.diags(sec) @ X).toarray().astype('float32')       # X' W X
    b = X.T @ (xg_row * 3600)                                       # X' W y, with y = xG per 60
    yy = float((xg_row ** 2 * 3600 ** 2 / sec).sum())               # y' W y
    stats[y] = dict(G=G, b=b, yy=yy, off=off, dfn=dfn, zone=zone, sec=sec, xg=xg_row)
    print(f"{y}: {X.shape[0]:,} rows  ({time.time() - t0:.0f}s)")

penalty = np.r_[np.ones(2 * P), np.full(len(CONTROLS), 1e-6)]      # shrink players only

def fit(weights, lam, st=None, pen=None):
    """Ridge fit on a weighted combination of seasons. weights = {season: weight}."""
    st, pen = (stats if st is None else st), (penalty if pen is None else pen)
    G = sum(w * st[s]['G'].astype('float64') for s, w in weights.items())
    b = sum(w * st[s]['b'] for s, w in weights.items())
    return solve(G + np.diag(pen * lam), b, assume_a='pos')

def sse(beta, y, st=None):
    """Weighted squared error of `beta` on season y's stints."""
    s = (stats if st is None else st)[y]
    return s['yy'] - 2 * beta @ s['b'] + beta @ (s['G'].astype('float64') @ beta)

def window(S, w1, w2):
    # Season weights for the 3-year versions. For a finished season: 1 / w1 / w2 on S, S-1, S-2.
    # A season in progress is not treated as a full season of its own: on its first day the seasons before it
    # keep the weights they had (1 / w1 / w2 on S-1, S-2, S-3), and they slide to the usual ones as it fills in.
    f = SEASON_DONE.get(S, 1.0)
    done = {S: 1.0, S - 1: w1, S - 2: w2}
    start = {S: 1.0, S - 1: 1.0, S - 2: w1, S - 3: w2}
    w = {y: f * done.get(y, 0.0) + (1 - f) * start[y] for y in start}
    return {y: v for y, v in w.items() if v > 0 and y in stats}

## 5. Choose the shrinkage and the season weights

For every combination of shrinkage strength and season weights, the model is fitted on a window ending in season S and then scored on how well it predicts the stints of season S+1, which it has not seen. The score shown is the improvement over a model that knows deployment, score and home ice but nothing about the players, averaged over all the seasons that can be tested.

- The best row with weights `(0, 0)` gives the shrinkage for the **1-year** rating.
- The best row overall gives the weights and shrinkage for the **3-year** rating.

The improvements are small in absolute terms because a single stint is mostly noise. What matters is the ranking of the rows.

In [ ]:
targets = [S for S in SEASONS if S - 2 in stats and S + 1 in stats]
assert targets, "Need at least four seasons to choose the weights."
null = {y: sse(fit({y: 1.0}, 1e12), y) for y in SEASONS}       # players shrunk to nothing

rows, t0 = [], time.time()
for (w1, w2) in WEIGHT_GRID:
    for lam in LAMBDAS:
        gain = np.mean([1 - sse(fit(window(S, w1, w2), lam), S + 1) / null[S + 1] for S in targets])
        rows.append({'w_last': w1, 'w_two_back': w2, 'lambda': lam, 'gain_pct': gain * 100})
grid = pd.DataFrame(rows)
print(f"tested on seasons {[S + 1 for S in targets]}  ({time.time() - t0:.0f}s)\n")
print(grid.pivot_table(index=['w_last', 'w_two_back'], columns='lambda', values='gain_pct')
          .sort_index(ascending=False).round(4))

one = grid[(grid.w_last == 0) & (grid.w_two_back == 0)]
LAM_1Y = float(one.loc[one.gain_pct.idxmax(), 'lambda'])
best = grid.loc[grid.gain_pct.idxmax()]
W1, W2, LAM_3Y = float(best.w_last), float(best.w_two_back), float(best['lambda'])
print(f"\n1-year: lambda {LAM_1Y:g}  (gain {one.gain_pct.max():.4f}%)")
print(f"3-year: weights 1 / {W1:g} / {W2:g}, lambda {LAM_3Y:g}  (gain {best.gain_pct:.4f}%)")
if SEASON_DONE[LATEST] < 1:
    print(f"3-year weights for {LATEST}-{str(LATEST + 1)[2:]} while it is in progress: "
          + ", ".join(f"{y}-{str(y + 1)[2:]} {v:.2f}" for y, v in sorted(window(LATEST, W1, W2).items(), reverse=True)))
if LAM_3Y in (LAMBDAS[0], LAMBDAS[-1]) or LAM_1Y in (LAMBDAS[0], LAMBDAS[-1]):
    print("NOTE: a chosen lambda is at the edge of LAMBDAS; widen the list in section 1 and rerun.")

## 6. Fit the ratings

For every season: a 1-year fit on that season alone, and a 3-year fit on that season and the two before it with the chosen weights. Early seasons use as many prior seasons as exist.

- `off` is the player's effect on his team's xG **for** per 60. Positive is good.
- `def` is his effect on xG **against** per 60 with the sign flipped, so positive is also good.

Also printed: what the model learned about deployment, score and home ice in the latest 3-year fit, in xG per 60 relative to a tied game, away, in open play.

In [ ]:
idx = {p: i for i, p in enumerate(players)}
out, net3 = [], {}
for S in SEASONS:
    b1, b3 = fit({S: 1.0}, LAM_1Y), fit(window(S, W1, W2), LAM_3Y)
    win = window(S, W1, W2)
    net3[S] = b3[:P] - b3[P:2 * P]          # offense + defense, the quality measure used in section 9
    toi1 = np.bincount(stats[S]['off'].ravel(), weights=np.repeat(stats[S]['sec'], 5), minlength=P) / 60
    toi3 = sum(w * np.bincount(stats[s]['off'].ravel(), weights=np.repeat(stats[s]['sec'], 5), minlength=P)
               for s, w in win.items()) / 60
    keep = toi3 > 0
    out.append(pd.DataFrame({'season': S, 'player_id': players[keep],
                             'min_5v5_1y': toi1[keep].round(1), 'min_5v5_3y_weighted': toi3[keep].round(1),
                             'seasons_in_3y': len(win),
                             'off_1y': b1[:P][keep], 'def_1y': -b1[P:2 * P][keep],
                             'off_3y': b3[:P][keep], 'def_3y': -b3[P:2 * P][keep]}))
ratings = pd.concat(out, ignore_index=True)
ratings.loc[ratings.min_5v5_1y == 0, ['off_1y', 'def_1y']] = np.nan

print(pd.Series(b3[2 * P:], index=CONTROLS).round(3).to_string())
print(f"\n{len(ratings):,} player-seasons")

## 7. Raw and zone-adjusted rates, and percentiles

Three versions of each number, so you can see what each adjustment did:

1. **Raw:** on-ice xG for and against per 60.
2. **Zone-adjusted:** raw, minus what an average player would be expected to get from the same mix of zone starts, plus the league average.
3. **Isolated:** the regression rating from section 6 (deployment, teammates, opponents, score and home ice all accounted for).

Then percentiles within forwards and within defensemen, for players above the minutes cutoffs in section 1. A player's team for a season is the one he played the most games for.

In [ ]:
def onice(S):
    s = stats[S]
    league = np.bincount(s['zone'], weights=s['xg'], minlength=4) / np.bincount(s['zone'], weights=s['sec'], minlength=4)
    expected = league[s['zone']] * s['sec']                       # xG an average line gets from this start
    tot = lambda ix, w: np.bincount(ix.ravel(), weights=np.repeat(w, 5), minlength=P)
    return dict(sec=tot(s['off'], s['sec']), xgf=tot(s['off'], s['xg']), xga=tot(s['dfn'] - P, s['xg']),
                exp_f=tot(s['off'], expected), exp_a=tot(s['dfn'] - P, expected),
                avg=s['xg'].sum() / s['sec'].sum() * 3600)

oi = {S: onice(S) for S in SEASONS}
def rates(S, weights, tag):
    t = {k: sum(w * oi[s][k] for s, w in weights.items()) for k in ('sec', 'xgf', 'xga', 'exp_f', 'exp_a')}
    avg = sum(w * oi[s]['avg'] * oi[s]['sec'].sum() for s, w in weights.items()) / sum(w * oi[s]['sec'].sum() for s, w in weights.items())
    with np.errstate(divide='ignore', invalid='ignore'):
        return pd.DataFrame({'season': S, 'player_id': players,
                             f'xgf60_{tag}': t['xgf'] / t['sec'] * 3600, f'xga60_{tag}': t['xga'] / t['sec'] * 3600,
                             f'xgf60_zone_adj_{tag}': (t['xgf'] - t['exp_f']) / t['sec'] * 3600 + avg,
                             f'xga60_zone_adj_{tag}': (t['xga'] - t['exp_a']) / t['sec'] * 3600 + avg})

for tag, wfun in (('1y', lambda S: {S: 1.0}), ('3y', lambda S: window(S, W1, W2))):
    ratings = ratings.merge(pd.concat([rates(S, wfun(S), tag) for S in SEASONS]), on=['season', 'player_id'], how='left')

# names, positions, team and games played
bio = pd.read_csv(f'{DATA_DIR}/players.csv', usecols=['player_id', 'player', 'position', 'shoots_catches', 'birth_country', 'birth_date'])
bio['pos_group'] = np.where(bio.position == 'D', 'D', 'F')
toi = pd.concat([pd.read_csv(f'{DATA_DIR}/toi_{S}.csv', usecols=['game_id', 'player_id', 'team']).assign(season=S) for S in SEASONS])
team = (toi.groupby(['season', 'player_id']).agg(team=('team', lambda t: t.value_counts().index[0]), gp=('game_id', 'nunique')).reset_index())
ratings = ratings.merge(bio, on='player_id', how='left').merge(team, on=['season', 'player_id'], how='left')

# percentiles within position, for players above the minutes cutoff
for tag, col, cut in (('1y', 'min_5v5_1y', MIN_MIN_1Y), ('3y', 'min_5v5_3y_weighted', MIN_MIN_3Y)):
    ok = ratings[col] >= cut * season_share(ratings, tag)
    for m in ('off', 'def'):
        ratings[f'{m}_{tag}_pct'] = (ratings[ok].groupby(['season', 'pos_group'])[f'{m}_{tag}']
                                     .rank(pct=True).mul(100).round(0))
    ratings[f'qualified_{tag}'] = ok
print(ratings.groupby('season')[['qualified_1y', 'qualified_3y']].sum())

## 8. Finishing, Playmaking and Production

**What goes into each**

| Score | Ingredients | Tuned to predict, next season |
|---|---|---|
| **Finishing** | Goals above expected (goals minus the player's own xG) and raw goals, each at even strength, on the power play and shorthanded | Goals above expected per 60 |
| **Playmaking** | Primary and secondary assists at even strength and on the power play, plus shorthanded assists pooled | Assists per 60 |
| **Production** | Goals, primary and secondary assists at even strength and on the power play, plus shorthanded points pooled | Points per 60 |

Goals and ice time with either net empty are left out. Goals above expected uses the unadjusted `xg_final`, because the flurry adjustment is meant for team totals.

**How the weights are found.** For every player with enough ice time in two consecutive seasons, this season's rate of each ingredient is used to predict the target next season. The fitted coefficients are the weights, so an ingredient that says more about the future counts for more. Forwards and defensemen are fitted separately, and weights cannot go negative. Each weight is printed with a +/- figure from refitting on resampled players: a weight that is small next to its +/- is not well pinned down.

For Playmaking and Production the target is the even-strength rate by default (`PROD_TARGET` in section 1). For Finishing it is goals above expected in all situations, which is already measured against expectation and so is not inflated by power-play time.

All rates are per 60 minutes of the player's total ice time (even strength, power play and shorthanded together).

**Each score** is the weighted sum of a player's rates. The 3-year version uses season weights chosen by how well they predict the following season.

In [ ]:
ROLES = [('G', 'shooter_id'), ('A1', 'assist1_id'), ('A2', 'assist2_id')]
SITS = ('EV', 'PP', 'SH')
COUNTS = [f'{s}_{r}' for s in SITS for r, _ in ROLES]
IND = [f'{k}_{s}' for k in ('ixg', 'gx') for s in SITS]       # own xG, and goals on the shots the xG model scored

def season_counts(y):
    d = data[y]
    out = d['toi_sit'].reindex(columns=list(SITS), fill_value=0).add_prefix('sec_')
    g = d['goals'][d['goals'].sit.isin(SITS)]
    for role, col in ROLES:
        c = g.dropna(subset=[col]).groupby([col, 'sit']).size().unstack(fill_value=0)
        c.index = c.index.astype('int64')
        out = out.join(c.add_suffix(f'_{role}'), how='left')
    out = out.join(d['ind'], how='left')
    out = out.reindex(columns=['sec_EV', 'sec_PP', 'sec_SH'] + COUNTS + IND, fill_value=0).fillna(0)
    for st in SITS:
        out[f'GAX_{st}'] = out[f'gx_{st}'] - out[f'ixg_{st}']               # goals above expected
    out['SH_P'], out['SH_A'] = out[['SH_G', 'SH_A1', 'SH_A2']].sum(axis=1), out[['SH_A1', 'SH_A2']].sum(axis=1)
    out['sec_all'] = out[['sec_EV', 'sec_PP', 'sec_SH']].sum(axis=1)
    out['P_all'], out['P_EV'] = out[COUNTS].sum(axis=1), out[['EV_G', 'EV_A1', 'EV_A2']].sum(axis=1)
    out['G_all'] = out[['EV_G', 'PP_G', 'SH_G']].sum(axis=1)
    out['A_all'], out['A_EV'] = out.P_all - out.G_all, out[['EV_A1', 'EV_A2']].sum(axis=1)
    out['ixg_all'], out['GAX_all'] = out[['ixg_EV', 'ixg_PP', 'ixg_SH']].sum(axis=1), out[['GAX_EV', 'GAX_PP', 'GAX_SH']].sum(axis=1)
    return out.rename_axis('player_id').reset_index().assign(season=y)

pc = pd.concat([season_counts(y) for y in SEASONS], ignore_index=True)
NUM = [c for c in pc.columns if c not in ('player_id', 'season')]
bio_pos = pd.read_csv(f'{DATA_DIR}/players.csv', usecols=['player_id', 'position']).set_index('player_id').position
pos_of = lambda ids: np.where(pd.Index(ids).map(bio_pos) == 'D', 'D', 'F')
pc['pos_group'] = pos_of(pc.player_id)

def prod_window(S, weights):
    """Weighted totals of counts and ice time over the seasons in `weights`, one row per player."""
    parts = [pc[pc.season == s].set_index('player_id')[NUM].mul(w) for s, w in weights.items()]
    t = pd.concat(parts).groupby(level=0).sum()
    return t[t.sec_all > 0]

per60 = lambda t, cols: t[cols].div(t.sec_all, axis=0) * 3600
ev = PROD_TARGET == 'ev'
BLOCKS = {
    'fin':  dict(title='Finishing', comp=['GAX_EV', 'GAX_PP', 'GAX_SH', 'EV_G', 'PP_G', 'SH_G'],
                 target=lambda t: t.GAX_all / t.sec_all * 3600, plain='gax60'),
    'play': dict(title='Playmaking', comp=['EV_A1', 'EV_A2', 'PP_A1', 'PP_A2', 'SH_A'],
                 target=lambda t: t.A_EV / t.sec_EV * 3600 if ev else t.A_all / t.sec_all * 3600, plain='a60'),
    'prod': dict(title='Production', comp=['EV_G', 'EV_A1', 'EV_A2', 'PP_G', 'PP_A1', 'PP_A2', 'SH_P'],
                 target=lambda t: t.P_EV / t.sec_EV * 3600 if ev else t.P_all / t.sec_all * 3600, plain='pts60'),
}

# --- context: how often each thing happens, per 60 minutes of its own situation
tot = pc.groupby('pos_group')[NUM].sum()
print("Per 60 minutes in each situation, all seasons:")
print(pd.DataFrame({f'{st} {lab}': tot[col.format(st)] / tot[f'sec_{st}'] * 3600 for st in SITS
                    for lab, col in (('goals', '{}_G'), ('own xG', 'ixg_{}'), ('primary assists', '{}_A1'),
                                     ('secondary assists', '{}_A2'))}).round(3).T)

def season_pairs(b, w1=0.0, w2=0.0, cut=MIN_PROD_FIT):
    """This-window rates and next-season target for every player with enough ice time in both."""
    rows = []
    for S in SEASONS:
        if S + 1 not in SEASONS or (w2 > 0 and S - 2 not in SEASONS):
            continue
        a, nx = prod_window(S, window(S, w1, w2)), prod_window(S + 1, {S + 1: 1.0})
        cur1 = prod_window(S, {S: 1.0})
        both = a.index.intersection(nx.index).intersection(cur1.index)
        a, nx, cur1 = a.loc[both], nx.loc[both], cur1.loc[both]
        ok = (cur1.sec_all >= cut * 60) & (nx.sec_all >= MIN_PROD_FIT * 60)
        rows.append(per60(a, b['comp'])[ok].assign(y=b['target'](nx)[ok], w=np.minimum(a.sec_all, nx.sec_all)[ok] / 60, season=S))
    out = pd.concat(rows)
    return out.assign(pos_group=pos_of(out.index))

def fit_weights(d, comp):
    sw = np.sqrt(d.w.values)
    xbar, ybar = np.average(d[comp], axis=0, weights=d.w), np.average(d.y, weights=d.w)
    return nnls((d[comp].values - xbar) * sw[:, None], (d.y.values - ybar) * sw)[0]

def score(t, b):
    w = np.vstack([b['W'][g].values for g in pos_of(t.index)]) if len(t) else np.zeros((0, len(b['comp'])))
    return pd.Series((per60(t, b['comp']).values * w).sum(axis=1), index=t.index)

rng = np.random.default_rng(0)
for name, b in BLOCKS.items():
    print(f"\n{'=' * 22} {b['title']} {'=' * 22}")
    d0, W, SE = season_pairs(b), {}, {}
    for grp, d in d0.groupby('pos_group'):
        W[grp] = pd.Series(fit_weights(d, b['comp']), index=b['comp'])
        ids = d.index.unique()
        rows_of = {p: np.flatnonzero(d.index == p) for p in ids}
        boots = [fit_weights(d.iloc[np.concatenate([rows_of[p] for p in rng.choice(ids, len(ids))])], b['comp'])
                 for _ in range(200)]
        SE[grp] = pd.Series(np.std(boots, axis=0), index=b['comp'])
    b['W'], b['wt'] = W, pd.DataFrame(W)
    print(f"Weights with their uncertainty (+/-); {len(d0):,} player-season pairs:")
    print(pd.concat({'weight': b['wt'], '+/-': pd.DataFrame(SE)}, axis=1).swaplevel(axis=1)
            .sort_index(axis=1, ascending=[True, False]).round(3))

    # season weights for the 3-year score
    res = []
    for (w1, w2) in WEIGHT_GRID:
        d = season_pairs(b, w1, w2, cut=MIN_ALL_1Y)
        d = d[d.season.isin([S for S in SEASONS if S - 2 in SEASONS])]         # same seasons for every row
        sc = (d[b['comp']] * np.vstack([W[g].values for g in d.pos_group])).sum(axis=1)
        r = {g: np.corrcoef(sc[d.pos_group == g], d.y[d.pos_group == g])[0, 1] for g in ('F', 'D')}
        res.append({'w_last': w1, 'w_two_back': w2, 'forwards': r['F'], 'defensemen': r['D'],
                    'both': (r['F'] * (d.pos_group == 'F').sum() + r['D'] * (d.pos_group == 'D').sum()) / len(d)})
    res = pd.DataFrame(res)
    print("\nCorrelation of the score with next season's target, by season weights:")
    print(res.round(4).to_string(index=False))
    b['w1'], b['w2'] = (float(v) for v in res.loc[res.both.idxmax(), ['w_last', 'w_two_back']])
    b['corr'] = {'F': float(res.loc[res.both.idxmax(), 'forwards']), 'D': float(res.loc[res.both.idxmax(), 'defensemen'])}
    print(f"3-year {b['title']} weights: 1 / {b['w1']:g} / {b['w2']:g}")
wt, PW1, PW2 = BLOCKS['prod']['wt'], BLOCKS['prod']['w1'], BLOCKS['prod']['w2']

In [ ]:
# --- scores for every player and season
out = []
for S in SEASONS:
    one = prod_window(S, {S: 1.0})
    ref = prod_window(S, window(S, PW1, PW2))
    t = pd.DataFrame({'min_all_3y_weighted': ref.sec_all / 60, 'gax60_3y': ref.GAX_all / ref.sec_all * 3600})
    for name, b in BLOCKS.items():
        three = prod_window(S, window(S, b['w1'], b['w2']))
        t = t.join(pd.DataFrame({f'{name}_3y': score(three, b)}), how='left').join(
            pd.DataFrame({f'{name}_1y': score(one, b)}), how='left')
    t = t.join(pd.DataFrame({'min_all_1y': one.sec_all / 60,
                             'pts60_1y': one.P_all / one.sec_all * 3600, 'a60_1y': one.A_all / one.sec_all * 3600,
                             'gax60_1y': one.GAX_all / one.sec_all * 3600,
                             'goals': one.G_all, 'a1': one[['EV_A1', 'PP_A1', 'SH_A1']].sum(axis=1),
                             'a2': one[['EV_A2', 'PP_A2', 'SH_A2']].sum(axis=1),
                             'ixg': one.ixg_all, 'gax': one.GAX_all,
                             'ev_points': one.P_EV, 'pp_points': one[['PP_G', 'PP_A1', 'PP_A2']].sum(axis=1),
                             'sh_points': one.SH_P}), how='left')
    out.append(t.rename_axis('player_id').reset_index().assign(season=S))
prod = pd.concat(out, ignore_index=True)
prod['points'] = prod.goals + prod.a1 + prod.a2
ratings = ratings.drop(columns=[c for c in prod.columns if c in ratings.columns and c not in ('season', 'player_id')])
ratings = ratings.merge(prod, on=['season', 'player_id'], how='left')

for tag, col, cut in (('1y', 'min_all_1y', MIN_ALL_1Y), ('3y', 'min_all_3y_weighted', MIN_ALL_3Y)):
    ok = ratings[col] >= cut * season_share(ratings, tag)
    for name in BLOCKS:
        ratings[f'{name}_{tag}_pct'] = ratings[ok].groupby(['season', 'pos_group'])[f'{name}_{tag}'].rank(pct=True).mul(100).round(0)
    ratings[f'prod_qualified_{tag}'] = ok
print(ratings.groupby('season')[['prod_qualified_1y', 'prod_qualified_3y']].sum())

## 9. Teammates, Competition and Penalties

**Teammates and Competition** are context, not skill. For every 5v5 stint, a player's teammates are the four other skaters on his side and his competition is the five skaters on the other side. Each of those players' quality is his 3-year isolated rating from section 6 (offense plus defense, in xG per 60). A player's Teammates number is the average quality of his teammates, weighted by the ice time shared with them; Competition is the same for his opponents. Higher means better linemates, or tougher opponents.

**Penalties** is minor penalties drawn minus minor penalties taken, per 60 minutes of all ice time. A double minor counts as two. Majors, misconducts and bench minors are left out: fighting majors come in pairs and do not create a power play, and a bench minor belongs to no one. The first table printed shows every penalty class in your data and how many there are, so you can see exactly what is and is not counted.

Each has a 1-year and a 3-year version. Teammates and Competition use the same season weights as the 5v5 ratings. For Penalties the weights are chosen, as elsewhere, by how well the 3-year number predicts the following season.

In [ ]:
# ---------- Teammates and Competition (5v5)
def quality_context(S):
    s, q = stats[S], net3[S]
    own, opp, sec = s['off'], s['dfn'] - P, s['sec']          # each stint appears once from each team's side
    q_own, q_opp = q[own].sum(axis=1), q[opp].sum(axis=1)
    tm, cp, den = np.zeros(P), np.zeros(P), np.zeros(P)
    for k in range(5):
        i = own[:, k]
        tm += np.bincount(i, weights=sec * (q_own - q[i]) / 4, minlength=P)     # his four teammates
        cp += np.bincount(i, weights=sec * q_opp / 5, minlength=P)              # the five opponents
        den += np.bincount(i, weights=sec, minlength=P)
    return dict(tm=tm, cp=cp, den=den)

qc = {S: quality_context(S) for S in SEASONS}
out = []
for S in SEASONS:
    row = {'season': S, 'player_id': players}
    for tag, w in (('1y', {S: 1.0}), ('3y', window(S, W1, W2))):
        den = sum(x * qc[s]['den'] for s, x in w.items())
        with np.errstate(divide='ignore', invalid='ignore'):
            row[f'teammates_{tag}'] = sum(x * qc[s]['tm'] for s, x in w.items()) / den
            row[f'competition_{tag}'] = sum(x * qc[s]['cp'] for s, x in w.items()) / den
    out.append(pd.DataFrame(row))
new = pd.concat(out, ignore_index=True)
ratings = ratings.drop(columns=[c for c in new.columns if c in ratings.columns and c not in ('season', 'player_id')])
ratings = ratings.merge(new, on=['season', 'player_id'], how='left')
for tag in ('1y', '3y'):
    ok = ratings[f'qualified_{tag}']
    for m in ('teammates', 'competition'):
        ratings[f'{m}_{tag}_pct'] = ratings[ok].groupby(['season', 'pos_group'])[f'{m}_{tag}'].rank(pct=True).mul(100).round(0)
print("spread across qualified players (xG per 60):")
print(ratings[ratings.qualified_3y].groupby('pos_group')[['teammates_3y', 'competition_3y']].describe()
      .loc[:, (slice(None), ['mean', 'std', 'min', 'max'])].round(3))

In [ ]:
# ---------- Penalties
pen = pd.concat([pd.read_csv(f'{DATA_DIR}/events_{S}.csv', usecols=['game_id', 'event', 'player_id', 'player2_id',
                                                                  'penalty_class', 'penalty_minutes', 'penalty_type'])
                   .query("event == 'penalty'").assign(season=S) for S in SEASONS], ignore_index=True)
print("Penalty classes in the data (rows = class, columns = minutes):")
print(pen.assign(penalty_class=pen.penalty_class.fillna('none'), penalty_minutes=pen.penalty_minutes.fillna(-1))
         .pivot_table(index='penalty_class', columns='penalty_minutes', values='game_id', aggfunc='size', fill_value=0))

minor = pen[pen.penalty_class == 'MIN'] if (pen.penalty_class == 'MIN').any() else pen[pen.penalty_minutes == 2]
minor = minor.assign(n=(minor.penalty_minutes.fillna(2) / 2).clip(1, 2))            # a double minor counts as two
print(f"\ncounted: {len(minor):,} of {len(pen):,} penalties "
      f"({minor.player_id.notna().mean():.1%} name who took it, {minor.player2_id.notna().mean():.1%} name who drew it)")
taken = minor.dropna(subset=['player_id']).groupby(['season', 'player_id']).n.sum()
drawn = minor.dropna(subset=['player2_id']).groupby(['season', 'player2_id']).n.sum()
sec_total = pd.concat({S: data[S]['toi_sit'].sum(axis=1) for S in SEASONS}, names=['season', 'player_id'])
pk = pd.DataFrame({'sec': sec_total}).join(taken.rename('pen_taken')).join(
    drawn.rename('pen_drawn').rename_axis(['season', 'player_id'])).fillna(0).reset_index()

def pen_window(S, weights):
    t = pd.concat([pk[pk.season == s].set_index('player_id')[['sec', 'pen_taken', 'pen_drawn']].mul(w)
                   for s, w in weights.items()]).groupby(level=0).sum()
    t = t[t.sec > 0]
    return t.assign(diff60=(t.pen_drawn - t.pen_taken) / t.sec * 3600)

res = []
for (w1, w2) in WEIGHT_GRID:
    rs = []
    for S in SEASONS:
        if S - 2 not in SEASONS or S + 1 not in SEASONS:
            continue
        a, b = pen_window(S, window(S, w1, w2)), pen_window(S + 1, {S + 1: 1.0})
        both = a.index.intersection(b.index)
        a, b = a.loc[both], b.loc[both]
        ok = (pen_window(S, {S: 1.0}).reindex(both).sec.fillna(0) >= MIN_ALL_1Y * 60) & (b.sec >= MIN_ALL_1Y * 60)
        rs.append(pd.DataFrame({'x': a.diff60[ok], 'y': b.diff60[ok]}))
    d = pd.concat(rs)
    res.append({'w_last': w1, 'w_two_back': w2, 'corr_next_season': d.x.corr(d.y), 'pairs': len(d)})
res = pd.DataFrame(res)
print("\nHow well penalty differential predicts next season's, by season weights:")
print(res.round(4).to_string(index=False))
QW1, QW2 = res.loc[res.corr_next_season.idxmax(), ['w_last', 'w_two_back']]
print(f"\n3-year Penalties weights: 1 / {QW1:g} / {QW2:g}")

out = []
for S in SEASONS:
    one, three = pen_window(S, {S: 1.0}), pen_window(S, window(S, QW1, QW2))
    t = pd.DataFrame({'pen_diff60_3y': three.diff60}).join(
        pd.DataFrame({'pen_diff60_1y': one.diff60, 'pen_taken': one.pen_taken, 'pen_drawn': one.pen_drawn}), how='left')
    out.append(t.rename_axis('player_id').reset_index().assign(season=S))
new = pd.concat(out, ignore_index=True)
ratings = ratings.drop(columns=[c for c in new.columns if c in ratings.columns and c not in ('season', 'player_id')])
ratings = ratings.merge(new, on=['season', 'player_id'], how='left')
for tag in ('1y', '3y'):
    ok = ratings[f'prod_qualified_{tag}']
    ratings[f'pen_{tag}_pct'] = ratings[ok].groupby(['season', 'pos_group'])[f'pen_diff60_{tag}'].rank(pct=True).mul(100).round(0)

## 10. Power Play and Penalty Kill

The same kind of regression as at 5v5, run on every power-play stint. The target is the xG per 60 of the team with the man advantage. Each skater on that team gets a **power-play offense** effect; each skater killing the penalty gets a **penalty-kill defense** effect. Unit-mates and opponents are in the same model, so a player is not credited just for being on a strong first unit. Controls: two-man advantages and 4-on-3s.

The shrinkage and season weights are chosen the same way as in section 5, by how well they predict the following season's power-play stints. Special-teams samples are much smaller than 5v5, so expect heavier shrinkage and less certain ratings.

- `pp_off` is the effect on xG for per 60 power-play minutes. Positive is good.
- `pk_def` is the effect on xG against per 60 shorthanded minutes, sign flipped, so positive is also good.

In [ ]:
ST_CONTROLS = ['two_man_advantage', 'four_on_three', 'intercept']
K_ST = 2 * P + len(ST_CONTROLS)

def design_special(d):
    d = d.reset_index(drop=True)
    n, rr, cc = len(d), [], []
    for col, offset in (('att', 0), ('dfn', P)):
        e = d[col].str.split('|').explode()
        ids = e.values.astype('int64')
        pos = np.minimum(np.searchsorted(players, ids), P - 1)
        ok = players[pos] == ids                                   # skaters who never played 5v5 are skipped
        rr.append(e.index.values[ok]); cc.append(pos[ok] + offset)
    r = np.arange(n)
    for mask, col in (((d.n_att - d.n_def == 2).values, 0), ((d.n_att == 4).values, 1), (np.ones(n, bool), 2)):
        rr.append(r[mask]); cc.append(np.full(mask.sum(), 2 * P + col))
    rr, cc = np.concatenate(rr), np.concatenate(cc)
    return sp.csr_matrix((np.ones(len(rr)), (rr, cc)), shape=(n, K_ST)), d

stats_st = {}
for y in SEASONS:
    X, d = design_special(data[y]['special'])
    sec, xg_row = d.sec.values, d.xg.values
    G = (X.T @ sp.diags(sec) @ X).toarray().astype('float32')
    on = np.asarray((X.T @ sec)).ravel()                           # seconds on the ice, per column
    stats_st[y] = dict(G=G, b=X.T @ (xg_row * 3600), yy=float((xg_row ** 2 * 3600 ** 2 / sec).sum()),
                       pp_sec=on[:P], pk_sec=on[P:2 * P])
    print(f"{y}: {len(d):,} power-play stints, {sec.sum() / 3600:,.0f} hours, "
          f"{xg_row.sum() / sec.sum() * 3600:.2f} xG per 60 for the power play")
penalty_st = np.r_[np.ones(2 * P), np.full(len(ST_CONTROLS), 1e-6)]

null_st = {y: sse(fit({y: 1.0}, 1e12, stats_st, penalty_st), y, stats_st) for y in SEASONS}
rows, t0 = [], time.time()
for (w1, w2) in WEIGHT_GRID:
    for lam in LAMBDAS_ST:
        gain = np.mean([1 - sse(fit(window(S, w1, w2), lam, stats_st, penalty_st), S + 1, stats_st) / null_st[S + 1]
                        for S in targets])
        rows.append({'w_last': w1, 'w_two_back': w2, 'lambda': lam, 'gain_pct': gain * 100})
grid_st = pd.DataFrame(rows)
print()
print(f"tested on seasons {[S + 1 for S in targets]}  ({time.time() - t0:.0f}s)")
print(grid_st.pivot_table(index=['w_last', 'w_two_back'], columns='lambda', values='gain_pct')
             .sort_index(ascending=False).round(4))
one = grid_st[(grid_st.w_last == 0) & (grid_st.w_two_back == 0)]
LAM_ST_1Y = float(one.loc[one.gain_pct.idxmax(), 'lambda'])
best = grid_st.loc[grid_st.gain_pct.idxmax()]
SW1, SW2, LAM_ST_3Y = float(best.w_last), float(best.w_two_back), float(best['lambda'])
print()
print(f"1-year: lambda {LAM_ST_1Y:g}  (gain {one.gain_pct.max():.4f}%)")
print(f"3-year: weights 1 / {SW1:g} / {SW2:g}, lambda {LAM_ST_3Y:g}  (gain {best.gain_pct:.4f}%)")
if LAM_ST_3Y in (LAMBDAS_ST[0], LAMBDAS_ST[-1]) or LAM_ST_1Y in (LAMBDAS_ST[0], LAMBDAS_ST[-1]):
    print("NOTE: a chosen lambda is at the edge of LAMBDAS_ST; widen the list in section 1 and rerun.")

In [ ]:
out = []
for S in SEASONS:
    b1 = fit({S: 1.0}, LAM_ST_1Y, stats_st, penalty_st)
    win = window(S, SW1, SW2)
    b3 = fit(win, LAM_ST_3Y, stats_st, penalty_st)
    out.append(pd.DataFrame({
        'season': S, 'player_id': players,
        'pp_min_1y': stats_st[S]['pp_sec'] / 60, 'pk_min_1y': stats_st[S]['pk_sec'] / 60,
        'pp_min_3y_weighted': sum(w * stats_st[s]['pp_sec'] for s, w in win.items()) / 60,
        'pk_min_3y_weighted': sum(w * stats_st[s]['pk_sec'] for s, w in win.items()) / 60,
        'pp_off_1y': b1[:P], 'pk_def_1y': -b1[P:2 * P], 'pp_off_3y': b3[:P], 'pk_def_3y': -b3[P:2 * P]}))
new = pd.concat(out, ignore_index=True)
ratings = ratings.drop(columns=[c for c in new.columns if c in ratings.columns and c not in ('season', 'player_id')])
ratings = ratings.merge(new, on=['season', 'player_id'], how='left')
for side, m in (('pp', 'pp_off'), ('pk', 'pk_def')):
    ratings.loc[ratings[f'{side}_min_1y'] == 0, f'{m}_1y'] = np.nan
    for tag, col, cut in (('1y', f'{side}_min_1y', MIN_PP_1Y), ('3y', f'{side}_min_3y_weighted', MIN_PP_3Y)):
        ok = ratings[col] >= cut * season_share(ratings, tag)
        ratings[f'{m}_{tag}_pct'] = ratings[ok].groupby(['season', 'pos_group'])[f'{m}_{tag}'].rank(pct=True).mul(100).round(0)
print(pd.Series(b3[2 * P:], index=ST_CONTROLS).round(3).to_string())

## 11. Results: goals above expected on own shots and on teammates' shots

Everything so far in Impact and WAR is built on expected goals. This section adds what was actually scored.

One regression over every unblocked shot. The target is the goal (1 or 0) minus the shot's xG. Each shot contributes:

- a **shooting** effect for the shooter: how much more often his own shots go in than expected;
- a **setup** effect for each of the other skaters on his side: how much more often their teammates' shots go in while they are on the ice. This is where a passer's value shows up, since the xG model cannot see the pass.

A goal above expected is credited once, split between the shooter and his linemates by the regression, so the two effects do not overlap.

**Shrinkage decides what these numbers are.** Heavy shrinkage keeps only what repeats (predictive); light shrinkage reproduces what happened (descriptive). The tables show both for every setting: `predicts` is how much better next season's shots are predicted than with no player information, and `describes` is the same for this season's own shots. The `shoot` and `setup` columns saved from this section use the setting that predicts best. Which version goes into Impact and WAR is decided in section 12, on team results.

The setup effect is tested first, on next-season prediction alone: it is tried at several strengths against a model with the shooter alone, and it is used only if it predicts better. The result is printed.

In [ ]:
SH_CONTROLS = ['power_play', 'shorthanded', 'intercept']
K_SH = 2 * P + len(SH_CONTROLS)

def design_shots(d):
    d = d.reset_index(drop=True)
    n, r = len(d), np.arange(len(d))
    sid = d.shooter_id.values.astype('int64')
    spos = np.minimum(np.searchsorted(players, sid), P - 1)
    sok = players[spos] == sid
    e = d.on_ice_for.str.split('|').explode()
    ids, rows_ = e.values.astype('int64'), e.index.values
    pos = np.minimum(np.searchsorted(players, ids), P - 1)
    ok = (players[pos] == ids) & (ids != sid[rows_])                 # the other skaters on his side
    pairs = np.unique(np.stack([rows_[ok], pos[ok]]), axis=1)        # a player listed twice counts once
    rr = [r[sok], pairs[0], r[(d.sit == 'PP').values], r[(d.sit == 'SH').values], r]
    cc = [spos[sok], pairs[1] + P, np.full((d.sit == 'PP').sum(), 2 * P), np.full((d.sit == 'SH').sum(), 2 * P + 1),
          np.full(n, 2 * P + 2)]
    rr, cc = np.concatenate(rr), np.concatenate(cc)
    return sp.csr_matrix((np.ones(len(rr)), (rr, cc)), shape=(n, K_SH)), d

stats_sh = {}
for y in SEASONS:
    X, d = design_shots(data[y]['shotlvl'])
    yv = (d.is_goal - d.xg_final).values
    n_cols = np.asarray(X.sum(axis=0)).ravel()
    stats_sh[y] = dict(G=(X.T @ X).toarray().astype('float32'), b=X.T @ yv, yy=float(yv @ yv),
                       shots=n_cols[:P], tm_shots=n_cols[P:2 * P])
    print(f"{y}: {len(d):,} shots, {d.is_goal.sum():,} goals, {d.xg_final.sum():,.0f} expected")
def pen_for(ratio):          # ratio = how much harder the setup effects are shrunk than the shooting effects
    return np.r_[np.ones(P), np.full(P, 1e9 if ratio is None else float(ratio)), np.full(len(SH_CONTROLS), 1e-6)]
null_sh = {y: sse(fit({y: 1.0}, 1e12, stats_sh, pen_for(1)), y, stats_sh) for y in SEASONS}
gain_sh = lambda beta, y: (1 - sse(beta, y, stats_sh) / null_sh[y]) * 100

# --- step 1: is there a setup effect, and how hard should it be shrunk? (judged on next-season prediction only)
t0, scan = time.time(), []
for ratio in SETUP_RATIOS:
    best = max(np.mean([gain_sh(fit(window(S, 0.8, 0.6), lam, stats_sh, pen_for(ratio)), S + 1) for S in targets])
               for lam in SHOT_LAMBDAS)
    scan.append({'setup_shrunk_x': 'shooter alone' if ratio is None else ratio, 'best_next_season_gain_pct': best})
scan = pd.DataFrame(scan)
print(scan.round(4).to_string(index=False))
SETUP_RATIO = SETUP_RATIOS[int(scan.best_next_season_gain_pct.idxmax())]
USE_SETUP = SETUP_RATIO is not None
PEN_SH = pen_for(SETUP_RATIO)
print(f"Setup effect -> {'INCLUDED, shrunk ' + str(SETUP_RATIO) + 'x as hard as shooting' if USE_SETUP else 'LEFT OUT: the shooter alone predicts next season best'}")

# --- step 2: shrinkage and season weights, balancing prediction and description
rows = []
for (w1, w2) in SHOT_WEIGHTS:
    for lam in SHOT_LAMBDAS:
        fits = [(S, fit(window(S, w1, w2), lam, stats_sh, PEN_SH)) for S in targets]
        rows.append({'w_last': w1, 'w_two_back': w2, 'lambda': lam,
                     'predicts': np.mean([gain_sh(bb, S + 1) for S, bb in fits]),
                     'describes': np.mean([gain_sh(bb, S) for S, bb in fits])})
fr = pd.DataFrame(rows)
print(f"({time.time() - t0:.0f}s)")

def show(d, label):
    print()
    print(f"{label}: gain over no player information, in percent")
    print(d.round(4).to_string(index=False))
    return d.loc[d.predicts.idxmax()]

pred1 = show(fr[(fr.w_last == 0) & (fr.w_two_back == 0)], "1-year")
pred3 = show(fr[fr.w_last > 0], "3-year")
LAM_SH_1Y = float(pred1['lambda'])
HW1, HW2, LAM_SH_3Y = float(pred3.w_last), float(pred3.w_two_back), float(pred3['lambda'])
USE_RESULTS = bool(fr.predicts.max() > 0)
USE_SETUP = USE_SETUP and USE_RESULTS
print()
print(f"best for prediction: 1-year lambda {LAM_SH_1Y:g}; 3-year weights 1 / {HW1:g} / {HW2:g}, lambda {LAM_SH_3Y:g}")
print(f"Results predict next season at all: {'YES' if USE_RESULTS else 'NO'} (best gain {fr.predicts.max():.4f}%)")

In [ ]:
out = []
for S in SEASONS:
    b1 = fit({S: 1.0}, LAM_SH_1Y, stats_sh, PEN_SH)
    b3 = fit(window(S, HW1, HW2), LAM_SH_3Y, stats_sh, PEN_SH)
    out.append(pd.DataFrame({'season': S, 'player_id': players,
                             'shots_1y': stats_sh[S]['shots'], 'tm_shots_1y': stats_sh[S]['tm_shots'],
                             'shoot_1y': b1[:P], 'setup_1y': b1[P:2 * P], 'shoot_3y': b3[:P], 'setup_3y': b3[P:2 * P]}))
new = pd.concat(out, ignore_index=True)
ratings = ratings.drop(columns=[c for c in new.columns if c in ratings.columns and c not in ('season', 'player_id')])
ratings = ratings.merge(new, on=['season', 'player_id'], how='left')
for m in ('shoot', 'setup'):
    for tag in ('1y', '3y'):
        ok = ratings[f'prod_qualified_{tag}']
        ratings[f'{m}_{tag}_pct'] = ratings[ok].groupby(['season', 'pos_group'])[f'{m}_{tag}'].rank(pct=True).mul(100).round(0)
print("goals above expected per 100 shots, spread across ranked players (3-year):")
print((ratings[ratings.prod_qualified_3y].groupby('pos_group')[['shoot_3y', 'setup_3y']].describe()
       .loc[:, (slice(None), ['std', 'min', 'max'])] * 100).round(2))

## 12. Skaters: goals above average by component

**Step 1: every component in goals.** Each rating is a rate; multiplying it by the minutes it applies to gives goals above average.

| Component | Rate | Minutes |
|---|---|---|
| 5v5 offense, 5v5 defense | isolated xG per 60 | 5v5 |
| Power play, penalty kill | isolated xG per 60 | power play, shorthanded |
| Results | goals above expected: whichever version does best on the team test below | his shots, teammates' shots |
| Penalties | drawn minus taken per 60, times the goal value of a minor | all |

Playmaking and Production are card rows and are not added here: a player's assists are already inside his on-ice ratings.

**Choosing the results component: prediction and description together.** The expected-goals components are the core. On top of them, several versions of "goals above expected" are tried: none at all, the Finishing score from section 8, the shot model from section 11 at several shrinkage strengths, and the raw unshrunk total. Each is judged twice on real team results. *Describes*: do the players' values, summed by team, match the team's goal difference that season? *Predicts*: do the same ratings, applied to next season's rosters and minutes, match next season's goal difference? Each is scaled to its best value and weighted by `DESCRIPTIVE_WEIGHT`. The option with the best average score across the 1-year and 3-year versions is used for both, so the two share one definition. The options are often close; `RESULTS_FORCE` in section 1 overrides the choice. In the output columns the chosen version appears as `gaa_shoot` (own shots) and `gaa_setup` (teammates' shots, shot model only).

**Two numbers come straight from results, not from assumptions:**

- **The goal value of a minor penalty:** power-play goals minus shorthanded goals, divided by the minors counted.
- **Goals per win:** from every team-season, how much winning percentage rises per goal of goal difference per game. A game tied after overtime counts as half a win for each side, because shootout goals are not in the data and a shootout is close to a coin flip.

Impact and WAR themselves are computed in section 14, after the goalies, so that skaters and goalies are put on the same scale together.

**Individualized Impact** is goals above average per 60 minutes of all ice time, then a percentile within position.

**Wins Above Replacement.** On each team the top 13 forwards and top 7 defensemen by ice time are regulars; everyone else is replacement level. The replacement players' average rate in each component is the baseline. A player's goals above replacement is his goals above average minus that baseline over his minutes, and WAR is that divided by goals per win.

**Active players.** A player needs `MIN_CURRENT_5V5` 5v5 minutes in the current season to be ranked in any 3-year percentile. Others keep their rows and ratings but are not ranked.

**1-year and 3-year.** The 1-year versions use the 1-year ratings and that season's minutes. The 3-year versions use the 3-year ratings and the player's average minutes per season over the seasons he played in the window, so they read as a typical season at his current level.

In [ ]:
# ---------- goals per win, from results
tg = []
for S in SEASONS:
    g = data[S]['games']
    for team, gf, ga in (('home_team', 'home_goals', 'away_goals'), ('away_team', 'away_goals', 'home_goals')):
        tg.append(pd.DataFrame({'season': S, 'team': g[team].values, 'gf': g[gf].values, 'ga': g[ga].values}))
tg = pd.concat(tg)
tg['win'] = np.where(tg.gf > tg.ga, 1.0, np.where(tg.gf == tg.ga, 0.5, 0.0))
tg['tied'] = (tg.gf == tg.ga).astype(float)
team_res = tg.groupby(['season', 'team']).agg(gp=('win', 'size'), gf=('gf', 'sum'), ga=('ga', 'sum'),
                                             wins=('win', 'sum'), tied=('tied', 'sum')).reset_index()
team_res['gd_pg'] = (team_res.gf - team_res.ga) / team_res.gp
team_res['win_pct'] = team_res.wins / team_res.gp

def wins_per_goal(d):                       # slope of (win% - 0.5) on goal difference per game, through the origin
    return (d.gp * d.gd_pg * (d.win_pct - 0.5)).sum() / (d.gp * d.gd_pg ** 2).sum()
GPW = 1 / wins_per_goal(team_res)
by_season = team_res.groupby('season').apply(lambda d: pd.Series({
    'teams': len(d), 'goals_per_game': d.gf.sum() / d.gp.sum() * 2, 'share_tied_after_ot': d.tied.sum() / d.gp.sum(),
    'goals_per_win': 1 / wins_per_goal(d)}))
print(by_season.round(3))
fitted = 0.5 + team_res.gd_pg / GPW
print()
print(f"GOALS PER WIN = {GPW:.3f}  (all {len(team_res)} team-seasons; goal difference explains "
      f"{np.corrcoef(fitted, team_res.win_pct)[0, 1] ** 2:.1%} of the variation in winning percentage)")

# ---------- goal value of a minor penalty, from results
gs = pd.concat([data[S]['goals'].assign(season=S) for S in SEASONS])
pv = pd.DataFrame({'pp_goals': gs[gs.sit == 'PP'].groupby('season').size(),
                   'sh_goals': gs[gs.sit == 'SH'].groupby('season').size(),
                   'minors': minor.groupby('season').n.sum()}).fillna(0)
pv['goals_per_minor'] = (pv.pp_goals - pv.sh_goals) / pv.minors
PEN_VALUE = (pv.pp_goals.sum() - pv.sh_goals.sum()) / pv.minors.sum()
print()
print(pv.round(3))
print(f"GOAL VALUE OF A MINOR = {PEN_VALUE:.3f}")

In [ ]:
# ---------- average minutes per season played, for the 3-year versions
all_sec = {S: pc[pc.season == S].set_index('player_id').sec_all.reindex(players).fillna(0).values for S in SEASONS}
def avg_min(S, w1, w2, sec_of, per=60):
    win = window(S, w1, w2)
    num = sum(w * sec_of(s) for s, w in win.items())
    den = sum(w * (sec_of(s) > 0) * SEASON_DONE[s] for s, w in win.items())      # a part season counts as a part
    with np.errstate(divide='ignore', invalid='ignore'):
        return np.where(den > 0, num / den, 0) / per
out = []
for S in SEASONS:
    out.append(pd.DataFrame({'season': S, 'player_id': players,
        'avg5_3y': avg_min(S, W1, W2, lambda s: oi[s]['sec']),
        'avgpp_3y': avg_min(S, SW1, SW2, lambda s: stats_st[s]['pp_sec']),
        'avgpk_3y': avg_min(S, SW1, SW2, lambda s: stats_st[s]['pk_sec']),
        'avgall_3y': avg_min(S, PW1, PW2, lambda s: all_sec[s]),
        'avgshots_3y': avg_min(S, HW1, HW2, lambda s: stats_sh[s]['shots'], per=1),
        'avgtm_3y': avg_min(S, HW1, HW2, lambda s: stats_sh[s]['tm_shots'], per=1)}))
new = pd.concat(out, ignore_index=True)
ratings = ratings.drop(columns=[c for c in new.columns if c in ratings.columns and c not in ('season', 'player_id')])
ratings = ratings.merge(new, on=['season', 'player_id'], how='left')

# ---------- goals above average, by component
def center(x, w, by_position=True):         # x minus its weighted league average that season
    keys = [ratings.season, ratings.pos_group] if by_position else [ratings.season]
    w = w.fillna(0)
    avg = (x.fillna(0) * w).groupby(keys).transform('sum') / w.groupby(keys).transform('sum')
    return (x - avg).fillna(0)

f = lambda c: ratings[c].fillna(0)
COMPONENTS = ['off', 'def', 'pp', 'pk', 'shoot', 'setup', 'pen']
CORE = ['off', 'def', 'pp', 'pk', 'pen']
MIN = {'1y': dict(off='min_5v5_1y', pp='pp_min_1y', pk='pk_min_1y', all='min_all_1y'),
       '3y': dict(off='avg5_3y', pp='avgpp_3y', pk='avgpk_3y', all='avgall_3y')}
SHOTS = {'1y': dict(shoot='shots_1y', setup='tm_shots_1y'), '3y': dict(shoot='avgshots_3y', setup='avgtm_3y')}
for tag, mm in MIN.items():
    # The regression ratings are measured from the model's own baseline, not from the league average,
    # so each is centered on the league's ice-time-weighted mean. League goals above average then sum to zero.
    ratings[f'gaa_off_{tag}'] = center(ratings[f'off_{tag}'], f(mm['off']), False) * f(mm['off']) / 60
    ratings[f'gaa_def_{tag}'] = center(ratings[f'def_{tag}'], f(mm['off']), False) * f(mm['off']) / 60
    ratings[f'gaa_pp_{tag}'] = center(ratings[f'pp_off_{tag}'], f(mm['pp']), False) * f(mm['pp']) / 60
    ratings[f'gaa_pk_{tag}'] = center(ratings[f'pk_def_{tag}'], f(mm['pk']), False) * f(mm['pk']) / 60
    ratings[f'gaa_pen_{tag}'] = center(ratings[f'pen_diff60_{tag}'], f(mm['all'])) * PEN_VALUE * f(mm['all']) / 60
    ratings[f'gaa_core_{tag}'] = ratings[[f'gaa_{c}_{tag}' for c in CORE]].sum(axis=1)

# ---------- which results component? Judged on team results, for description and for prediction
gd_actual = (team_res.gf - team_res.ga).set_axis(pd.MultiIndex.from_frame(team_res[['season', 'team']]))
nxt = ratings[['season', 'player_id', 'team', 'min_all_1y']].rename(columns={'team': 'team_next', 'min_all_1y': 'min_next'})
nxt['season'] -= 1
link = ratings[['season', 'player_id']].merge(nxt, on=['season', 'player_id'], how='left')
link.index = ratings.index

def team_sums(total, tag):
    mins = f(MIN[tag]['all'])
    rate = (total / mins.where(mins > 0)).fillna(0)                              # goals above average per minute
    now = (rate * f('min_all_1y')).groupby([ratings.season, ratings.team]).sum()  # this season's minutes
    fut = (rate * link.min_next.fillna(0)).groupby([ratings.season + 1, link.team_next]).sum()   # next season's
    fut.index.names = now.index.names = ['season', 'team']
    return now, fut

def team_eval(total, tag):
    corr = lambda x: np.corrcoef(*x.rename('v').to_frame().join(gd_actual.rename('gd'), how='inner').values.T)[0, 1]
    now, fut = team_sums(total, tag)
    return corr(now), corr(fut)

def shot_model(lam):
    rows = []
    for S in SEASONS:
        b1 = fit({S: 1.0}, lam, stats_sh, PEN_SH)
        b3 = fit(window(S, HW1, HW2), lam, stats_sh, PEN_SH)
        rows.append(pd.DataFrame({'season': S, 'player_id': players, 's_1y': b1[:P], 't_1y': b1[P:2 * P],
                                  's_3y': b3[:P], 't_3y': b3[P:2 * P]}))
    c = ratings[['season', 'player_id']].merge(pd.concat(rows), on=['season', 'player_id'], how='left')
    c.index = ratings.index
    return {tag: (center(c[f's_{tag}'], f(SHOTS[tag]['shoot']), False) * f(SHOTS[tag]['shoot']),
                  center(c[f't_{tag}'], f(SHOTS[tag]['setup']), False) * f(SHOTS[tag]['setup']) * USE_SETUP)
            for tag in MIN}

zero = pd.Series(0.0, index=ratings.index)
is_fwd = (ratings.pos_group == 'F').astype(float)
CAND = {'expected goals only': {tag: (zero, zero) for tag in MIN},
        'Finishing score (section 8), forwards': {
            tag: (center(ratings[f'fin_{tag}'], f(mm['all'])) * f(mm['all']) / 60 * is_fwd, zero) for tag, mm in MIN.items()},
        'actual goals above expected, no shrinkage': {
            tag: (center(ratings[f'gax60_{tag}'], f(mm['all'])) * f(mm['all']) / 60, zero) for tag, mm in MIN.items()}}
if USE_RESULTS:
    for lam in RESULT_LAMBDAS:
        CAND[f'shot model, lambda {lam:g}'] = shot_model(lam)

rows = {}
for name, c in CAND.items():
    r = {}
    for tag in MIN:
        r[f'describes_{tag}'], r[f'predicts_{tag}'] = team_eval(ratings[f'gaa_core_{tag}'] + c[tag][0] + c[tag][1], tag)
    rows[name] = r
ev = pd.DataFrame(rows).T
for tag in MIN:
    scaled = lambda col: ev[col] / ev[col].max() if ev[col].max() > 0 else 0.0
    ev[f'score_{tag}'] = (1 - DESCRIPTIVE_WEIGHT) * scaled(f'predicts_{tag}') + DESCRIPTIVE_WEIGHT * scaled(f'describes_{tag}')
ev['score'] = ev[['score_1y', 'score_3y']].mean(axis=1)        # one definition for both versions
RESULTS_USED = RESULTS_FORCE if RESULTS_FORCE in ev.index else ev.score.idxmax()
CHOICE = {tag: RESULTS_USED for tag in MIN}
print("Results component, judged on team goal difference:")
print("  describes = correlation of summed player value with the team's goal difference the same season")
print("  predicts  = the same ratings applied to next season's rosters and minutes, against next season's goal difference")
print(ev.round(4).to_string())
print(f"used for both the 1-year and 3-year versions: {RESULTS_USED}"
      f"{' (forced by RESULTS_FORCE)' if RESULTS_FORCE in ev.index else ' (best overall score)'}")
for tag in MIN:
    ratings[f'gaa_shoot_{tag}'], ratings[f'gaa_setup_{tag}'] = CAND[CHOICE[tag]][tag]
    ratings[f'gaa_{tag}'] = ratings[[f'gaa_{c}_{tag}' for c in COMPONENTS]].sum(axis=1)
SK_TEAM = {tag: team_sums(ratings[f'gaa_{tag}'], tag) for tag in MIN}      # skater totals by team, for sections 13-14
RAW_GAA = {tag: {c: ratings[f'gaa_{c}_{tag}'].copy() for c in COMPONENTS} for tag in MIN}   # before calibration

## 13. Goalies

Every unblocked shot a goalie faced with his net occupied, set against its expected-goal value. All numbers are **goals saved above average**: positive means he stopped more than an average goalie would have on the same shots.

| Measure | What it is |
|---|---|
| GSAx | Expected goals against minus goals against, on all shots faced |
| 5v5 GSAx, PK GSAx | The same, split by situation. "PK" is shots faced while his team is shorthanded, that is, against the opponent's power play |
| Low, medium, high danger GSAx | The same, split by the shot's xG: under 5%, 5 to 15%, over 15% |
| Team-adjusted GSAA | A regression over every shot with an effect for the goalie and an effect for the team in front of him, so a goalie is not credited or blamed for what his team does that the xG model cannot see. Optionally also adjusted for the finishing skill of the shooters he faced |
| Rebound control | For every save, the chance it produces a rebound shot within 3 seconds is predicted from the shot's distance, angle, type and situation. Rebounds allowed below that expectation are converted to goals at the average xG of a rebound shot |
| Win probability added | Each shot's goals saved, weighted by how much a goal at that moment would have changed the chance of winning, from a win-probability model fitted to your games (score and time) |

**Above average.** Every split is measured from the league average for that season, situation and danger band, so the league adds up to zero and a goalie's splits add up to his total.

**What goes into goalie WAR.** Goalie results carry very little from one season to the next, so the two versions are built differently on purpose:

- **1-year WAR** uses what actually happened: this season's goals saved above average, adjusted for the shooters faced (`GOALIE_1Y`). Nothing is shrunk, so it describes the season.
- **3-year WAR** uses the team-adjusted model, with its shrinkage and season weights tuned to predict the next season's shots (`GOALIE_3Y = None`). Shrinkage pulls a goalie with few shots toward average, which is what a gauge of current value should do; section 14 then puts the result back on the scale of real goals.

The team-adjusted rating is also kept as its own row on the card in both versions. A table compares every option on team results, and either setting in section 1 can be pointed at any row of it. Whatever the team effect takes away from a goalie is credited to nobody, so the team-adjusted rows will always look a little worse on that table.

In [ ]:
# ---------- every shot, with the goalie in net
G_COLS = ['game_id', 'event_id', 'event', 'is_goal', 'goalie_id', 'shooter_id', 'is_home', 'home_team', 'away_team',
          'period', 'game_seconds', 'strength', 'empty_net_against', 'own_net_empty', 'rebound', 'home_score',
          'away_score', 'shot_type', 'distance', 'angle']
parts = []
for S in SEASONS:
    d = pd.read_csv(f'{DATA_DIR}/shots_{S}.csv', usecols=G_COLS).assign(season=S)
    d = d.merge(xg[['game_id', 'event_id', 'xg_final']], on=['game_id', 'event_id'], how='left')
    nxt = d.groupby('game_id')[['rebound', 'xg_final']].shift(-1)          # the next shot attempt in the game
    d['gave_rebound'] = ((d.event == 'shot-on-goal') & (nxt.rebound == 1)).astype(int)
    d['rebound_xg'] = nxt.xg_final.fillna(0) * d.gave_rebound
    parts.append(d)
shots_all = pd.concat(parts, ignore_index=True)
shots_all['def_team'] = np.where(shots_all.is_home == 1, shots_all.away_team, shots_all.home_team)
sk_n = shots_all.strength.str.split('v', expand=True).astype(float)
both_in = (shots_all.empty_net_against == 0) & (shots_all.own_net_empty == 0)
shots_all['sit'] = np.select([(sk_n[0] == 5) & (sk_n[1] == 5) & both_in, (sk_n[0] > sk_n[1]) & both_in], ['5v5', 'pk'], 'other')

faced = shots_all[shots_all.xg_final.notna() & (shots_all.empty_net_against == 0) & shots_all.goalie_id.notna()].copy()
faced['goalie_id'] = faced.goalie_id.astype('int64')
eff = ratings[['season', 'player_id', 'shoot_3y']].rename(columns={'player_id': 'shooter_id', 'shoot_3y': 'shooter_effect'})
faced = faced.merge(eff, on=['season', 'shooter_id'], how='left')
faced['shooter_effect'] = faced.shooter_effect.fillna(0) * float(ADJUST_FOR_SHOOTERS and USE_RESULTS)
faced['danger'] = pd.cut(faced.xg_final, [0, .05, .15, 1], labels=['low', 'med', 'high'], include_lowest=True).astype(str)
# goals saved on this shot, measured from the league average for that season, situation and danger band,
# so every split is "above average" and the splits add up to the total
cell = [faced.season, faced.sit, faced.danger]
above_avg = lambda x: x - x.groupby(cell).transform('mean')
faced['saved_raw'] = above_avg(faced.xg_final - faced.is_goal)
faced['saved'] = above_avg(faced.xg_final - faced.is_goal + faced.shooter_effect)      # ... allowing for who took it

t = faced.groupby('sit').agg(shots=('is_goal', 'size'), goals=('is_goal', 'sum'), xG=('xg_final', 'sum'))
t2 = faced.groupby('danger').agg(shots=('is_goal', 'size'), goals=('is_goal', 'sum'), xG=('xg_final', 'sum')).reindex(['low', 'med', 'high'])
lg = pd.concat([t, t2])
lg['goals_per_100_shots'] = (lg.goals / lg.shots * 100).round(2)
print(f"{len(faced):,} shots faced by {faced.goalie_id.nunique()} goalies")
print("league totals on those shots (expected and actual goals should be close; any gap is removed by measuring from the average):")
print(lg.round(0).assign(goals_per_100_shots=lg.goals_per_100_shots))

In [ ]:
# ---------- win probability: chance the home team wins, from the score and the time played
res_games = pd.concat([data[S]['games'] for S in SEASONS])
home_win = pd.Series(np.where(res_games.home_goals > res_games.away_goals, 1.0,
                              np.where(res_games.home_goals == res_games.away_goals, 0.5, 0.0)), index=res_games.index)
wp_feats = lambda d, u: np.column_stack([d, d * u, d * u ** 2, d * u ** 8])
reg = shots_all[(shots_all.period <= 3) & shots_all.game_seconds.notna() & shots_all.home_score.notna()]
d_, u_ = (reg.home_score - reg.away_score).values.astype(float), (reg.game_seconds / 3600).clip(0, 1).values
hw = reg.game_id.map(home_win).values
X2, y2, w2 = np.vstack([wp_feats(d_, u_)] * 2), np.r_[np.ones(len(reg)), np.zeros(len(reg))], np.r_[hw, 1 - hw]
keep = w2 > 0                                               # a game tied after overtime counts half each way
wp = LogisticRegression(C=1e6, max_iter=2000).fit(X2[keep], y2[keep], sample_weight=w2[keep])
p_home = lambda d, u: wp.predict_proba(wp_feats(np.asarray(d, float), np.asarray(u, float)))[:, 1]

mins = [0, 20, 40, 50, 58]
print("chance the home team wins, by lead and minutes played:")
print(pd.DataFrame({f'{m} min': p_home(np.array([-2, -1, 0, 1, 2]), np.full(5, m / 60)) for m in mins},
                   index=['down 2', 'down 1', 'tied', 'up 1', 'up 2']).round(3))

# what a goal against at this moment would cost the goalie's team, in win probability
d0 = (faced.home_score.fillna(0) - faced.away_score.fillna(0)).values.astype(float)
d1 = d0 + np.where(faced.is_home == 1, 1, -1)              # the shooting team scores
u = (faced.game_seconds / 3600).clip(0, 1).fillna(0).values
faced['leverage'] = np.where(faced.period.values > 3, 0.5, np.abs(p_home(d0, u) - p_home(d1, u)))
faced['wpa'] = faced.saved_raw * faced.leverage
print(f"a goal against costs {faced.leverage.mean():.3f} wins on average, {faced.leverage.quantile(.95):.3f} at the 95th percentile")

# ---------- rebound control: expected chance that a save produces a rebound shot
sv = shots_all[(shots_all.event == 'shot-on-goal') & shots_all.goalie_id.notna() & (shots_all.empty_net_against == 0)].copy()
sv['goalie_id'] = sv.goalie_id.astype('int64')
Xr = pd.DataFrame({'distance': sv.distance.values, 'angle': sv.angle.values,
                   'shot_type': pd.Categorical(sv.shot_type).codes, 'sit': pd.Categorical(sv.sit).codes})
expected = np.zeros(len(sv))
for tr, te in GroupKFold(n_splits=5).split(Xr, groups=sv.game_id.values):
    m = HistGradientBoostingClassifier(max_iter=120, learning_rate=0.1).fit(Xr.iloc[tr], sv.gave_rebound.values[tr])
    expected[te] = m.predict_proba(Xr.iloc[te])[:, 1]
sv['reb_expected'] = expected
REB_VALUE = sv.rebound_xg.sum() / max(sv.gave_rebound.sum(), 1)
fewer = sv.reb_expected - sv.gave_rebound                              # rebounds avoided on this save
sv['reb_saved'] = (fewer - fewer.groupby(sv.season).transform('mean')) * REB_VALUE
print(f"{len(sv):,} saves, {sv.gave_rebound.mean():.1%} followed by a rebound shot; a rebound shot is worth {REB_VALUE:.3f} goals")

In [ ]:
# ---------- totals per goalie, team and season
keys = ['season', 'goalie_id', 'def_team']
gt = faced.groupby(keys).agg(shots=('saved_raw', 'size'), xga=('xg_final', 'sum'), ga=('is_goal', 'sum'),
                             gsax=('saved_raw', 'sum'), gsax_shooter_adj=('saved', 'sum'), wpa=('wpa', 'sum'),
                             games=('game_id', 'nunique'))
for col, names in (('sit', ['5v5', 'pk', 'other']), ('danger', ['low', 'med', 'high'])):
    tot = faced.groupby(keys + [col]).saved_raw.sum().unstack(fill_value=0).reindex(columns=names, fill_value=0)
    cnt = faced.groupby(keys + [col]).size().unstack(fill_value=0).reindex(columns=names, fill_value=0)
    gt = gt.join(tot.add_prefix('gsax_')).join(cnt.add_prefix('shots_'))
gt = gt.join(sv.groupby(keys).agg(saves=('gave_rebound', 'size'), rebounds=('gave_rebound', 'sum'),
                                  rebounds_expected=('reb_expected', 'sum'), reb_saved=('reb_saved', 'sum')))
gt = gt.fillna(0).reset_index().rename(columns={'def_team': 'team'})
NUMS = [c for c in gt.columns if c not in ('season', 'goalie_id', 'team')]

# ---------- team-adjusted model: one effect per goalie and one per team, on goals saved per shot
goalies, g_teams = np.sort(faced.goalie_id.unique()), np.sort(faced.def_team.unique())
NG, NT = len(goalies), len(g_teams)

def design_goalies(d):
    n, r = len(d), np.arange(len(d))
    rr = [r, r, r[(d.sit == 'pk').values], r[(d.sit == 'other').values], r]
    cc = [np.searchsorted(goalies, d.goalie_id.values), NG + np.searchsorted(g_teams, d.def_team.values),
          np.full((d.sit == 'pk').sum(), NG + NT), np.full((d.sit == 'other').sum(), NG + NT + 1), np.full(n, NG + NT + 2)]
    rr, cc = np.concatenate(rr), np.concatenate(cc)
    return sp.csr_matrix((np.ones(len(rr)), (rr, cc)), shape=(n, NG + NT + 3))

stats_g = {}
for S in SEASONS:
    d = faced[faced.season == S]
    X, yv = design_goalies(d), d.saved.values
    stats_g[S] = dict(G=(X.T @ X).toarray(), b=X.T @ yv, yy=float(yv @ yv))
pen_g = lambda ratio: np.r_[np.ones(NG), np.full(NT, 1e9 if ratio is None else float(ratio)), np.full(3, 1e-6)]
null_g = {y: sse(fit({y: 1.0}, 1e12, stats_g, pen_g(1)), y, stats_g) for y in SEASONS}
gain_g = lambda beta, y: (1 - sse(beta, y, stats_g) / null_g[y]) * 100

rows = []
for ratio in list(G_TEAM_RATIOS) + [None]:
    for (w1, w2) in SHOT_WEIGHTS:
        for lam in G_LAMBDAS:
            rows.append({'team_shrunk_x': np.nan if ratio is None else ratio, 'w_last': w1, 'w_two_back': w2, 'lambda': lam,
                         'predicts': np.mean([gain_g(fit(window(S, w1, w2), lam, stats_g, pen_g(ratio)), S + 1) for S in targets])})
gsearch = pd.DataFrame(rows)
by_ratio = gsearch.groupby('team_shrunk_x', dropna=False).predicts.max().rename('best_next_season_gain_pct').reset_index()
by_ratio['team_shrunk_x'] = by_ratio.team_shrunk_x.astype(object).where(by_ratio.team_shrunk_x.notna(), 'no team effect')
print("team effect in the goalie model (judged on next-season prediction):")
print(by_ratio.round(4).to_string(index=False))
adj = gsearch[gsearch.team_shrunk_x.notna()]
best3 = adj[adj.w_last > 0].loc[adj[adj.w_last > 0].predicts.idxmax()]
G_RATIO, GW1, GW2, LAM_G_3Y = float(best3.team_shrunk_x), float(best3.w_last), float(best3.w_two_back), float(best3['lambda'])
one = adj[(adj.w_last == 0) & (adj.team_shrunk_x == G_RATIO)]
LAM_G_1Y = float(one.loc[one.predicts.idxmax(), 'lambda'])
print()
print(f"team-adjusted model: team effect shrunk {G_RATIO:g}x; 1-year lambda {LAM_G_1Y:g}; "
      f"3-year weights 1 / {GW1:g} / {GW2:g}, lambda {LAM_G_3Y:g}")

In [ ]:
# ---------- per-goalie arrays by season, and "typical season" totals for the 3-year versions
gsea = gt.groupby(['season', 'goalie_id'])[NUMS].sum()
A = {S: {c: gsea.xs(S, level='season')[c].reindex(goalies).fillna(0).values for c in NUMS} for S in SEASONS}
def weighted(S, name):
    return sum(w * A[s][name] for s, w in window(S, GW1, GW2).items())
def typical(S, name):                       # average per season he played, over the 3-year window
    den = sum(w * (A[s]['shots'] > 0) * SEASON_DONE[s] for s, w in window(S, GW1, GW2).items())
    with np.errstate(divide='ignore', invalid='ignore'):
        return np.where(den > 0, weighted(S, name) / den, 0.0)
def ratio_of(num, den):
    with np.errstate(divide='ignore', invalid='ignore'):
        return np.where(den > 0, num / den, 0.0)

# ---------- which version of goals saved goes into WAR? Judged on team results together with the skaters
def model_rates(lam):
    return {'1y': {S: fit({S: 1.0}, lam, stats_g, pen_g(G_RATIO))[:NG] for S in SEASONS},
            '3y': {S: fit(window(S, GW1, GW2), lam, stats_g, pen_g(G_RATIO))[:NG] for S in SEASONS}}
def simple_rates(col):
    return {'1y': {S: ratio_of(A[S][col], A[S]['shots']) for S in SEASONS},
            '3y': {S: ratio_of(weighted(S, col), weighted(S, 'shots')) for S in SEASONS}}
G_DEFAULT = 'team-adjusted model, tuned to predict'
GCAND = {'raw GSAx': simple_rates('gsax'), 'shooter-adjusted GSAx': simple_rates('gsax_shooter_adj'),
         G_DEFAULT: {'1y': model_rates(LAM_G_1Y)['1y'], '3y': model_rates(LAM_G_3Y)['3y']}}
for lam in G_LAMBDAS:
    GCAND[f'team-adjusted model, lambda {lam:g}'] = model_rates(lam)

gi = np.searchsorted(goalies, gt.goalie_id.values)
gt_next = gt[['season', 'goalie_id', 'team', 'shots']].copy()
def goalie_team_sums(rates):                # goals saved per shot -> team totals, this season and next
    per_shot = np.array([rates[S][i] for S, i in zip(gt.season.values, gi)])
    now = pd.Series(per_shot * gt.shots.values).groupby([gt.season, gt.team]).sum()
    prev = np.array([rates[S - 1][i] if S - 1 in rates else 0.0 for S, i in zip(gt.season.values, gi)])
    fut = pd.Series(prev * gt.shots.values).groupby([gt.season, gt.team]).sum()
    fut = fut[fut.index.get_level_values(0) > SEASONS[0]]
    now.index.names = fut.index.names = ['season', 'team']
    return now, fut
def multi_r(sk, g):                         # how well skaters and goalies together track goal difference
    D = pd.concat([sk.rename('sk'), g.rename('g'), gd_actual.rename('gd')], axis=1, join='inner').dropna()
    X = np.column_stack([np.ones(len(D)), D.sk, D.g])
    return np.corrcoef(X @ np.linalg.lstsq(X, D.gd.values, rcond=None)[0], D.gd.values)[0, 1]

rows = {'skaters only, no goalies': {f'{k}_{tag}': multi_r(SK_TEAM[tag][i], SK_TEAM[tag][i] * 0)
                                     for tag in MIN for i, k in ((0, 'describes'), (1, 'predicts'))}}
for name, rates in GCAND.items():
    r_ = {}
    for tag in MIN:
        now, fut = goalie_team_sums(rates[tag])
        r_[f'describes_{tag}'], r_[f'predicts_{tag}'] = multi_r(SK_TEAM[tag][0], now), multi_r(SK_TEAM[tag][1], fut)
    rows[name] = r_
gev = pd.DataFrame(rows).T
for tag in MIN:
    gev[f'score_{tag}'] = ((1 - DESCRIPTIVE_WEIGHT) * gev[f'predicts_{tag}'] / gev[f'predicts_{tag}'].max()
                           + DESCRIPTIVE_WEIGHT * gev[f'describes_{tag}'] / gev[f'describes_{tag}'].max())
gev['score'] = gev[['score_1y', 'score_3y']].mean(axis=1)
G_1Y = GOALIE_1Y if GOALIE_1Y in GCAND else G_DEFAULT
G_3Y = GOALIE_3Y if GOALIE_3Y in GCAND else G_DEFAULT
GOALIE_USED = f"1-year: {G_1Y}; 3-year: {G_3Y}"
print("Goalie value, judged on team goal difference together with the skaters:")
print("  describes = this season's ratings and shots against this season's goal difference")
print("  predicts  = last season's ratings on this season's shots against this season's goal difference")
print(gev.round(4).to_string())
print(f"used for goalie Impact and WAR: {GOALIE_USED}")
print("The team-adjusted rows score lower here by construction: what the team effect takes from a goalie is credited to nobody.")
G_RATES, G_MODEL = {'1y': GCAND[G_1Y]['1y'], '3y': GCAND[G_3Y]['3y']}, GCAND[G_DEFAULT]

# ---------- one row per goalie and season
RATE_SPECS = [('gsax100', 'gsax', 'shots', 100), ('gsax5v5_100', 'gsax_5v5', 'shots_5v5', 100),
              ('gsaxpk_100', 'gsax_pk', 'shots_pk', 100), ('gsaxlow_100', 'gsax_low', 'shots_low', 100),
              ('gsaxmed_100', 'gsax_med', 'shots_med', 100), ('gsaxhigh_100', 'gsax_high', 'shots_high', 100),
              ('reb100', 'reb_saved', 'saves', 100), ('wpa1000', 'wpa', 'shots', 1000)]
out = []
for S in SEASONS:
    row = {'season': S, 'goalie_id': goalies}
    for c in NUMS:
        row[c] = A[S][c]
    row['shots_3y_weighted'] = weighted(S, 'shots')
    for c in ('shots', 'saves', 'shots_5v5', 'shots_pk', 'shots_other'):
        row[f'{c}_typ'] = typical(S, c)
    for name, num, den, k in RATE_SPECS:
        row[f'{name}_1y'] = ratio_of(A[S][num], A[S][den]) * k
        row[f'{name}_3y'] = ratio_of(weighted(S, num), weighted(S, den)) * k
    for tag in MIN:
        row[f'adj100_{tag}'] = G_MODEL[tag][S] * 100
        row[f'rate_{tag}'] = G_RATES[tag][S]
    out.append(pd.DataFrame(row))
gk = pd.concat(out, ignore_index=True)
gk = gk[gk.shots_3y_weighted > 0].reset_index(drop=True)
names = pd.read_csv(f'{DATA_DIR}/players.csv', usecols=['player_id', 'player']).drop_duplicates('player_id').set_index('player_id').player
gk.insert(2, 'player', gk.goalie_id.map(names).fillna(gk.goalie_id.astype(str)))
unnamed = gk.player.str.isdigit()
if unnamed.any():                           # goalies missing from players.csv: take their names from the ice-time files
    try:
        tn = pd.concat([pd.read_csv(fn, usecols=['player_id', 'player']) for fn in glob.glob(f'{DATA_DIR}/toi_*.csv')])
        tn = tn.dropna().drop_duplicates('player_id').set_index('player_id').player
        gk.loc[unnamed, 'player'] = gk.loc[unnamed, 'goalie_id'].map(tn).fillna(gk.loc[unnamed, 'player'])
    except ValueError:
        pass
main_team = gt.sort_values('shots').groupby(['season', 'goalie_id']).team.last().rename('team').reset_index()
gk = gk.merge(main_team, on=['season', 'goalie_id'], how='left')
# the model's ratings are measured from its own baseline: center them on the league's shot-weighted average
for tag, wcol in (('1y', 'shots'), ('3y', 'shots_typ')):
    for c, k in ((f'rate_{tag}', 1), (f'adj100_{tag}', 1)):
        avg = (gk[c] * gk[wcol]).groupby(gk.season).transform('sum') / gk[wcol].groupby(gk.season).transform('sum')
        gk[c] = gk[c] - avg
# goals saved above average, before calibration
gk['gsaa_1y'], gk['gsaa_3y'] = gk.rate_1y * gk.shots, gk.rate_3y * gk.shots_typ
gk['reb_1y'], gk['reb_3y'] = gk.reb_saved, gk.reb100_3y / 100 * gk.saves_typ
print()
print(f"{len(gk):,} goalie-seasons; goals saved above average sum to {gk.gsaa_1y.sum():+.2f} across the league")

## 14. Calibration to real goals, Individualized Impact and WAR

**Is the spread right?** The regression ratings are shrunk toward average on purpose, which makes them reliable but also compresses them: added up by team, they move less than the team's real goal difference does. This section measures that directly. For every team-season it regresses the real goal difference on two totals, the skaters' goals above average and the goalies' goals saved above average. A coefficient of 1 would mean the spread is already right; above 1 means the values are too compressed and by how much.

With `CALIBRATE = True` each group is multiplied by its coefficient, so that team totals reproduce real goal differences one for one. Rankings and percentiles do not change, because every skater is multiplied by the same number. WAR totals do change: this is what sets how many wins the best players are worth. The 1-year and 3-year versions get their own coefficients, since 3-year ratings are shrunk less.

**What the multiplier does and does not mean.** A team's total is known far better than how it divides among its players. The multiplier hands the missing part to each player in proportion to the value the model already gave him. That is the natural assumption, but it is an assumption: set `CALIBRATE = False` to see the values as built.

A second table shows the same regression component by component. It is a diagnostic only: it shows which components are most compressed, but the individual numbers are too noisy to apply.

**Rebound control** is a card row and is left out of WAR by default. The rebound shots a goalie gives up are already inside his teammates' on-ice expected goals against, and on the real data it added nothing to team results beyond that (see "goalie rebounds" in the diagnostic table). `REBOUNDS_IN_WAR = True` in section 1 adds it to goalie WAR at face value.

In [ ]:
# ---------- team totals before calibration: skaters, goalies, real goal difference
gk_i = gk.set_index(['season', 'goalie_id'])
pair = pd.MultiIndex.from_frame(gt[['season', 'goalie_id']])
def goalie_now(tag):                        # goalie goals saved above average, by team, this season
    rate = gk_i[f'rate_{tag}'].reindex(pair).values
    reb = (gk_i[f'reb100_{tag}'].reindex(pair).values / 100 * gt.saves.values) if tag == '3y' else gt.reb_saved.values
    return (pd.Series(rate * gt.shots.values).groupby([gt.season, gt.team]).sum(),
            pd.Series(reb).groupby([gt.season, gt.team]).sum())

SCALE, TEAMTAB = {}, {}
for tag in MIN:
    saves_t, reb_t = goalie_now(tag)
    D = pd.concat([SK_TEAM[tag][0].rename('skaters'), saves_t.rename('goalies'), reb_t.rename('reb'), gd_actual.rename('gd')],
                  axis=1, join='inner').dropna()
    coef = nnls(D[['skaters', 'goalies']].values, D.gd.values)[0]
    fitted_coef = coef.copy()
    if not CALIBRATE:
        coef = np.ones(2)
    coef = np.where(coef > 0, coef, 1.0)        # a group with no measurable link to results is left as built
    SCALE[tag] = dict(sk=float(coef[0]), g=float(coef[1]))
    TEAMTAB[tag] = D
    extra = D.reb * float(REBOUNDS_IN_WAR)
    before = np.corrcoef(D.skaters + D.goalies + extra, D.gd)[0, 1]
    after = np.corrcoef(coef[0] * D.skaters + coef[1] * D.goalies + extra, D.gd)[0, 1]
    print(f"{tag[0]}-year: real goals per model goal: skaters {fitted_coef[0]:.2f}, goalie saves {fitted_coef[1]:.2f}; "
          f"applied {coef[0]:.2f} and {coef[1]:.2f}")
    print(f"        team totals against goal difference: correlation {before:.3f} as built, {after:.3f} with those multipliers")

# diagnostic: the same regression, component by component (1-year)
comp_t = pd.concat([RAW_GAA['1y'][c].groupby([ratings.season, ratings.team]).sum().rename(c) for c in COMPONENTS], axis=1)
s1, r1 = goalie_now('1y')
comp_t = pd.concat([comp_t, s1.rename('goalie saves'), r1.rename('goalie rebounds'), gd_actual.rename('gd')], axis=1, join='inner').dropna()
cols = [c for c in comp_t.columns if c != 'gd' and comp_t[c].std() > 0]
diag = pd.DataFrame({'real_goals_per_model_goal': nnls(comp_t[cols].values, comp_t.gd.values)[0],
                     'spread_across_teams': comp_t[cols].std().values}, index=cols)
print()
print("diagnostic, by component (not applied):")
print(diag.round(2))

# ---------- skaters: apply the calibration, then Impact, replacement level and WAR
for tag, mm in MIN.items():
    for c in COMPONENTS:
        ratings[f'gaa_{c}_{tag}'] = RAW_GAA[tag][c] * SCALE[tag]['sk']
    ratings[f'gaa_{tag}'] = ratings[[f'gaa_{c}_{tag}' for c in COMPONENTS]].sum(axis=1)
    with np.errstate(divide='ignore', invalid='ignore'):
        ratings[f'impact_{tag}'] = np.where(f(mm['all']) > 0, ratings[f'gaa_{tag}'] / f(mm['all']) * 60, np.nan)

played = ratings.min_all_1y.fillna(0) > 0
rank = ratings[played].groupby(['season', 'team', 'pos_group']).min_all_1y.rank(ascending=False, method='first')
limit = ratings.pos_group.map({'F': REPLACEMENT_F, 'D': REPLACEMENT_D})
ratings['replacement_level'] = (played & ratings.team.notna() & (rank.reindex(ratings.index) > limit)).fillna(False)
MCOL = dict(off='off', pp='pp', pk='pk', shoot='all', setup='all', pen='all')
MCOL['def'] = 'off'
repl = ratings[ratings.replacement_level]
REPL = {}
for g in ('F', 'D'):
    d = repl[repl.pos_group == g]
    REPL[g] = {c: d[f'gaa_{c}_1y'].sum() / max(d[MIN['1y'][MCOL[c]]].fillna(0).sum(), 1) for c in COMPONENTS}   # goals per minute
print()
print(f"replacement-level skaters: {int(ratings.replacement_level.sum()):,} player-seasons "
      f"({ratings.replacement_level.sum() / played.sum():.0%} of those who played)")
print("replacement baseline, goals per 60 minutes of that component's ice time:")
print((pd.DataFrame(REPL) * 60).round(3))

for tag, mm in MIN.items():
    gar = 0
    for c in COMPONENTS:
        base = ratings.pos_group.map({g: REPL[g][c] for g in REPL}).fillna(0)
        part = ratings[f'gaa_{c}_{tag}'] - base * ratings[mm[MCOL[c]]].fillna(0)
        ratings[f'war_{c}_{tag}'] = part / GPW                   # this component's share of WAR
        gar = gar + part
    ratings[f'gar_{tag}'], ratings[f'war_{tag}'] = gar, gar / GPW

for tag, col, cut in (('1y', 'min_all_1y', MIN_ALL_1Y), ('3y', 'min_all_3y_weighted', MIN_ALL_3Y)):
    ok = ratings[col] >= cut * season_share(ratings, tag)
    for m in ('impact', 'war'):
        ratings[f'{m}_{tag}_pct'] = ratings[ok].groupby(['season', 'pos_group'])[f'{m}_{tag}'].rank(pct=True).mul(100).round(0)
# 3-year percentiles only among players active this season
ratings['active'] = ratings.min_5v5_1y.fillna(0) >= MIN_CURRENT_5V5 * season_share(ratings, '1y')
for col in [c for c in ratings.columns if c.endswith('_3y_pct')]:
    base = {'pen_3y': 'pen_diff60_3y'}.get(col[:-4], col[:-4])
    ok = ratings[col].notna() & ratings.active
    ratings[col] = ratings[ok].groupby(['season', 'pos_group'])[base].rank(pct=True).mul(100).round(0)
for q in ('qualified_3y', 'prod_qualified_3y'):
    ratings[q] = ratings[q] & ratings.active
print()
print(f"skaters with a 3-year rating but under {MIN_CURRENT_5V5} 5v5 minutes this season are kept in the file "
      f"and left out of the 3-year percentiles: {int((~ratings.active & (ratings.season == SEASONS[-1])).sum())} in {SEASONS[-1]}")

In [ ]:
# ---------- goalies: apply the calibration, then Impact, replacement level and WAR
SHOTS_PER_GAME = len(faced) / (2 * len(res_games))                 # unblocked shots a team faces in a game
SH = {'1y': dict(shots='shots', s5='shots_5v5', spk='shots_pk', soth='shots_other'),
      '3y': dict(shots='shots_typ', s5='shots_5v5_typ', spk='shots_pk_typ', soth='shots_other_typ')}
for tag in MIN:
    k = SCALE[tag]['g']
    gk[f'gaa_saves_{tag}'] = gk[f'gsaa_{tag}'] * k
    gk[f'gaa_reb_{tag}'] = gk[f'reb_{tag}'] * float(REBOUNDS_IN_WAR)      # already in goals: no multiplier
    gk[f'gaa_{tag}'] = gk[f'gaa_saves_{tag}'] + gk[f'gaa_reb_{tag}']
    gk[f'impact_{tag}'] = ratio_of(gk[f'gaa_{tag}'].values, gk[SH[tag]['shots']].values) * SHOTS_PER_GAME

rank = gk[gk.shots > 0].groupby(['season', 'team']).shots.rank(ascending=False, method='first')
gk['replacement_level'] = ((gk.shots > 0) & (rank.reindex(gk.index) > REPLACEMENT_G)).fillna(False)
rg = gk[gk.replacement_level]
G_BASE = rg.gaa_1y.sum() / max(rg.shots.sum(), 1)                  # goals saved per shot by replacement goalies
print(f"replacement-level goalies: {int(gk.replacement_level.sum())} goalie-seasons; "
      f"baseline {G_BASE * SHOTS_PER_GAME:+.3f} goals per game's worth of shots")

for tag in MIN:
    k, sh = SCALE[tag]['g'], SH[tag]
    gk[f'gar_{tag}'] = gk[f'gaa_{tag}'] - G_BASE * gk[sh['shots']]
    gk[f'war_{tag}'] = gk[f'gar_{tag}'] / GPW
    raw = {}
    for part, rate, shots in (('5v5', f'gsax5v5_100_{tag}', sh['s5']), ('pk', f'gsaxpk_100_{tag}', sh['spk'])):
        raw[part] = gk[rate] / 100 * gk[shots] * k
        gk[f'war_{part}_{tag}'] = (raw[part] - G_BASE * gk[shots]) / GPW
    other_raw = gk[f'gsax100_{tag}'] / 100 * gk[sh['shots']] * k - raw['5v5'] - raw['pk']
    gk[f'war_other_{tag}'] = (other_raw - G_BASE * gk[sh['soth']]) / GPW
    gk[f'war_adj_{tag}'] = (gk[f'gaa_saves_{tag}'] - gk[f'gsax100_{tag}'] / 100 * gk[sh['shots']] * k) / GPW   # team and shooter adjustment, shrinkage
    gk[f'war_reb_{tag}'] = gk[f'gaa_reb_{tag}'] / GPW

gk['active'] = gk.shots >= MIN_G_CURRENT * season_share(gk, '1y')
gk['qualified_1y'] = gk.shots >= MIN_G_1Y * season_share(gk, '1y')
gk['qualified_3y'] = (gk.shots_3y_weighted >= MIN_G_3Y) & gk.active
G_PCT = ['war', 'impact', 'adj100', 'gsax100', 'gsax5v5_100', 'gsaxpk_100', 'gsaxlow_100', 'gsaxmed_100', 'gsaxhigh_100',
         'reb100', 'wpa1000']
for tag in MIN:
    ok = gk[f'qualified_{tag}']
    for m in G_PCT:
        gk[f'{m}_{tag}_pct'] = gk[ok].groupby('season')[f'{m}_{tag}'].rank(pct=True).mul(100).round(0)

# ---------- team WAR: skaters plus goalies
share = gt.merge(gk[['season', 'goalie_id', 'war_1y', 'shots']].rename(columns={'shots': 'season_shots'}), on=['season', 'goalie_id'])
share = share.merge(gk[['season', 'goalie_id', 'gaa_1y']], on=['season', 'goalie_id'])
part = ratio_of(share.shots.values, share.season_shots.values)          # a traded goalie is split by shots faced
share['war'], share['gaa'] = share.war_1y * part, share.gaa_1y * part
sk_t = ratings.groupby(['season', 'team']).agg(skater_war=('war_1y', 'sum'), skater_gaa=('gaa_1y', 'sum')).reset_index()
g_t = share.groupby(['season', 'team']).agg(goalie_war=('war', 'sum'), goalie_gaa=('gaa', 'sum')).reset_index()
team_war = team_res.merge(sk_t, on=['season', 'team'], how='left').merge(g_t, on=['season', 'team'], how='left')
WCOLS = ['skater_war', 'goalie_war', 'skater_gaa', 'goalie_gaa']
team_war[WCOLS] = team_war[WCOLS].fillna(0)
team_war['total_war'] = team_war.skater_war + team_war.goalie_war

print()
print("league totals by season:")
tot = team_war.groupby('season').agg(games=('gp', 'sum'), wins=('wins', 'sum'), skater_war=('skater_war', 'sum'),
                                     goalie_war=('goalie_war', 'sum'), total_war=('total_war', 'sum'))
tot['replacement_team_win_pct'] = (tot.wins - tot.total_war) / tot.games
print(tot.round(3))
print()
print("SPREAD CHECK")
lat = ratings[ratings.season == SEASONS[-1]]
print(f"  skater WAR this season: best {lat.war_1y.max():.1f}, 10th best {lat.war_1y.nlargest(10).iloc[-1]:.1f}, "
      f"median regular {lat[~lat.replacement_level & (lat.min_all_1y > 0)].war_1y.median():.1f}")
print(f"  with no multiplier the best would be about {lat.war_1y.max() / SCALE['1y']['sk']:.1f}")
glat = gk[(gk.season == SEASONS[-1]) & (gk.shots > 0)]
print(f"  goalie WAR this season: best {glat.war_1y.max():.1f}, worst {glat.war_1y.min():.1f}")
print(f"  team WAR against winning percentage: skaters only {team_war.skater_war.div(team_war.gp).corr(team_war.win_pct):.3f}, "
      f"with goalies {team_war.total_war.div(team_war.gp).corr(team_war.win_pct):.3f}")

## 15. Checks

**Leaders.** The top of each list for the latest season should be recognizable. If it is not, something is wrong.

**Repeatability.** How well does each number predict the same player's 1-year isolated rating the following season? Higher is better. The 3-year rating should beat the 1-year rating, and offense is normally more repeatable than defense. Raw on-ice rates often look more repeatable than isolated ones because they also carry the player's team, which mostly stays the same; that is the part the regression removes on purpose.

In [ ]:
L = SEASONS[-1]
cur = ratings[(ratings.season == L) & ratings.qualified_3y]
show = ['player', 'team', 'gp', 'min_5v5_1y', 'xgf60_3y', 'xgf60_zone_adj_3y', 'off_3y', 'off_3y_pct', 'off_1y']
for grp, name in (('F', 'forwards'), ('D', 'defensemen')):
    print(f"\nTop 5v5 offense, {name}, 3-year rating ending {L}-{str(L + 1)[2:]}")
    print(cur[cur.pos_group == grp].nlargest(15, 'off_3y')[show].round(2).to_string(index=False))
show_d = ['player', 'team', 'gp', 'min_5v5_1y', 'xga60_3y', 'xga60_zone_adj_3y', 'def_3y', 'def_3y_pct', 'def_1y']
for grp, name in (('F', 'forwards'), ('D', 'defensemen')):
    print(f"\nTop 5v5 defense, {name}, 3-year rating ending {L}-{str(L + 1)[2:]}")
    print(cur[cur.pos_group == grp].nlargest(15, 'def_3y')[show_d].round(2).to_string(index=False))

In [ ]:
nxt = ratings[ratings.qualified_1y][['season', 'player_id', 'off_1y', 'def_1y']].copy()
nxt['season'] -= 1
pair = ratings[ratings.qualified_1y].merge(nxt, on=['season', 'player_id'], suffixes=('', '_next'))
rep = pd.DataFrame({
    'offense': {'raw on-ice, 1-year': pair.xgf60_1y.corr(pair.off_1y_next),
                'isolated, 1-year': pair.off_1y.corr(pair.off_1y_next),
                'isolated, 3-year': pair.off_3y.corr(pair.off_1y_next)},
    'defense': {'raw on-ice, 1-year': (-pair.xga60_1y).corr(pair.def_1y_next),
                'isolated, 1-year': pair.def_1y.corr(pair.def_1y_next),
                'isolated, 3-year': pair.def_3y.corr(pair.def_1y_next)}})
print(f"correlation with next season's 1-year isolated rating ({len(pair):,} player-season pairs)")
print(rep.round(3))

**Finishing, Playmaking and Production.** `goals`, `ixg` (the player's own expected goals) and `gax` (goals above expected) are this season's totals, with empty-net situations left out. The last table shows how well each score predicts the plain version of itself next season, against the plain version this season. Finishing is the hardest of the three to predict: goals above expected is mostly luck in any one season, so expect low numbers there.

In [ ]:
cur = ratings[(ratings.season == L) & ratings.prod_qualified_3y]
shows = {'fin':  ['player', 'team', 'gp', 'goals', 'ixg', 'gax', 'gax60_1y', 'fin_1y', 'fin_3y', 'fin_3y_pct'],
         'play': ['player', 'team', 'gp', 'a1', 'a2', 'a60_1y', 'play_1y', 'play_3y', 'play_3y_pct'],
         'prod': ['player', 'team', 'gp', 'points', 'goals', 'a1', 'a2', 'pp_points', 'pts60_1y', 'prod_1y', 'prod_3y', 'prod_3y_pct']}
for name, b in BLOCKS.items():
    for grp, gname in (('F', 'forwards'), ('D', 'defensemen')):
        print(f"\nTop {b['title']}, {gname}, 3-year score ending {L}-{str(L + 1)[2:]}")
        print(cur[cur.pos_group == grp].nlargest(12, f'{name}_3y')[shows[name]].round(2).to_string(index=False))

q = ratings[ratings.prod_qualified_1y]
rows = {}
for name, b in BLOCKS.items():
    plain = b['plain'] + '_1y'
    nxt = q[['season', 'player_id', plain]].copy(); nxt['season'] -= 1
    pair = q.merge(nxt, on=['season', 'player_id'], suffixes=('', '_next'))
    for grp, gname in (('F', 'forwards'), ('D', 'defensemen')):
        d = pair[pair.pos_group == grp]
        rows[(b['title'], gname)] = {'plain rate, 1-year': d[plain].corr(d[plain + '_next']),
                                     'score, 1-year': d[f'{name}_1y'].corr(d[plain + '_next']),
                                     'score, 3-year': d[f'{name}_3y'].corr(d[plain + '_next'])}
print(f"\ncorrelation with next season's plain rate (goals above expected, assists, points per 60)")
print(pd.DataFrame(rows).T.round(3))

**Teammates, Competition and Penalties.** The names at the top should make sense: top-line players and shutdown defensemen facing the toughest competition, stars' linemates having the best teammates, and known agitators at both ends of the penalty list. The repeatability table shows how well each number predicts itself next season.

In [ ]:
cur = ratings[(ratings.season == L) & ratings.qualified_3y]
for col, title in (('competition_3y', 'Toughest competition'), ('teammates_3y', 'Best teammates')):
    for grp, name in (('F', 'forwards'), ('D', 'defensemen')):
        print(f"\n{title}, {name}, 3-year ending {L}-{str(L + 1)[2:]}")
        print(cur[cur.pos_group == grp].nlargest(8, col)[['player', 'team', 'min_5v5_1y', 'teammates_3y', 'competition_3y',
              'teammates_3y_pct', 'competition_3y_pct']].round(3).to_string(index=False))
cur = ratings[(ratings.season == L) & ratings.prod_qualified_3y]
pcols = ['player', 'team', 'gp', 'pen_drawn', 'pen_taken', 'pen_diff60_1y', 'pen_diff60_3y', 'pen_3y_pct']
print(f"\nBest penalty differential, 3-year ending {L}-{str(L + 1)[2:]}")
print(cur.nlargest(10, 'pen_diff60_3y')[pcols].round(2).to_string(index=False))
print("\nWorst penalty differential")
print(cur.nsmallest(10, 'pen_diff60_3y')[pcols].round(2).to_string(index=False))

def next_corr(col1, col3, q):
    d = ratings[ratings[q]]
    nxt = d[['season', 'player_id', col1]].copy(); nxt['season'] -= 1
    p = d.merge(nxt, on=['season', 'player_id'], suffixes=('', '_next'))
    return {'1-year': p[col1].corr(p[col1 + '_next']), '3-year': p[col3].corr(p[col1 + '_next'])}
print("\ncorrelation with the same number next season (1-year version)")
print(pd.DataFrame({'teammates': next_corr('teammates_1y', 'teammates_3y', 'qualified_1y'),
                    'competition': next_corr('competition_1y', 'competition_3y', 'qualified_1y'),
                    'penalties': next_corr('pen_diff60_1y', 'pen_diff60_3y', 'prod_qualified_1y')}).round(3))

**Special teams, Impact and WAR.**

- **Team check.** Adding up every player's WAR by team, skaters and goalies, should track how the team actually did. This is the most important test of the whole framework. It will not be perfect: most skater components are expected goals, not goals, and luck is real.
- **Leaders** for the latest season, with each component's goals above average.
- **Repeatability.** How well each number predicts the 1-year version next season.

In [ ]:
tw = team_war.copy()
tw['gd'], tw['gaa'] = tw.gf - tw.ga, tw.skater_gaa + tw.goalie_gaa
print(f"TEAM CHECK ({len(tw)} team-seasons)")
print(f"  correlation with winning percentage: skater WAR {tw.skater_war.div(tw.gp).corr(tw.win_pct):.3f}, "
      f"goalie WAR {tw.goalie_war.div(tw.gp).corr(tw.win_pct):.3f}, both {tw.total_war.div(tw.gp).corr(tw.win_pct):.3f}")
print(f"  correlation of summed goals above average with goal difference: {tw.gaa.corr(tw.gd):.3f}")
print(f"  one goal above average is worth {np.polyfit(tw.gaa, tw.gd, 1)[0]:.2f} goals of actual goal difference (1.00 = the spread is right)")
cur_t = tw[tw.season == L].sort_values('total_war', ascending=False)
print()
print(f"Teams by WAR, {L}-{str(L + 1)[2:]}")
print(cur_t[['team', 'skater_war', 'goalie_war', 'total_war', 'gaa', 'gd', 'win_pct']].round(2).to_string(index=False))

cur = ratings[(ratings.season == L) & ratings.prod_qualified_1y]
cols = ['player', 'team', 'gp', 'war_1y', 'war_3y', 'impact_3y_pct', 'gaa_off_1y', 'gaa_def_1y', 'gaa_pp_1y', 'gaa_pk_1y',
        'gaa_shoot_1y', 'gaa_setup_1y', 'gaa_pen_1y']
for grp, name in (('F', 'forwards'), ('D', 'defensemen')):
    print()
    print(f"Top WAR, {name}, {L}-{str(L + 1)[2:]}")
    print(cur[cur.pos_group == grp].nlargest(15, 'war_1y')[cols].round(2).to_string(index=False))
rs_cols = ['player', 'team', 'goals', 'ixg', 'shots_1y', 'tm_shots_1y', 'shoot_3y', 'setup_3y', 'gaa_shoot_1y', 'gaa_setup_1y']
curq = ratings[(ratings.season == L) & ratings.prod_qualified_3y]
print()
print("Top shooting results (own shots), 3-year")
print(curq.nlargest(12, 'shoot_3y')[rs_cols].round(3).to_string(index=False))
print()
print("Top setup results (teammates' shots while he is on the ice), 3-year")
print(curq.nlargest(12, 'setup_3y')[rs_cols].round(3).to_string(index=False))
st_cols = ['player', 'team', 'pp_min_1y', 'pp_off_3y', 'pp_off_3y_pct', 'pk_min_1y', 'pk_def_3y', 'pk_def_3y_pct']
cur3 = ratings[ratings.season == L]
print()
print("Top power-play offense, 3-year")
print(cur3[cur3.pp_off_3y_pct.notna()].nlargest(12, 'pp_off_3y')[st_cols].round(2).to_string(index=False))
print()
print("Top penalty-kill defense, 3-year")
print(cur3[cur3.pk_def_3y_pct.notna()].nlargest(12, 'pk_def_3y')[st_cols].round(2).to_string(index=False))

def next_corr2(col1, col3, mask):
    d = ratings[mask]
    nxt = d[['season', 'player_id', col1]].copy(); nxt['season'] -= 1
    p = d.merge(nxt, on=['season', 'player_id'], suffixes=('', '_next'))
    if len(p) < 3:
        return {'1-year': np.nan, '3-year': np.nan, 'pairs': len(p)}
    return {'1-year': p[col1].corr(p[col1 + '_next']), '3-year': p[col3].corr(p[col1 + '_next']), 'pairs': len(p)}
print()
print("correlation with the 1-year version next season")
print(pd.DataFrame({'impact': next_corr2('impact_1y', 'impact_3y', ratings.prod_qualified_1y),
                    'war': next_corr2('war_1y', 'war_3y', ratings.prod_qualified_1y),
                    'shooting': next_corr2('shoot_1y', 'shoot_3y', ratings.prod_qualified_1y),
                    'setup': next_corr2('setup_1y', 'setup_3y', ratings.prod_qualified_1y),
                    'pp offense': next_corr2('pp_off_1y', 'pp_off_3y', ratings.pp_min_1y >= MIN_PP_1Y),
                    'pk defense': next_corr2('pk_def_1y', 'pk_def_3y', ratings.pk_min_1y >= MIN_PP_1Y)}).round(3))

**Goalies.** Leaders for the latest season, then how well each number predicts the same goalie's 1-year version next season. Goalie results are the noisiest thing in hockey: expect low numbers, with the 3-year versions and the team-adjusted rating doing best.

In [ ]:
gcur = gk[(gk.season == L) & gk.qualified_1y]
gcols = ['player', 'team', 'games', 'shots', 'war_1y', 'war_3y', 'impact_1y', 'gsax', 'adj100_1y', 'gsax_5v5', 'gsax_pk',
         'gsax_low', 'gsax_med', 'gsax_high', 'reb_saved', 'wpa']
print(f"Goalies by WAR, {L}-{str(L + 1)[2:]}  (gsax columns are season totals in goals; adj100 is the team-adjusted rating per 100 shots)")
print(gcur.sort_values('war_1y', ascending=False)[gcols].round(2).to_string(index=False))

gq = gk[gk.qualified_1y]
rows = {}
for m, label in (('war', 'WAR'), ('gsax100', 'GSAx per 100 shots'), ('adj100', 'team-adjusted per 100 shots'),
                 ('gsax5v5_100', '5v5 GSAx'), ('gsaxpk_100', 'GSAx against the power play'), ('gsaxlow_100', 'low danger'),
                 ('gsaxmed_100', 'medium danger'), ('gsaxhigh_100', 'high danger'), ('reb100', 'rebound control'),
                 ('wpa1000', 'win probability added')):
    nx = gq[['season', 'goalie_id', f'{m}_1y']].copy(); nx['season'] -= 1
    p_ = gq.merge(nx, on=['season', 'goalie_id'], suffixes=('', '_next'))
    rows[label] = {'1-year': p_[f'{m}_1y'].corr(p_[f'{m}_1y_next']) if len(p_) > 2 else np.nan,
                   '3-year': p_[f'{m}_3y'].corr(p_[f'{m}_1y_next']) if len(p_) > 2 else np.nan, 'pairs': len(p_)}
print()
print("correlation with the same goalie's 1-year number next season")
print(pd.DataFrame(rows).T.round(3))

## 16. Save

`player_ratings.csv`: one row per skater per season, with the 1-year and 3-year 5v5 ratings (xG per 60), raw and zone-adjusted on-ice rates, Finishing, Playmaking and Production scores, goal, assist and expected-goal totals, teammate and competition quality, penalty differential, power-play and penalty-kill ratings, shooting and setup results, Impact, WAR and each component's goals above average, percentiles, minutes, games, team and position. The Finishing, Playmaking and Production weights are saved alongside as `scoring_weights.csv`.

`goalie_ratings.csv`: one row per goalie per season, with WAR, Impact, the team-adjusted rating, every GSAx split (totals in goals, rates per 100 shots, 1-year and 3-year), rebound control, win probability added, percentiles among goalies, and each part's share of WAR.

`team_war.csv`: one row per team per season, with results and the summed WAR of its skaters and goalies. `team_results.csv` is the same without the WAR columns.

In [ ]:
first = ['season', 'player_id', 'player', 'position', 'pos_group', 'team', 'gp', 'min_5v5_1y', 'min_5v5_3y_weighted',
         'seasons_in_3y', 'qualified_1y', 'qualified_3y', 'war_1y', 'war_3y', 'impact_1y', 'impact_3y',
         'impact_1y_pct', 'impact_3y_pct', 'war_1y_pct', 'war_3y_pct', 'off_3y', 'def_3y', 'off_3y_pct', 'def_3y_pct',
         'off_1y', 'def_1y', 'off_1y_pct', 'def_1y_pct', 'fin_3y', 'fin_3y_pct', 'play_3y', 'play_3y_pct',
         'fin_1y', 'fin_1y_pct', 'play_1y', 'play_1y_pct', 'prod_3y', 'prod_3y_pct', 'prod_1y', 'prod_1y_pct',
         'points', 'goals', 'a1', 'a2', 'teammates_3y_pct', 'competition_3y_pct', 'pen_3y_pct',
         'teammates_3y', 'competition_3y', 'pen_diff60_3y', 'pen_taken', 'pen_drawn',
         'pp_off_3y', 'pp_off_3y_pct', 'pk_def_3y', 'pk_def_3y_pct', 'pp_off_1y', 'pk_def_1y']
ratings = ratings[first + [c for c in ratings.columns if c not in first]]
ratings['season_done'] = ratings.season.map(SEASON_DONE)
ratings.to_csv(f'{DATA_DIR}/player_ratings.csv', index=False)
pd.concat({b['title']: b['wt'] for b in BLOCKS.values()}).to_csv(f'{DATA_DIR}/scoring_weights.csv')
print(f"saved {DATA_DIR}/player_ratings.csv: {len(ratings):,} rows, {ratings.shape[1]} columns")
for b in BLOCKS.values():
    print(f"{b['title']}: 3-year weights 1 / {b['w1']:g} / {b['w2']:g}")
print(f"Penalties: 3-year weights 1 / {QW1:g} / {QW2:g}")
print(f"Special teams: 3-year weights 1 / {SW1:g} / {SW2:g}, lambda {LAM_ST_3Y:g}; 1-year lambda {LAM_ST_1Y:g}")
print(f"Results component in Impact and WAR: {RESULTS_USED}")
print(f"Goals per win {GPW:.3f}; goal value of a minor {PEN_VALUE:.3f}")
team_res.to_csv(f'{DATA_DIR}/team_results.csv', index=False)
gfirst = ['season', 'goalie_id', 'player', 'team', 'games', 'shots', 'qualified_1y', 'qualified_3y', 'war_1y', 'war_3y',
          'impact_1y', 'impact_3y', 'war_1y_pct', 'war_3y_pct', 'impact_1y_pct', 'impact_3y_pct', 'gaa_1y', 'gaa_3y',
          'gsax', 'gsax_5v5', 'gsax_pk', 'gsax_other', 'gsax_low', 'gsax_med', 'gsax_high', 'reb_saved', 'wpa']
gk = gk[gfirst + [c for c in gk.columns if c not in gfirst]]
gk['season_done'], team_war['season_done'] = gk.season.map(SEASON_DONE), team_war.season.map(SEASON_DONE)
gk.to_csv(f'{DATA_DIR}/goalie_ratings.csv', index=False)
team_war.to_csv(f'{DATA_DIR}/team_war.csv', index=False)
print(f"saved {DATA_DIR}/goalie_ratings.csv: {len(gk):,} rows, {gk.shape[1]} columns; and team_war.csv")
print(f"Goalie value in Impact and WAR: {GOALIE_USED}; team effect shrunk {G_RATIO:g}x; 3-year weights 1 / {GW1:g} / {GW2:g}")
print(f"Calibration multipliers (skaters, goalies): 1-year {SCALE['1y']['sk']:.2f}, {SCALE['1y']['g']:.2f}; "
      f"3-year {SCALE['3y']['sk']:.2f}, {SCALE['3y']['g']:.2f}")
print(f"3-year weights 1 / {W1:g} / {W2:g}, lambda {LAM_3Y:g}; 1-year lambda {LAM_1Y:g}")

## 17. Optional: one team's report

Set `TEAM` and run. It reads the saved `player_ratings.csv`, so it works any time after section 16 has run once (in a fresh session, run the two cells of section 1 first).

For every skater on the team: WAR this season and the 3-year typical-season WAR, then league percentiles within forwards or defensemen for Impact and each card row. 100 is best. A dash means the player is not ranked on that row: too few minutes there, or under the current-season minutes rule. The team's goalies follow, with percentiles among goalies.

`Compet` and `Mates` are context, not skill: a high number means tougher opponents or better linemates. A player is listed under the team he played the most games for that season.

In [ ]:
TEAM = 'WSH'          # team abbreviation
VERSION = '3y'        # '3y' = current-value ratings, '1y' = this season only
SEASON = None         # None = latest season in the file, or a start year such as 2024

r = pd.read_csv(f'{DATA_DIR}/player_ratings.csv')
yr = int(r.season.max()) if SEASON is None else SEASON
t = r[(r.season == yr) & (r.team == TEAM)]
v = VERSION
table = pd.DataFrame({
    'player': t.player, 'pos': t.position, 'gp': t.gp,
    'WAR_1y': t.war_1y.round(2), 'WAR_3y': t.war_3y.round(2),
    'Impact': t[f'impact_{v}_pct'], '5v5 Off': t[f'off_{v}_pct'], '5v5 Def': t[f'def_{v}_pct'],
    'PP': t[f'pp_off_{v}_pct'], 'PK': t[f'pk_def_{v}_pct'], 'Finish': t[f'fin_{v}_pct'],
    'Playmk': t[f'play_{v}_pct'], 'Penalt': t[f'pen_{v}_pct'],
    'Compet': t[f'competition_{v}_pct'], 'Mates': t[f'teammates_{v}_pct'], 'grp': t.pos_group})
pct_cols = ['Impact', '5v5 Off', '5v5 Def', 'PP', 'PK', 'Finish', 'Playmk', 'Penalt', 'Compet', 'Mates']
table[pct_cols + ['gp']] = table[pct_cols + ['gp']].astype('Int64').astype(str).replace('<NA>', '-')
print(f"{TEAM} {yr}-{str(yr + 1)[2:]}: WAR, then league percentiles within position ({'3-year' if v == '3y' else '1-year'} versions)")
print(f"summed skater WAR this season: {t.war_1y.sum():.1f}")
for grp, name in (('F', 'Forwards'), ('D', 'Defensemen')):
    print()
    print(name)
    print(table[table.grp == grp].drop(columns='grp').sort_values('WAR_1y', ascending=False)
          .to_string(index=False, na_rep='-'))

import os
if os.path.exists(f'{DATA_DIR}/goalie_ratings.csv'):
    g = pd.read_csv(f'{DATA_DIR}/goalie_ratings.csv')
    g = g[(g.season == yr) & (g.team == TEAM) & (g.shots > 0)]
    gtab = pd.DataFrame({
        'player': g.player, 'gp': g.games, 'shots': g.shots, 'GSAx': g.gsax.round(1),
        'WAR_1y': g.war_1y.round(2), 'WAR_3y': g.war_3y.round(2),
        'Impact': g[f'impact_{v}_pct'], 'TeamAdj': g[f'adj100_{v}_pct'], '5v5': g[f'gsax5v5_100_{v}_pct'],
        'vs PP': g[f'gsaxpk_100_{v}_pct'], 'Low': g[f'gsaxlow_100_{v}_pct'], 'Med': g[f'gsaxmed_100_{v}_pct'],
        'High': g[f'gsaxhigh_100_{v}_pct'], 'Rebound': g[f'reb100_{v}_pct'], 'WPA': g[f'wpa1000_{v}_pct']})
    gpct = ['gp', 'shots', 'Impact', 'TeamAdj', '5v5', 'vs PP', 'Low', 'Med', 'High', 'Rebound', 'WPA']
    gtab[gpct] = gtab[gpct].round(0).astype('Int64').astype(str).replace('<NA>', '-')
    print()
    print("Goalies (GSAx = goals saved above expected this season; the rest are percentiles among goalies)")
    print(gtab.sort_values('WAR_1y', ascending=False).to_string(index=False, na_rep='-'))
    print()
    print(f"summed goalie WAR this season: {g.war_1y.sum():.1f}   skaters and goalies together: {t.war_1y.sum() + g.war_1y.sum():.1f}")

## 18. Optional: team WAR against results

One dot per team-season: the WAR of every skater and goalie added up, against how the team did. It reads `team_war.csv`, which section 16 saves, and saves the chart to Drive as `team_war_vs_win_pct.png` (the name follows `Y`). The ringed dots are the ten biggest misses.

- `Y` chooses the vertical axis: `'win_pct'` (winning percentage), `'points'` (standings points), `'points_pct'` (points percentage) or `'wins'`. The two points options download the final standings from the NHL the first time and keep them in `team_points.csv`.
- With a percentage on the vertical axis, WAR is shown on an 82-game pace so the two shortened seasons line up. For `'wins'` and `'points'`, `PER_82 = True` does the same.
- `WHO` set to `'skaters'` or `'goalies'` shows one group alone.

In [ ]:
import os
import numpy as np, pandas as pd, matplotlib.pyplot as plt

Y = 'win_pct'         # 'win_pct' = winning percentage, 'points' = standings points, 'points_pct' = points percentage,
                      # 'wins' = wins. The two points options download the final standings from the NHL once.
PER_82 = False        # for 'wins' and 'points': True = scale every team-season to an 82-game pace
N_LABELS = 10         # how many of the biggest misses to label
WHO = 'both'          # 'both' = skaters and goalies, 'skaters' or 'goalies' = that group alone

t = pd.read_csv(f'{DATA_DIR}/team_war.csv')          # saved by the ratings notebook: results and summed WAR per team-season
if 'season_done' in t.columns:
    t = t[t.season_done >= 1].copy()                  # a season in progress is left off the chart
t['war'] = {'both': t.total_war, 'skaters': t.skater_war, 'goalies': t.goalie_war}[WHO]
NAME = {'both': 'Team WAR (skaters and goalies)', 'skaters': 'Team skater WAR', 'goalies': 'Team goalie WAR'}[WHO]
XLAB = {'both': 'Summed WAR of skaters and goalies', 'skaters': 'Summed skater WAR', 'goalies': 'Summed goalie WAR'}[WHO]

def standings_points(seasons):
    # final regular-season standings from the NHL, kept in team_points.csv so they are only downloaded once
    cache = f'{DATA_DIR}/team_points.csv'
    if os.path.exists(cache):
        saved = pd.read_csv(cache)
        if set(seasons) <= set(saved.season):
            return saved
    import requests
    listing = requests.get('https://api-web.nhle.com/v1/standings-season', timeout=30).json()['seasons']
    last_day = {int(str(x['id'])[:4]): x['standingsEnd'] for x in listing}
    rows = []
    for y in sorted(set(seasons)):
        for r in requests.get(f'https://api-web.nhle.com/v1/standings/{last_day[y]}', timeout=30).json()['standings']:
            rows.append({'season': y, 'team': r['teamAbbrev']['default'], 'points': r['points'], 'points_gp': r['gamesPlayed']})
    saved = pd.DataFrame(rows)
    saved.to_csv(cache, index=False)
    return saved

if Y in ('points', 'points_pct'):
    try:
        n_before = len(t)
        t = t.merge(standings_points(t.season.unique()), on=['season', 'team'])
        if len(t) < n_before:
            print(f"note: {n_before - len(t)} team-seasons had no match in the NHL standings and are left out")
        t['points_pct'] = t.points / (2 * t.points_gp)
    except Exception as err:
        print(f"Could not get the standings from the NHL ({type(err).__name__}). Showing winning percentage instead.")
        Y = 'win_pct'
RATE = Y in ('win_pct', 'points_pct')                 # a rate on the y axis needs WAR as a rate too
if RATE or PER_82:
    t['war'] = t.war / t.gp * 82
t['y'] = t[Y] * (82 / (t.points_gp if Y == 'points' else t.gp) if PER_82 and not RATE else 1)
YNAME = {'win_pct': 'winning percentage', 'points': 'standings points', 'points_pct': 'points percentage', 'wins': 'wins'}[Y]
slope, intercept = np.polyfit(t.war, t.y, 1)
t['miss'] = t.y - (intercept + slope * t.war)
corr = t.war.corr(t.y)

INK, INK2, MUTED, GRID, SURFACE, BLUE = '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#fcfcfb', '#2a78d6'
fig, ax = plt.subplots(figsize=(9, 7), dpi=110, facecolor=SURFACE)
ax.set_facecolor(SURFACE)
xs = np.array([t.war.min(), t.war.max()])
ax.plot(xs, intercept + slope * xs, color=INK2, lw=1.5, zorder=2)
ax.scatter(t.war, t.y, s=70, color=BLUE, edgecolors=SURFACE, linewidths=1.5, alpha=0.9, zorder=3)
ax.margins(x=0.07, y=0.10)                      # room at the edges so nothing is cut off

# --- labels: each one goes in the nearest free spot that does not cover a dot or another label
fig.canvas.draw()
px, FS = fig.dpi / 72, 8
dots = ax.transData.transform(t[['war', 'y']].values)
left, bottom, right, top = ax.bbox.extents
placed, dot_r = [], 6 * px
def free(b):
    if b[0] < left or b[1] < bottom or b[2] > right or b[3] > top:
        return False
    if ((dots[:, 0] > b[0] - dot_r) & (dots[:, 0] < b[2] + dot_r) & (dots[:, 1] > b[1] - dot_r) & (dots[:, 1] < b[3] + dot_r)).any():
        return False
    return not any(b[0] < q[2] and q[0] < b[2] and b[1] < q[3] and q[1] < b[3] for q in placed)

labelled = t.reindex(t.miss.abs().sort_values(ascending=False).index).head(N_LABELS)
ax.scatter(labelled.war, labelled.y, s=70, color=BLUE, edgecolors=INK, linewidths=1.3, zorder=4)   # ring = has a label
for _, row in labelled.iterrows():
    text = f"{row.team} {int(row.season) % 100:02d}-{(int(row.season) + 1) % 100:02d}"
    w, h = 0.58 * FS * len(text) * px, 1.3 * FS * px
    x0, y0 = ax.transData.transform((row.war, row.y))
    up = 1 if row.miss > 0 else -1               # prefer the side away from the line
    dirs = [(0, up), (1, up), (-1, up), (1, 0), (-1, 0), (0, -up), (1, -up), (-1, -up)]
    spot = None
    for dist in (9, 16, 24, 34, 46):
        for dx, dy in dirs:
            cx = x0 + dx * (dist * px * (0.7 if dy else 1) + w / 2)
            cy = y0 + dy * (dist * px * (0.7 if dx else 1) + h / 2)
            b = (cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2)
            if free(b):
                spot = (cx, cy, b, dist); break
        if spot: break
    if spot is None:                              # crowded everywhere: fall back to straight above or below
        cx, cy = x0, y0 + up * (9 * px + h / 2)
        spot = (cx, cy, (cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2), 9)
    cx, cy, b, dist = spot
    placed.append(b)
    ax.annotate(text, (row.war, row.y), xytext=((cx - x0) / px, (cy - y0) / px), textcoords='offset points',
                ha='center', va='center', fontsize=FS, color=INK2, zorder=5,
                arrowprops=dict(arrowstyle='-', color=INK2, lw=0.7, shrinkA=6, shrinkB=5) if dist > 9 else None)

pace = ' (82-game pace)' if (PER_82 or RATE) else ''
ax.set_title(f"{NAME} against {YNAME}, {int(t.season.min())}-{int(t.season.min()) % 100 + 1:02d} to "
             f"{int(t.season.max())}-{int(t.season.max()) % 100 + 1:02d}", loc='left', fontsize=13, color=INK, pad=24)
ax.text(0, 1.02, f"R² = {corr ** 2:.2f}", transform=ax.transAxes, fontsize=9, color=INK2)
ax.set_xlabel(f"{XLAB}{pace}", color=INK2, fontsize=10)
ax.set_ylabel(YNAME.capitalize() + ('' if RATE else pace), color=INK2, fontsize=10)
if RATE:
    ax.yaxis.set_major_formatter(lambda v, pos: f"{v:.3f}".lstrip('0'))
ax.grid(color=GRID, lw=0.8, zorder=0); ax.tick_params(colors=MUTED, labelsize=9, length=0)
for sp in ax.spines.values(): sp.set_visible(False)
if Y in ('win_pct', 'wins'):
    ax.text(0, -0.11, "Shootout games count as half a win for each side.", transform=ax.transAxes, fontsize=8, color=MUTED)
ax.text(1, -0.15, 'pasttheeyetest.com', transform=ax.transAxes, ha='right', fontsize=8, color=MUTED)
plt.savefig(f'{DATA_DIR}/team_war_vs_{Y}.png', dpi=200, bbox_inches='tight', facecolor=SURFACE)
plt.show()

## 19. Optional: WAR repeatability and reliability

One cell, two tests. Skaters and goalies are always kept apart, and skaters are also split into forwards and defensemen. It only reads the saved files, so it runs in seconds.

- **WAR to next season's WAR.** For every player with enough ice time (or shots faced) in two seasons running: the correlation between his WAR in one season and his 1-year WAR in the next. The 1-year and 3-year versions are both tested as the predictor, and Impact is shown next to WAR because WAR also rewards simply playing a lot. The chart is saved to Drive as `war_year_to_year.png`.
- **Cronbach's alpha.** A reliability score from 0 to 1: how consistently a number ranks the same players when it is measured several times. Here the repeated measurements are seasons, so each row is one player's 2, 3 or 5 consecutive seasons. It rises with the number of seasons, which is the case for multi-year ratings. The bar chart is saved to Drive as `war_cronbach_alpha.png`.

With `PER_82 = True`, 1-year WAR is put on an 82-game pace first so the two shortened seasons compare fairly.

In [ ]:
import os
import numpy as np, pandas as pd, matplotlib.pyplot as plt

PER_82 = True          # put 1-year WAR on an 82-game pace, so the two shortened seasons compare fairly
WINDOWS = [2, 3, 5]    # Cronbach's alpha: how many consecutive seasons are treated as repeated measurements

rs = pd.read_csv(f'{DATA_DIR}/player_ratings.csv', low_memory=False)
rg = pd.read_csv(f'{DATA_DIR}/goalie_ratings.csv').rename(columns={'goalie_id': 'player_id'})
if 'season_done' in rs.columns and 'season_done' in rg.columns:                            # a season in progress is left out
    rs, rg = rs[rs.season_done >= 1], rg[rg.season_done >= 1]
rs, rg = rs[rs.prod_qualified_1y == True].copy(), rg[rg.qualified_1y == True].copy()     # enough ice time or shots that season
if PER_82 and os.path.exists(f'{DATA_DIR}/team_war.csv'):
    team_gp = pd.read_csv(f'{DATA_DIR}/team_war.csv')[['season', 'team', 'gp']].rename(columns={'gp': 'team_gp'})
    for d in (rs, rg):
        d['war_1y'] = d.war_1y * 82 / d[['season', 'team']].merge(team_gp, on=['season', 'team'], how='left').team_gp.fillna(82).values
GROUPS = {'Forwards': rs[rs.pos_group == 'F'], 'Defensemen': rs[rs.pos_group == 'D'], 'All skaters': rs, 'Goalies': rg}

def next_season(d):
    # each player-season next to the same player's following season
    nxt = d[['season', 'player_id', 'war_1y', 'impact_1y']].copy()
    nxt['season'] -= 1
    return d.merge(nxt, on=['season', 'player_id'], suffixes=('', '_next'))

def cronbach(d, col, k):
    # rows = every run of k consecutive seasons by one player; columns = those k seasons
    wide = d.pivot_table(index='player_id', columns='season', values=col)
    years = list(wide.columns)
    runs = [wide[years[i:i + k]].dropna().values for i in range(len(years) - k + 1) if years[i + k - 1] - years[i] == k - 1]
    m = np.vstack(runs) if runs else np.empty((0, k))
    if len(m) < 10:
        return np.nan, len(m)
    return k / (k - 1) * (1 - m.var(axis=0, ddof=1).sum() / m.sum(axis=1).var(ddof=1)), len(m)

# ---------- 1. WAR in one season against WAR in the next
rows, pairs = {}, {}
for name, d in GROUPS.items():
    p = pairs[name] = next_season(d)
    rows[name] = {'pairs': len(p),
                  'WAR 1y -> next': p.war_1y.corr(p.war_1y_next), 'WAR 3y -> next': p.war_3y.corr(p.war_1y_next),
                  'Impact 1y -> next': p.impact_1y.corr(p.impact_1y_next), 'Impact 3y -> next': p.impact_3y.corr(p.impact_1y_next)}
yy = pd.DataFrame(rows).T
yy['pairs'] = yy.pairs.astype(int)
print("WAR TO NEXT SEASON'S WAR (correlation; the next season is always the 1-year number)")
print(yy.round(3).to_string())

# ---------- 2. Cronbach's alpha, with seasons as the repeated measurements
rows = []
for name, d in GROUPS.items():
    for k in WINDOWS:
        a_war, n = cronbach(d, 'war_1y', k)
        a_imp, _ = cronbach(d, 'impact_1y', k)
        rows.append({'group': name, 'seasons': k, 'rows': n, 'WAR': a_war, 'Impact': a_imp})
ca = pd.DataFrame(rows)
print()
print("CRONBACH'S ALPHA on 1-year values (0.7 and up is usually called acceptable, 0.8 good, 0.9 excellent)")
print("rows = runs of that many consecutive seasons by one player")
print(ca.round(3).to_string(index=False))

# ---------- chart: this season's WAR against next season's
INK, INK2, MUTED, GRID, SURFACE, BLUE = '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#fcfcfb', '#2a78d6'
fig, axes = plt.subplots(1, 2, figsize=(12, 5.4), dpi=110, facecolor=SURFACE)
for ax, name in zip(axes, ('All skaters', 'Goalies')):
    p = pairs[name]
    ax.set_facecolor(SURFACE)
    ax.scatter(p.war_1y, p.war_1y_next, s=14 if len(p) > 800 else 34, color=BLUE, alpha=0.35 if len(p) > 800 else 0.75,
               edgecolors='none', zorder=3)
    if len(p) > 2:
        slope, intercept = np.polyfit(p.war_1y, p.war_1y_next, 1)
        xs = np.array([p.war_1y.min(), p.war_1y.max()])
        ax.plot(xs, intercept + slope * xs, color=INK2, lw=1.5, zorder=4)
        ax.text(0, 1.02, f"r = {p.war_1y.corr(p.war_1y_next):.2f}  ·  {len(p):,} pairs of seasons", transform=ax.transAxes,
                fontsize=9, color=INK2)
    ax.set_title(name.replace('All skaters', 'Skaters'), loc='left', fontsize=13, color=INK, pad=22)
    ax.set_xlabel('WAR this season' + (' (82-game pace)' if PER_82 else ''), color=INK2, fontsize=10)
    ax.set_ylabel('WAR next season' + (' (82-game pace)' if PER_82 else ''), color=INK2, fontsize=10)
    ax.grid(color=GRID, lw=0.8, zorder=0); ax.tick_params(colors=MUTED, labelsize=9, length=0)
    for sp in ax.spines.values(): sp.set_visible(False)
fig.suptitle('WAR from one season to the next', x=0.06, ha='left', fontsize=14, color=INK, y=1.04)
fig.text(0.94, -0.02, 'pasttheeyetest.com', ha='right', fontsize=8, color=MUTED)
plt.savefig(f'{DATA_DIR}/war_year_to_year.png', dpi=200, bbox_inches='tight', facecolor=SURFACE)
plt.show()

# ---------- chart: Cronbach's alpha by group and number of seasons
RAMP = ['#86b6ef', '#2a78d6', '#104281', '#0d366b']          # fewer seasons = lighter
names = list(GROUPS)
fig, axes = plt.subplots(1, 2, figsize=(13, 5.6), dpi=110, facecolor=SURFACE, sharey=True)
low = min(0, np.nanmin(ca[['WAR', 'Impact']].values)) if ca[['WAR', 'Impact']].notna().any().any() else 0
width = 0.8 / len(WINDOWS)
for ax, metric in zip(axes, ('WAR', 'Impact')):
    ax.set_facecolor(SURFACE)
    for level, word in ((0.7, 'acceptable'), (0.8, 'good'), (0.9, 'excellent')):
        ax.axhline(level, color=MUTED, lw=0.8, ls=(0, (4, 4)), zorder=1)
        if metric == 'Impact':
            ax.text(len(names) - 0.42, level, f" {level:.1f} {word}", va='center', ha='left', fontsize=8.5, color=MUTED)
    for j, k in enumerate(WINDOWS):
        part = ca[ca.seasons == k].set_index('group').reindex(names)
        xs = np.arange(len(names)) - 0.4 + width * (j + 0.5)
        ax.bar(xs, part[metric].fillna(0), width=width * 0.92, color=RAMP[min(j, len(RAMP) - 1)], zorder=3,
               label=f"{k} seasons")
        for x, val in zip(xs, part[metric]):
            ax.text(x, (max(val, 0) if pd.notna(val) else 0) + 0.012, 'n/a' if pd.isna(val) else f"{val:.2f}",
                    ha='center', va='bottom', fontsize=8.5, color=INK2, zorder=5,
                    bbox=dict(facecolor=SURFACE, edgecolor='none', pad=1.0))
    ax.axhline(0, color=INK2, lw=1, zorder=4)
    ax.set_xticks(np.arange(len(names)))
    ax.set_xticklabels(names, fontsize=10, color=INK)
    ax.set_xlim(-0.6, len(names) - 0.4)
    ax.set_ylim(low - (0.05 if low < 0 else 0), 1.04)
    ax.set_title(metric, loc='left', fontsize=13, color=INK, pad=10)
    ax.grid(axis='y', color=GRID, lw=0.8, zorder=0); ax.tick_params(colors=MUTED, labelsize=9, length=0)
    for sp in ax.spines.values(): sp.set_visible(False)
axes[0].set_ylabel("Cronbach's alpha", color=INK2, fontsize=10)
axes[0].legend(loc='upper left', bbox_to_anchor=(0, 1.2), ncol=len(WINDOWS), frameon=False, fontsize=9, labelcolor=INK2,
               handlelength=1.2, columnspacing=1.6)
fig.suptitle("How consistently WAR and Impact rank the same players from season to season", x=0.06, ha='left', fontsize=14,
             color=INK, y=1.1)
fig.text(0.94, -0.02, 'pasttheeyetest.com', ha='right', fontsize=8, color=MUTED)
plt.savefig(f'{DATA_DIR}/war_cronbach_alpha.png', dpi=200, bbox_inches='tight', facecolor=SURFACE)
plt.show()

## 20. Optional: impact tiers and impact by season

Sorts every player-season into the roster spot his total impact was worth, then charts one player's seasons against those tiers. It only reads the saved files, so it runs in seconds.

**How the tiers are set.** Each season, players are ranked within their position by total impact, and the ranks are cut by roster spots per team. With 32 teams, the top 96 forwards are 1st line, the next 96 are 2nd line, and so on to the 4th line; the top 64 defensemen are 1st pair, then 2nd and 3rd pair; the top 32 goalies are starters (1), the next 16 are 1B, and the rest are backups. Skaters below the last line or pair are "Depth". `SLOTS` at the top changes the cut points.

**Three printouts and a chart:**

1. **Which measure should set the tiers?** WAR, GAR and GAA are each tested on how well they rank-predict next season's goals above average and goals above replacement, and on how often a player stays in the same tier. The best one is used; `METRIC` overrides it. WAR is GAR divided by goals per win, so those two always rank players identically and differ only in units.
2. **What each tier is worth.** Per tier: the usual value needed to make it, average WAR, GAR and GAA, and for context games, minutes and points (or shots and GSAx for goalies).
3. **One player's total impact by season.** Set `PLAYER`. His value each season is drawn over bands showing where that season's tiers fell, so you can read off whether each year was 1st-line, 2nd-line and so on. Saved to Drive as `impact_by_season_<name>.png`.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt

PLAYER = 'Tom Wilson'     # whose seasons to chart (skater or goalie, spelled as in the ratings files)
METRIC = None             # None = use the measure that tests best below; or 'WAR', 'GAR' or 'GAA'

# roster spots per team in each tier, best first; everyone ranked below the last one is in the final tier
SLOTS = {'F': [('1st line', 3), ('2nd line', 3), ('3rd line', 3), ('4th line', 3)],
         'D': [('1st pair', 2), ('2nd pair', 2), ('3rd pair', 2)],
         'G': [('Starter (1)', 1), ('1B', 0.5)]}
LAST = {'F': 'Depth', 'D': 'Depth', 'G': 'Backup'}
COLS = {'WAR': 'war_1y', 'GAR': 'gar_1y', 'GAA': 'gaa_1y'}

ts = pd.read_csv(f'{DATA_DIR}/player_ratings.csv', low_memory=False)
tg = pd.read_csv(f'{DATA_DIR}/goalie_ratings.csv').rename(columns={'goalie_id': 'player_id', 'games': 'gp'})
ts = ts[ts.min_all_1y.fillna(0) > 0].copy()                    # everyone who played that season
tg = tg[tg.shots.fillna(0) > 0].assign(pos_group='G')
for d in (ts, tg):
    if 'season_done' not in d.columns:
        d['season_done'] = 1.0                                  # files from before part seasons were supported
keep = ['season', 'player_id', 'player', 'team', 'pos_group', 'gp', 'season_done'] + list(COLS.values())
both = pd.concat([ts[keep + ['points', 'min_all_1y']], tg[keep + ['shots', 'gsax']]], ignore_index=True)
both['teams'] = both.groupby('season').team.transform('nunique')
finished = both.season_done >= 1                                # the tests below leave out a season in progress

def tiers_for(d, col):
    # rank within season and position, then turn the rank into roster spots per team
    spot = d.groupby(['season', 'pos_group'])[col].rank(ascending=False, method='first') / d.teams
    out = d.pos_group.map(LAST)
    for g, slots in SLOTS.items():
        top = 0
        for name, n in slots:
            out = out.mask((d.pos_group == g) & (spot > top) & (spot <= top + n), name)
            top += n
    return out
ORDER = {g: [n for n, _ in SLOTS[g]] + [LAST[g]] for g in SLOTS}

# ---------- 1. which measure should set the tiers?
nxt = both.loc[finished, ['season', 'player_id'] + list(COLS.values())].copy()
nxt['season'] -= 1
rows = []
for g, gname in (('F', 'Forwards'), ('D', 'Defensemen'), ('G', 'Goalies')):
    for m, col in COLS.items():
        d = both.assign(tier=tiers_for(both, col))
        n2 = nxt.assign(tier_next=d.tier[finished].values)
        p = d[(d.pos_group == g) & finished].merge(n2, on=['season', 'player_id'], suffixes=('', '_next'))
        step = (p.tier.map(ORDER[g].index) - p.tier_next.map(ORDER[g].index)).abs()
        rows.append({'group': gname, 'measure': m, 'pairs': len(p),
                     'predicts next GAA': p[col].corr(p.gaa_1y_next, method='spearman'),
                     'predicts next GAR': p[col].corr(p.gar_1y_next, method='spearman'),
                     'same tier next season': (step == 0).mean(), 'within one tier': (step <= 1).mean()})
test = pd.DataFrame(rows)
test['score'] = test[['predicts next GAA', 'predicts next GAR']].mean(axis=1)
print("WHICH MEASURE SHOULD SET THE TIERS?  (rank correlation with next season's goals above average and above replacement)")
print("WAR is GAR divided by goals per win, so the two always rank players the same way.")
print(test.round(3).to_string(index=False))
overall = test.groupby('measure').score.mean()
best = max(COLS, key=lambda m: (round(overall[m], 6), m == 'WAR'))          # a tie between WAR and GAR goes to WAR
USE = METRIC if METRIC in COLS else best
print()
print(f"best overall: {best}   (average score: " + ", ".join(f"{m} {overall[m]:.3f}" for m in COLS) + f")   ->  using {USE}")

# ---------- 2. what each tier is worth
both['tier'] = tiers_for(both, COLS[USE])
both['min_gp'] = both.min_all_1y / both.gp
print()
print(f"WHAT EACH TIER IS WORTH, tiers set by {USE}; averages per player-season over all seasons")
for g, gname in (('F', 'Forwards'), ('D', 'Defensemen'), ('G', 'Goalies')):
    d = both[(both.pos_group == g) & finished]
    extra = {'GP': ('gp', 'mean'), 'min per GP': ('min_gp', 'mean'), 'points': ('points', 'mean')}
    if g == 'G':
        extra = {'GP': ('gp', 'mean'), 'shots': ('shots', 'mean'), 'GSAx': ('gsax', 'mean')}
    t = d.groupby('tier').agg(**{'player-seasons': ('player_id', 'size'), f'lowest {USE}': (COLS[USE], 'min'),
                                 'WAR': ('war_1y', 'mean'), 'GAR': ('gar_1y', 'mean'), 'GAA': ('gaa_1y', 'mean'), **extra})
    lowest = d.groupby(['season', 'tier'])[COLS[USE]].min().groupby('tier').mean()      # the usual cutoff in a season
    t[f'lowest {USE}'] = lowest
    t = t.reindex([n for n in ORDER[g] if n in t.index]).rename(columns={f'lowest {USE}': f'{USE} to make it'})
    t.loc[t.index[-1], f'{USE} to make it'] = np.nan            # the bottom tier has no cutoff
    t['player-seasons'] = t['player-seasons'].astype(int)
    print()
    print(gname)
    print(t.round(2).to_string())

# ---------- 3. one player's total impact by season, against the tiers
who = both[both.player.str.lower() == PLAYER.lower()]
if who.empty:
    print()
    print(f"No player called {PLAYER!r}. Close matches:", sorted(both[both.player.str.contains(PLAYER.split()[-1], case=False, na=False)].player.unique())[:10])
else:
    who = who[who.player_id == who.sort_values('season').player_id.iloc[-1]].sort_values('season')
    g, col = who.pos_group.iloc[0], COLS[USE]
    seasons = sorted(both.season.unique())
    label = lambda y: f"{y}-{str(y + 1)[2:]}"
    shown = [n for n in ORDER[g] if (both[both.pos_group == g].tier == n).any()]          # tiers that have players
    cut = both[both.pos_group == g].groupby(['season', 'tier'])[col].min().unstack().reindex(seasons)[shown[:-1]]
    span = cut.max().max() - cut.min().min()
    lo = min(who[col].min(), cut.min().min() - 0.45 * span) - 0.08 * span
    hi = max(who[col].max(), cut.max().max() + 0.6 * span)
    hi = hi + 0.14 * (hi - lo)
    INK, INK2, MUTED, SURFACE, LINE = '#0b0b0b', '#52514e', '#898781', '#fcfcfb', '#eb6834'
    RAMP = ['#3987e5', '#6da7ec', '#9ec5f4', '#cde2fb', '#e9f1fc'] if g != 'G' else ['#3987e5', '#9ec5f4', '#e9f1fc']
    fig, ax = plt.subplots(figsize=(11, 5.6), dpi=110, facecolor=SURFACE)
    ax.set_facecolor(SURFACE)
    xs = np.arange(len(seasons))
    edges = np.r_[xs - 0.5, xs[-1] + 0.5]
    upper = np.full(len(seasons), hi)
    for i, name in enumerate(shown):
        lower = cut[name].values if name in cut else np.full(len(seasons), lo)
        ax.fill_between(edges, np.r_[lower, lower[-1]], np.r_[upper, upper[-1]], step='post', color=RAMP[i], lw=0, zorder=1)
        ax.plot(edges, np.r_[lower, lower[-1]], drawstyle='steps-post', color=SURFACE, lw=2, zorder=2)      # gap between bands
        ax.text(xs[-1] + 0.56, (min(upper[-1], hi) + max(lower[-1], lo)) / 2, name, va='center', ha='left', fontsize=10, color=INK2)
        upper = lower
    px = [seasons.index(s) for s in who.season]
    ax.plot(px, who[col], color=LINE, lw=2, zorder=4)
    ax.scatter(px, who[col], s=70, color=LINE, edgecolors=SURFACE, linewidths=2, zorder=5)
    for x, val in zip(px, who[col]):
        ax.annotate(f"{val:+.1f}", (x, val), xytext=(0, 10), textcoords='offset points', ha='center', fontsize=9.5, color=INK,
                    zorder=6, bbox=dict(facecolor=SURFACE, edgecolor='none', pad=1.2, alpha=0.85))
    ax.set_xlim(-0.5, xs[-1] + 0.5); ax.set_ylim(lo, hi)
    ax.set_xticks(xs); ax.set_xticklabels([label(s) for s in seasons], fontsize=10, color=INK2)
    ax.tick_params(colors=MUTED, labelsize=9, length=0)
    for sp in ax.spines.values(): sp.set_visible(False)
    unit = {'WAR': 'wins above replacement', 'GAR': 'goals above replacement', 'GAA': 'goals above average'}[USE]
    ax.set_ylabel(f"{USE} ({unit})", color=INK2, fontsize=10)
    group = {'F': 'forwards', 'D': 'defensemen', 'G': 'goalies'}[g]
    ax.set_title(f"{who.player.iloc[-1]}: total impact by season", loc='left', fontsize=14, color=INK, pad=24)
    ax.text(0, 1.02, f"Bands are where that season's {group} ranked by {USE}", transform=ax.transAxes, fontsize=9.5, color=INK2)
    fig.text(0.9, 0.02, 'pasttheeyetest.com', ha='right', fontsize=8, color=MUTED)
    plt.savefig(f"{DATA_DIR}/impact_by_season_{who.player.iloc[-1].replace(' ', '_')}.png", dpi=200, bbox_inches='tight', facecolor=SURFACE)
    plt.show()
    print(who.assign(season=who.season.map(label))[['season', 'team', 'gp', 'war_1y', 'gar_1y', 'gaa_1y', 'tier']].round(2).to_string(index=False))

## 21. Player dashboard (Streamlit)

Three cells.

1. **Write the app.** Saves `pte_dashboard.py` next to `player_ratings.csv` and `goalie_ratings.csv` on Drive. The app reads those two files, so run section 16 first.
2. **Start it.** Installs Streamlit, starts the app inside this Colab session, and prints a link. The dashboard runs for as long as this session stays open.
3. **Only if the link in step 2 shows a blank or stuck page:** an alternative way to reach the same app. It prints a temporary public web address; anyone who has that address can see the dashboard until the session ends.

**What the dashboard shows.** At the top of the sidebar, choose the player view or the leaderboard, then skaters or goalies, a season, a ratings version (3-year or 1-year) and optionally a team.

- **Impact by season.** On every player page: his WAR (or GAR or GAA) each season, drawn over bands showing where that season's 1st line, 2nd line and so on fell (pairs for defensemen; starter, 1B and backup for goalies). The tier for the selected season is also in the line under his name.
- **Save as PNG.** The button under the player's name downloads the whole page as one square image (2160 by 2160): headline numbers, percentiles, the WAR breakdown, impact by season and the z-scores. `Saved image` in the sidebar switches it between a dark and a light card.
- **Leaderboard.** One table of every player, ranked by WAR, with his tier. Click any column header to sort by it. WAR, goals above replacement and average, Impact and each WAR component are shown as raw values; then this season's totals (goals, assists, points, expected goals for skaters; goals against and GSAx splits for goalies) and the ratings. Filters for position and minimum games are in the sidebar, and the `Columns` box adds or removes groups of columns, including percentiles.

- **Skaters.** At the top: WAR for the season, 3-year WAR, and Individual Impact. Then percentile bars for WAR, Individual Impact, EV Offense, EV Defense, Power Play, Penalty Kill, Finishing, Playmaking, Penalties, Competition and Teammates. Then **where his WAR comes from**: one bar per component, to the right if it adds wins and to the left if it costs wins, adding up to his WAR. At the bottom: z-scores against the league average for raw on-ice xG for and against, finishing, playmaking, power play, penalty kill, quality of competition and quality of teammates, with the average z-score underneath.
- **Goalies.** The same layout: WAR, Impact and GSAx at the top; percentile bars for WAR, Impact, the team-adjusted rating, GSAx overall, at 5v5, against the power play and by danger, rebound control and win probability added; the WAR breakdown (saves at 5v5, against the power play, in other situations, the team and shooter adjustment, rebound control); and z-scores against the league average.

To host it permanently, put `pte_dashboard.py`, `player_ratings.csv` and `goalie_ratings.csv` in the same folder on any Streamlit host.

In [ ]:
%%writefile {DATA_DIR}/pte_dashboard.py
# PastTheEyeTest player dashboard. Run with:  streamlit run pte_dashboard.py
# Reads player_ratings.csv and goalie_ratings.csv from the same folder as this file.
from pathlib import Path
import io
import json
import numpy as np
import pandas as pd
import streamlit as st
import altair as alt
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Rectangle
from matplotlib.collections import PatchCollection

st.set_page_config(page_title="Player dashboard", layout="centered")
BLUE, MID, RED = (57, 135, 229), (138, 138, 134), (230, 103, 103)     # below average, average, above average
HERE = Path(__file__).parent
TRACK = "background:rgba(128,128,128,.14)"

@st.cache_data
def load(name):
    f = HERE / name
    return pd.read_csv(f, low_memory=False) if f.exists() else None

def blend_rgb(pct):
    # colour for a 0-100 percentile: blue at 0, grey at 50, red at 100
    a, b, u = (BLUE, MID, pct / 50) if pct < 50 else (MID, RED, (pct - 50) / 50)
    return tuple(round(a[i] + (b[i] - a[i]) * u) for i in range(3))

def blend(pct):
    return "rgb(%d,%d,%d)" % blend_rgb(pct)

CARD = {}                 # what the page has drawn, collected for the square image behind "Save as PNG"
CARD_THEMES = {"Dark": dict(bg="#121212", panel="#1f1f1f", ink="#f2f1ee", muted="#a3a29c", track="#2b2b2b"),
               "Light": dict(bg="#ffffff", panel="#f3f2ef", ink="#0b0b0b", muted="#6b6a66", track="#e6e5e1")}
SHORTER = [("Win probability added per 1,000 shots", "Win prob. added /1,000"), (" per 100 shots", " /100"),
           (" per 100 saves", " /100"), (" per 100", " /100"), (" per 60 (raw)", " /60"), ("Team-adjusted", "Team-adj."),
           ("Shooter and team adjustment", "Shooter, team adj.")]

def shorter(label):
    for long, short in SHORTER:
        label = label.replace(long, short)
    return label

def clean(x):
    # a number that can be stored in the card description: None when missing
    return None if x is None or pd.isna(x) else float(x)

@st.cache_data(show_spinner=False)
def render_card(payload):
    # the whole player page as one square PNG (2160 x 2160). Coordinates run 0-1000 in both directions.
    d = json.loads(payload)
    t = CARD_THEMES[d["theme"]]
    unit = lambda c: tuple(v / 255 for v in c)
    fig = plt.figure(figsize=(10, 10), dpi=216, facecolor=t["bg"])
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(0, 1000)
    ax.set_ylim(0, 1000)
    ax.axis("off")

    def text(x, y, words, size=9.2, color=None, weight="normal", ha="left"):
        ax.text(x, y, words, fontsize=size, color=color or t["ink"], fontweight=weight, ha=ha, va="center", zorder=5)

    def box(x, y, w, h, color, r=5, z=1, alpha=1.0):
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle=f"round,pad=0,rounding_size={r}", linewidth=0,
                                    facecolor=color, alpha=alpha, zorder=z))

    def signed(y, centre, half, value, limit):
        # a bar from the centre line: right and red for positive, left and blue for negative
        box(centre - half, y - 8, 2 * half, 16, t["track"], r=3)
        if value is not None:
            w = min(abs(value), limit) / limit * half
            ax.add_patch(Rectangle((centre if value >= 0 else centre - w, y - 8), w, 16, linewidth=0,
                                   facecolor=unit(RED if value >= 0 else BLUE), zorder=2))
        ax.plot([centre, centre], [y - 11, y + 11], color=t["muted"], lw=1.2, zorder=3)

    # ----- header and headline numbers
    text(40, 948, d["name"], size=30, weight="bold")
    text(40, 905, d["caption"], size=10.5, color=t["muted"])
    text(960, 962, "pasttheeyetest.com", size=10, color=t["muted"], ha="right")
    text(960, 940, d["version"], size=10, color=t["muted"], ha="right")
    for k, (label, value) in enumerate(d["tiles"]):
        x = 40 + k * 234
        box(x, 808, 218, 72, t["panel"], r=8)
        text(x + 14, 862, label, size=9, color=t["muted"])
        text(x + 14, 830, value, size=22, weight="bold")

    # ----- percentiles: 100 cells per row, the player's cell raised into a square with the number inside
    text(40, 772, d["pct_title"], size=12.5, weight="bold")
    cells, colours = [], []
    for k, (label, pct, raw) in enumerate(d["pct_rows"]):
        y = 742 - k * 30.5
        text(170, y, label, ha="right")
        if pct is None:
            box(182, y - 4, 310, 8, t["track"], r=2)
        else:
            here = min(99, max(0, int(pct)))
            for c in range(100):
                cells.append(Rectangle((182 + c * 3.1, y - 4), 2.4, 8))
                colours.append(unit(blend_rgb(c + 0.5)) + (0.4,))
            box(182 + (here + 0.5) * 3.1 - 12, y - 12, 24, 24, unit(blend_rgb(here + 0.5)), r=4, z=3)
            ax.text(182 + (here + 0.5) * 3.1, y - 0.5, str(int(pct)), fontsize=8, fontweight="bold", color="#0b0b0b",
                    ha="center", va="center", zorder=6)
        text(560, y, raw or "n/a", ha="right", weight="bold" if (raw and pct is not None) else "normal",
             color=None if (raw and pct is not None) else t["muted"])
    if cells:
        ax.add_collection(PatchCollection(cells, facecolors=colours, linewidths=0, zorder=2))

    # ----- where WAR comes from
    text(600, 772, "Where his WAR comes from", size=12.5, weight="bold")
    values = [v for _, v in d["parts"] if v is not None]
    limit = max(0.5, max(abs(v) for v in values)) if values else 0.5
    for k, (label, value) in enumerate(d["parts"]):
        y = 742 - k * 30.5
        text(745, y, label, ha="right")
        signed(y, 830, 75, value, limit)
        text(960, y, "n/a" if value is None else f"{value:+.2f}", ha="right", weight="bold")

    # ----- z-scores against the league average
    top = 742 - (max(len(d["parts"]), 1) - 1) * 30.5 - 62          # the z-scores start under the WAR rows
    text(600, top, "Against the league average (z-score)", size=12.5, weight="bold")
    y = top - 30
    step = min(46.0, (top - 30 - 140) / max(len(d["z_rows"]) - 1, 1))      # spread the rows down to the foot of the card
    for k, (label, z) in enumerate(d["z_rows"]):
        y = top - 30 - k * step
        text(775, y, label, size=8.8, ha="right")
        signed(y, 850, 65, z, 3.0)
        text(960, y, "n/a" if z is None else f"{z:+.2f}", ha="right", weight="bold" if z is not None else "normal",
             color=None if z is not None else t["muted"])
    text(600, y - 50, d["z_note"], color=t["muted"])
    text(960, y - 50, "–" if d["z_avg"] is None else f"{d['z_avg']:+.2f}", size=16, weight="bold", ha="right")

    # ----- impact by season over that season's tiers
    ch = d.get("chart")
    if ch:
        text(40, 392, f"Impact by season ({ch['measure']})", size=12.5, weight="bold")
        cx = fig.add_axes([0.075, 0.07, 0.405, 0.285], facecolor=t["bg"])
        n, lo, hi = len(ch["labels"]), ch["lo"], ch["hi"]
        shades = [0.85, 0.6, 0.4, 0.24, 0.1] if len(ch["tiers"]) > 3 else [0.85, 0.45, 0.12]
        upper = [hi] * n
        for j, name in enumerate(ch["tiers"]):
            lower = ch["lowers"][name]
            for i in range(n):
                cx.add_patch(Rectangle((i - 0.48, lower[i]), 0.96, max(upper[i] - lower[i], 0), linewidth=0,
                                       facecolor=unit(BLUE) + (shades[j],), zorder=1))
            cx.text(n - 0.44, (upper[-1] + lower[-1]) / 2, name, fontsize=7.5, color=t["muted"], va="center", ha="left", clip_on=False)
            upper = lower
        xs, ys = [q[0] for q in ch["pts"]], [q[1] for q in ch["pts"]]
        cx.plot(xs, ys, color=ORANGE, lw=2, zorder=3)
        cx.scatter(xs, ys, s=46, color=ORANGE, edgecolors="white", linewidths=1.2, zorder=4)
        for x_, y_ in zip(xs, ys):
            cx.annotate(f"{y_:+.1f}", (x_, y_), xytext=(0, 8), textcoords="offset points", ha="center", fontsize=8,
                        fontweight="bold", color="#0b0b0b", zorder=5,
                        bbox=dict(facecolor="white", edgecolor="none", pad=1.0, alpha=0.85))
        cx.set_xlim(-0.5, n - 0.5)
        cx.set_ylim(lo, hi)
        cx.set_xticks(range(n))
        cx.set_xticklabels(ch["labels"], fontsize=7.5, color=t["muted"])
        cx.tick_params(axis="y", colors=t["muted"], labelsize=7.5, length=0)
        cx.tick_params(axis="x", length=0)
        for side in cx.spines.values():
            side.set_visible(False)

    out = io.BytesIO()
    fig.savefig(out, format="png", dpi=216, facecolor=t["bg"])
    plt.close(fig)
    return out.getvalue()

def card_payload(theme, name, caption, v, tiles, group, p, rows, parts, zs, z_avg, z_note):
    # everything the page shows, as plain text and numbers, for render_card
    z_rows = []
    for _, row in zs.iterrows():
        z_rows.append((shorter(row.category), clean(row.z)))
    return json.dumps({
        "theme": theme, "name": name, "caption": caption, "version": "3-year ratings" if v == "3y" else "1-year ratings",
        "tiles": tiles, "pct_title": "Percentiles among " + group,
        "pct_rows": [(label, clean(p.get(col, np.nan)), raw_value(p, col)) for label, col, _ in rows],
        "parts": [(shorter(label), clean(p.get(f"{stem}_{v}", np.nan))) for label, stem, _ in parts],
        "z_rows": z_rows, "z_avg": clean(z_avg), "z_note": z_note, "chart": CARD.get("chart")}, sort_keys=True)

def season_label(y):
    return f"{y}-{str(y + 1)[2:]}"

def season_text(row_or_frame, y):
    # the season as text, marked when it is still being played
    done = row_or_frame.get("season_done", 1.0)
    done = float(done.min()) if hasattr(done, "min") else float(done)
    return season_label(y) + (" (in progress)" if done < 1 else "")

def num(x, d=2):
    return "–" if pd.isna(x) else f"{x:.{d}f}"

def signed_bar(value, limit):
    # a bar from the centre line: right and red for positive, left and blue for negative
    w = min(abs(value), limit) / limit * 50
    colour = "rgb(%d,%d,%d)" % (RED if value >= 0 else BLUE)
    side = "left:50%;border-radius:0 4px 4px 0" if value >= 0 else "right:50%;border-radius:4px 0 0 4px"
    return f'<div style="position:absolute;top:0;height:22px;width:{w:.1f}%;{side};background:{colour}"></div>'

def signed_rows(rows, limit, label_width):
    # rows: (label, value or NaN, text shown at the right, hover text)
    html = [f'<div style="display:grid;grid-template-columns:{label_width}px 1fr 52px;row-gap:10px;column-gap:12px;align-items:center">']
    for label, value, txt, tip in rows:
        bar = "" if pd.isna(value) else signed_bar(value, limit)
        html.append(f'<div style="text-align:right">{label}</div>'
                    f'<div title="{tip}" style="position:relative;height:22px;border-radius:4px;{TRACK}">{bar}'
                    f'<div style="position:absolute;left:50%;top:-3px;height:28px;width:2px;background:rgba(128,128,128,.9)"></div></div>'
                    f'<div style="text-align:right">{txt}</div>')
    html.append("</div>")
    st.markdown("".join(html), unsafe_allow_html=True)

PCT_STYLE = "Spectrum"

def spectrum_cells(val):
    # 100 cells, one per percentile, blue (low) through grey to red (high). The cell this player is in becomes a
    # square with the percentile inside it, centred on the row so it stands equally above and below the others.
    here = None if pd.isna(val) else min(99, max(0, int(val)))
    cells = []
    for k in range(100):
        if here is None:
            cells.append('<div style="flex:1;height:10px;border-radius:2px;background:rgba(128,128,128,.16)"></div>')
        elif k == here:
            cells.append(f'<div style="flex:0 0 30px;height:30px;margin:0 -13px;border-radius:5px;position:relative;z-index:1;'
                         f'background:{blend(k + 0.5)};color:#0b0b0b;font-weight:700;font-size:13px;line-height:30px;'
                         f'text-align:center">{int(val)}</div>')
        else:
            cells.append(f'<div style="flex:1;height:10px;border-radius:2px;opacity:.4;background:{blend(k + 0.5)}"></div>')
    return '<div style="display:flex;gap:1px;align-items:center;height:32px;padding:0 14px">' + "".join(cells) + "</div>"

RAW_DECIMALS = {"off": 3, "def": 3, "pp_off": 3, "pk_def": 3, "competition": 3, "teammates": 3, "adj100": 3}

def raw_value(p, pct_col):
    # the rating behind a percentile column, as signed text
    stem = pct_col[:-4]                                    # off_3y_pct -> off_3y
    col = stem.replace("pen_", "pen_diff60_") if stem.startswith("pen_") else stem
    x = p.get(col, np.nan)
    if pd.isna(x):
        return None
    return f"{x:+.{RAW_DECIMALS.get(stem[:-3], 2)}f}"

def percentile_bars(p, rows):
    wide = PCT_STYLE == "Spectrum"
    html = [f'<div style="display:grid;grid-template-columns:150px 1fr {"64px" if wide else "44px"};'
            f'row-gap:{6 if wide else 10}px;column-gap:{4 if wide else 12}px;align-items:center">']
    for label, col, tip in rows:
        val = p.get(col, np.nan)
        if wide:
            bar, raw = spectrum_cells(val), raw_value(p, col)
            txt = '<span style="opacity:.55">n/a</span>' if raw is None else (
                f"<b>{raw}</b>" if pd.notna(val) else f'<span style="opacity:.55">{raw}</span>')
        elif pd.isna(val):
            bar = f'<div style="height:22px;border-radius:4px;{TRACK}"></div>'
            txt = '<span style="opacity:.55">n/a</span>'
        else:
            bar = (f'<div style="height:22px;border-radius:4px;{TRACK}">'
                   f'<div style="height:22px;width:{max(val, 1.5):.0f}%;border-radius:4px;background:{blend(val)}"></div></div>')
            txt = f"<b>{int(val)}</b>"
        html.append(f'<div title="{tip}" style="text-align:right">{label}</div>'
                    f'<div title="{label}: {"not ranked" if pd.isna(val) else int(val)}. {tip}">{bar}</div>'
                    f'<div style="text-align:right;font-variant-numeric:tabular-nums">{txt}</div>')
    html.append("</div>")
    st.markdown("".join(html), unsafe_allow_html=True)

def war_breakdown(p, parts, v, who):
    # parts: (label, column stem, hover text). Each column is that part's share of WAR, in wins.
    vals = [(label, p.get(f"{stem}_{v}", np.nan), tip) for label, stem, tip in parts]
    vals = [x for x in vals if not pd.isna(x[1])]
    st.write("")
    st.subheader("Where " + who + " WAR comes from")
    if not vals:
        st.caption("The breakdown columns are not in the ratings file yet. Run the ratings notebook again to add them.")
        return None
    limit = max(0.5, max(abs(x[1]) for x in vals))
    signed_rows([(label, val, f"<b>{val:+.2f}</b>", f"{label}: {val:+.2f} wins. {tip}") for label, val, tip in vals], limit, 215)
    total = p.get(f"war_{v}", np.nan)
    return pd.DataFrame([(label, round(val, 3)) for label, val, _ in vals] + [("Total (WAR)", round(total, 3))],
                        columns=["part", "wins"])

def z_table(p, peers, zrows):
    # zrows: (label, value column, column that says whether a player is ranked there, +1 or -1)
    out = []
    for label, col, ranked, sign in zrows:
        ref = peers.loc[peers[ranked].notna(), col].dropna() if ranked in peers and col in peers else pd.Series(dtype=float)
        val = p.get(col, np.nan)
        unranked = pd.isna(val) or pd.isna(p.get(ranked, np.nan)) or len(ref) < 10 or ref.std() == 0
        z = np.nan if unranked else sign * (val - ref.mean()) / ref.std()
        out.append(dict(category=label, value=val, league_average=ref.mean() if len(ref) else np.nan, z=z, players=len(ref)))
    return pd.DataFrame(out)

def z_bars(zs, group, label_width=215, limit=3.0):
    rows = []
    for _, row in zs.iterrows():
        if pd.isna(row.z):
            rows.append((row.category, np.nan, '<span style="opacity:.55">n/a</span>', "not ranked on this row"))
        else:
            tip = f"{row.category}: {row.value:.3f} against a league average of {row.league_average:.3f} ({int(row.players)} {group})"
            rows.append((row.category, row.z, f"<b>{row.z:+.2f}</b>", tip))
    signed_rows(rows, limit, label_width)

# roster spots per team in each tier, best first; everyone ranked below the last one is in the final tier
TIER_SLOTS = {"F": [("1st line", 3), ("2nd line", 3), ("3rd line", 3), ("4th line", 3)],
              "D": [("1st pair", 2), ("2nd pair", 2), ("3rd pair", 2)],
              "G": [("Starter (1)", 1), ("1B", 0.5)]}
TIER_LAST = {"F": "Depth", "D": "Depth", "G": "Backup"}
TIER_ORDER = {g: [n for n, _ in TIER_SLOTS[g]] + [TIER_LAST[g]] for g in TIER_SLOTS}
TIER_COLS = {"WAR": "war_1y", "GAR": "gar_1y", "GAA": "gaa_1y"}
TIER_UNITS = {"WAR": "Wins above replacement", "GAR": "Goals above replacement", "GAA": "Goals above average"}
ORANGE = "#eb6834"

@st.cache_data
def with_tiers(kind):
    # every player-season, with the tier his 1-year WAR, GAR and GAA ranked in among his position that season
    if kind == "Skaters":
        d = load("player_ratings.csv").copy()
        d["played"] = d.min_all_1y.fillna(0) > 0
    else:
        d = load("goalie_ratings.csv").rename(columns={"goalie_id": "player_id"}).copy()
        d["pos_group"], d["played"] = "G", d.shots.fillna(0) > 0
    teams = d[d.played].groupby("season").team.nunique()
    for name, col in TIER_COLS.items():
        if col not in d.columns:
            continue
        spot = d[d.played].groupby(["season", "pos_group"])[col].rank(ascending=False, method="first")
        spot = (spot / d.loc[d.played, "season"].map(teams)).reindex(d.index)
        tier = d.pos_group.map(TIER_LAST).where(d.played)
        for g, slots in TIER_SLOTS.items():
            top = 0
            for label, n in slots:
                tier = tier.mask((d.pos_group == g) & (spot > top) & (spot <= top + n), label)
                top += n
        d[f"tier_{name}"] = tier
    return d

def impact_by_season(everyone, p):
    # one player's total impact each season, drawn over the tiers his position ranked in that season
    st.write("")
    st.subheader("Impact by season")
    choices = [m for m in TIER_COLS if f"tier_{m}" in everyone.columns]
    CARD["chart"] = None
    if not choices:
        return None
    measure = st.radio("Measure", choices, horizontal=True, label_visibility="collapsed")
    col, g = TIER_COLS[measure], p.pos_group
    league = everyone[everyone.played & (everyone.pos_group == g)]
    mine = league[league.player_id == p.player_id].sort_values("season")
    seasons = sorted(everyone.season.unique())
    labels = [season_label(y) for y in seasons]
    shown = [n for n in TIER_ORDER[g] if (league[f"tier_{measure}"] == n).any()]
    cut = league.groupby(["season", f"tier_{measure}"])[col].min().unstack().reindex(seasons)[shown[:-1]]
    span = cut.max().max() - cut.min().min()
    lo = min(mine[col].min(), cut.min().min() - 0.45 * span) - 0.08 * span
    hi = max(mine[col].max(), cut.max().max() + 0.6 * span)
    hi = hi + 0.14 * (hi - lo)
    bands, lowers = [], {name: [] for name in shown}
    for y, lab in zip(seasons, labels):
        upper = hi
        for name in shown:
            lower = cut.loc[y, name] if name in cut.columns and pd.notna(cut.loc[y, name]) else lo
            bands.append({"season": lab, "tier": name, "from": round(max(lower, lo), 2), "to": round(upper, 2)})
            lowers[name].append(float(max(lower, lo)))
            upper = max(lower, lo)
    CARD["chart"] = {"measure": measure, "labels": labels, "tiers": shown, "lowers": lowers, "lo": float(lo), "hi": float(hi),
                     "pts": [[seasons.index(a), float(b)] for a, b in zip(mine.season, mine[col])]}
    shades = [0.85, 0.6, 0.4, 0.24, 0.1] if len(shown) > 3 else [0.85, 0.45, 0.12]
    ramp = ["rgba(57,135,229,%.2f)" % a for a in shades[:len(shown)]]
    x = alt.X("season:O", sort=labels, title=None, scale=alt.Scale(domain=labels, paddingInner=0.04, paddingOuter=0),
              axis=alt.Axis(labelAngle=0))
    yscale = alt.Scale(domain=[lo, hi], nice=False, zero=False)
    back = alt.Chart(pd.DataFrame(bands)).mark_rect().encode(
        x=x, y=alt.Y("from:Q", title=TIER_UNITS[measure], scale=yscale), y2="to:Q",
        color=alt.Color("tier:N", scale=alt.Scale(domain=shown, range=ramp), legend=alt.Legend(title=None, orient="top")),
        tooltip=[alt.Tooltip("season:O", title="Season"), alt.Tooltip("tier:N", title="Tier"),
                 alt.Tooltip("from:Q", title="From", format="+.2f"), alt.Tooltip("to:Q", title="To", format="+.2f")])
    pts = pd.DataFrame({"season": mine.season.map(season_label).values, "value": mine[col].values,
                        "tier": mine[f"tier_{measure}"].values, "team": mine.team.values})
    tips = [alt.Tooltip("season:O", title="Season"), alt.Tooltip("team:N", title="Team"),
            alt.Tooltip("value:Q", title=measure, format="+.2f"), alt.Tooltip("tier:N", title="Tier")]
    line = alt.Chart(pts).mark_line(color=ORANGE, strokeWidth=2.5).encode(x=x, y=alt.Y("value:Q", scale=yscale))
    dots = alt.Chart(pts).mark_point(filled=True, size=130, color=ORANGE, stroke="white", strokeWidth=1.5, opacity=1).encode(
        x=x, y=alt.Y("value:Q", scale=yscale), tooltip=tips)
    label = alt.Chart(pts).encode(x=x, y=alt.Y("value:Q", scale=yscale), text=alt.Text("value:Q", format="+.1f"))
    halo = label.mark_text(dy=-15, fontSize=12, fontWeight="bold", color="white", stroke="white", strokeWidth=3.5, strokeJoin="round")
    text = label.mark_text(dy=-15, fontSize=12, fontWeight="bold", color="#0b0b0b")
    st.altair_chart((back + line + dots + halo + text).properties(height=380))
    out = mine[["season", "team"] + [c for c in TIER_COLS.values() if c in mine.columns] + [f"tier_{measure}"]].copy()
    out["season"] = out.season.map(season_label)
    return out.rename(columns={"war_1y": "WAR", "gar_1y": "GAR", "gaa_1y": "GAA", f"tier_{measure}": "tier"}).round(2)

GROUPS = ["WAR and Impact", "Season totals", "Ratings", "Percentiles"]

def leaderboard_columns(kind, v):
    # (group, header, column in the file, decimals, what it is)
    if kind == "Skaters":
        return [
            ("WAR and Impact", "WAR", f"war_{v}", 2, "Wins above replacement."),
            ("WAR and Impact", "GAR", f"gar_{v}", 1, "Goals above replacement."),
            ("WAR and Impact", "GAA", f"gaa_{v}", 1, "Goals above average."),
            ("WAR and Impact", "Impact/60", f"impact_{v}", 2, "Individual Impact: goals above average per 60 minutes."),
            ("WAR and Impact", "Off WAR", f"war_off_{v}", 2, "Wins from EV offense."),
            ("WAR and Impact", "Def WAR", f"war_def_{v}", 2, "Wins from EV defense."),
            ("WAR and Impact", "PP WAR", f"war_pp_{v}", 2, "Wins from the power play."),
            ("WAR and Impact", "PK WAR", f"war_pk_{v}", 2, "Wins from the penalty kill."),
            ("WAR and Impact", "Shoot WAR", f"war_shoot_{v}", 2, "Wins from goals above expected on his own shots."),
            ("WAR and Impact", "Setup WAR", f"war_setup_{v}", 2, "Wins from goals above expected on teammates' shots."),
            ("WAR and Impact", "Pen WAR", f"war_pen_{v}", 2, "Wins from penalties drawn minus taken."),
            ("Season totals", "G", "goals", 0, "Goals."),
            ("Season totals", "A1", "a1", 0, "Primary assists."),
            ("Season totals", "A2", "a2", 0, "Secondary assists."),
            ("Season totals", "PTS", "points", 0, "Points."),
            ("Season totals", "EVP", "ev_points", 0, "Even-strength points."),
            ("Season totals", "PPP", "pp_points", 0, "Power-play points."),
            ("Season totals", "SHP", "sh_points", 0, "Shorthanded points."),
            ("Season totals", "ixG", "ixg", 1, "His own expected goals."),
            ("Season totals", "GAx", "gax", 1, "Goals above expected on his own shots."),
            ("Season totals", "Pen drawn", "pen_drawn", 0, "Minor penalties drawn."),
            ("Season totals", "Pen taken", "pen_taken", 0, "Minor penalties taken."),
            ("Ratings", "EV Off", f"off_{v}", 3, "Isolated effect on xG for per 60 at 5v5."),
            ("Ratings", "EV Def", f"def_{v}", 3, "Isolated effect on xG against per 60 at 5v5. Higher is better."),
            ("Ratings", "PP", f"pp_off_{v}", 3, "Isolated effect on xG for per 60 on the power play."),
            ("Ratings", "PK", f"pk_def_{v}", 3, "Isolated effect on xG against per 60 on the penalty kill. Higher is better."),
            ("Ratings", "Finishing", f"fin_{v}", 2, "Finishing score."),
            ("Ratings", "Playmaking", f"play_{v}", 2, "Playmaking score."),
            ("Ratings", "Pen/60", f"pen_diff60_{v}", 2, "Minor penalties drawn minus taken per 60."),
            ("Ratings", "xGF/60", f"xgf60_{v}", 2, "Raw on-ice xG for per 60 at 5v5."),
            ("Ratings", "xGA/60", f"xga60_{v}", 2, "Raw on-ice xG against per 60 at 5v5."),
            ("Ratings", "QoC", f"competition_{v}", 3, "Quality of competition."),
            ("Ratings", "QoT", f"teammates_{v}", 3, "Quality of teammates."),
            ("Percentiles", "EV Off %", f"off_{v}_pct", 0, "Percentile within position."),
            ("Percentiles", "EV Def %", f"def_{v}_pct", 0, "Percentile within position."),
            ("Percentiles", "PP %", f"pp_off_{v}_pct", 0, "Percentile within position."),
            ("Percentiles", "PK %", f"pk_def_{v}_pct", 0, "Percentile within position."),
            ("Percentiles", "Finishing %", f"fin_{v}_pct", 0, "Percentile within position."),
            ("Percentiles", "Playmaking %", f"play_{v}_pct", 0, "Percentile within position."),
            ("Percentiles", "Penalties %", f"pen_{v}_pct", 0, "Percentile within position."),
            ("Percentiles", "QoC %", f"competition_{v}_pct", 0, "Percentile within position."),
            ("Percentiles", "QoT %", f"teammates_{v}_pct", 0, "Percentile within position.")]
    return [
        ("WAR and Impact", "WAR", f"war_{v}", 2, "Wins above replacement, built on GSAx."),
        ("WAR and Impact", "GAR", f"gar_{v}", 1, "Goals saved above replacement."),
        ("WAR and Impact", "GSAA", f"gaa_{v}", 1, "Goals saved above average, as used in WAR."),
        ("WAR and Impact", "Impact/GP", f"impact_{v}", 2, "Individual Impact: goals saved above average per game's worth of shots."),
        ("WAR and Impact", "5v5 WAR", f"war_5v5_{v}", 2, "Wins from GSAx at 5v5."),
        ("WAR and Impact", "vs PP WAR", f"war_pk_{v}", 2, "Wins from GSAx against the power play."),
        ("WAR and Impact", "Other WAR", f"war_other_{v}", 2, "Wins from GSAx in other situations."),
        ("WAR and Impact", "Adj WAR", f"war_adj_{v}", 2, "Shooter and team adjustment."),
        ("WAR and Impact", "Rebound WAR", f"war_reb_{v}", 2, "Wins from rebound control."),
        ("Season totals", "GA", "ga", 0, "Goals against on unblocked shots with the net occupied."),
        ("Season totals", "xGA", "xga", 1, "Expected goals against."),
        ("Season totals", "GSAx", "gsax", 1, "Goals saved above expected."),
        ("Season totals", "5v5 GSAx", "gsax_5v5", 1, "Goals saved above expected at 5v5."),
        ("Season totals", "vs PP GSAx", "gsax_pk", 1, "Goals saved above expected against the power play."),
        ("Season totals", "Low GSAx", "gsax_low", 1, "Low-danger shots."),
        ("Season totals", "Med GSAx", "gsax_med", 1, "Medium-danger shots."),
        ("Season totals", "High GSAx", "gsax_high", 1, "High-danger shots."),
        ("Season totals", "Rebound GS", "reb_saved", 1, "Goals saved through rebound control."),
        ("Season totals", "WPA", "wpa", 2, "Win probability added, in wins."),
        ("Ratings", "Team-adj/100", f"adj100_{v}", 3, "Team-adjusted goals saved above average per 100 shots."),
        ("Ratings", "GSAx/100", f"gsax100_{v}", 2, "Goals saved above expected per 100 shots."),
        ("Ratings", "5v5/100", f"gsax5v5_100_{v}", 2, "At 5v5, per 100 shots."),
        ("Ratings", "vs PP/100", f"gsaxpk_100_{v}", 2, "Against the power play, per 100 shots."),
        ("Ratings", "Low/100", f"gsaxlow_100_{v}", 2, "Low danger, per 100 shots."),
        ("Ratings", "Med/100", f"gsaxmed_100_{v}", 2, "Medium danger, per 100 shots."),
        ("Ratings", "High/100", f"gsaxhigh_100_{v}", 2, "High danger, per 100 shots."),
        ("Ratings", "Rebound/100", f"reb100_{v}", 2, "Rebound control per 100 saves."),
        ("Ratings", "WPA/1000", f"wpa1000_{v}", 2, "Win probability added per 1,000 shots."),
        ("Percentiles", "Team-adj %", f"adj100_{v}_pct", 0, "Percentile among goalies."),
        ("Percentiles", "GSAx %", f"gsax100_{v}_pct", 0, "Percentile among goalies."),
        ("Percentiles", "5v5 %", f"gsax5v5_100_{v}_pct", 0, "Percentile among goalies."),
        ("Percentiles", "vs PP %", f"gsaxpk_100_{v}_pct", 0, "Percentile among goalies."),
        ("Percentiles", "Low %", f"gsaxlow_100_{v}_pct", 0, "Percentile among goalies."),
        ("Percentiles", "Med %", f"gsaxmed_100_{v}_pct", 0, "Percentile among goalies."),
        ("Percentiles", "High %", f"gsaxhigh_100_{v}_pct", 0, "Percentile among goalies."),
        ("Percentiles", "Rebound %", f"reb100_{v}_pct", 0, "Percentile among goalies."),
        ("Percentiles", "WPA %", f"wpa1000_{v}_pct", 0, "Percentile among goalies.")]

def leaderboard(pool, kind, v, groups, title, line):
    # one sortable table: click any column header to sort by it
    st.markdown('<style>.block-container, [data-testid="stMainBlockContainer"] {max-width: 1500px;}</style>',
                unsafe_allow_html=True)
    st.title(title)
    st.caption(line)
    if kind == "Skaters":
        base = [("Player", "player"), ("Team", "team"), ("Pos", "position"), ("Tier", "tier_WAR"), ("GP", "gp"), ("Min", "min_all_1y")]
    else:
        base = [("Player", "player"), ("Team", "team"), ("Tier", "tier_WAR"), ("GP", "games"), ("Shots", "shots")]
    base = [b for b in base if b[1] in pool.columns]
    specs = [x for x in leaderboard_columns(kind, v) if x[0] in groups and x[2] in pool.columns]
    specs = [x for x in specs if not (x[2].startswith(("war_setup", "war_reb")) and pool[x[2]].abs().max() < 1e-9)]
    pool = pool.sort_values(f"war_{v}", ascending=False)
    table = pd.DataFrame({"#": np.arange(1, len(pool) + 1)})
    config = {"#": st.column_config.NumberColumn("#", help="Rank by WAR among the players shown.", pinned=True),
              "Player": st.column_config.TextColumn("Player", pinned=True),
              "Tier": st.column_config.TextColumn("Tier", help="Where his 1-year WAR ranked at his position this season.")}
    for head, col in base:
        table[head] = pool[col].values
        if head in ("GP", "Min", "Shots"):
            config[head] = st.column_config.NumberColumn(head, format="%d")
    for _, head, col, dec, tip in specs:
        table[head] = pool[col].astype(float).round(dec).values
        config[head] = st.column_config.NumberColumn(head, help=tip, format="%d" if dec == 0 else f"%.{dec}f")
    st.dataframe(table, hide_index=True, column_config=config, height=min(38 + 35 * max(len(table), 1), 1090))

skaters, goalies = load("player_ratings.csv"), load("goalie_ratings.csv")
if skaters is None:
    st.error("player_ratings.csv is not in the same folder as this app. Run the Save section of the ratings notebook first.")
    st.stop()

# ------------------------------------------------------------------ choices
with st.sidebar:
    st.header("Player dashboard")
    view = st.radio("View", ["Player", "Leaderboard"], horizontal=True)
    kind = st.radio("Show", ["Skaters", "Goalies"], horizontal=True) if goalies is not None else "Skaters"
    r = with_tiers(kind)
    season = st.selectbox("Season", sorted(r.season.unique(), reverse=True), format_func=season_label)
    version = st.radio("Ratings", ["3y", "1y"],
                       format_func=lambda v: "3-year (current value)" if v == "3y" else "1-year (this season only)")
    pool = r[(r.season == season) & r.team.notna()].sort_values("war_1y", ascending=False)
    teams = ["All teams"] + sorted(pool.team.unique())
    team = st.selectbox("Team", teams)
    if team != "All teams":
        pool = pool[pool.team == team]
    if view == "Player":
        pid = st.selectbox("Player (type to search)", pool.player_id,
                           format_func=lambda i: "%s  ·  %s" % tuple(pool.loc[pool.player_id == i, ["player", "team"]].iloc[0]))
        PCT_STYLE = st.radio("Percentiles", ["Spectrum", "Bars"], horizontal=True)
        card_theme = st.radio("Saved image", ["Dark", "Light"], horizontal=True)
    else:
        if kind == "Skaters":
            spot = st.selectbox("Position", ["All", "Forwards", "Defensemen"])
            if spot != "All":
                pool = pool[pool.pos_group == spot[0]]
        games = pool["gp" if kind == "Skaters" else "games"].fillna(0)
        most = int(games.max()) if len(games) else 0
        least = st.slider("Minimum games played", 0, max(most, 1), min(20 if kind == "Skaters" else 10, most // 4))
        pool = pool[games >= least]
        groups = st.multiselect("Columns", GROUPS, default=GROUPS[:3])

v = version
if view == "Leaderboard":
    bits = [season_text(pool, season), "3-year ratings" if v == "3y" else "1-year ratings", f"{len(pool)} {kind.lower()}"]
    if team != "All teams":
        bits.insert(0, team)
    leaderboard(pool, kind, v, groups, f"{kind[:-1]} leaderboard", "  ·  ".join(bits))
    st.caption("pasttheeyetest.com")
    st.stop()
p = r[(r.season == season) & (r.player_id == pid)].iloc[0]

# =================================================================== skaters
if kind == "Skaters":
    peers = r[(r.season == season) & (r.pos_group == p.pos_group)]
    group = "forwards" if p.pos_group == "F" else "defensemen"

    st.title(p.player)
    bits = [p.team, {"C": "Centre", "L": "Left wing", "R": "Right wing", "D": "Defense"}.get(p.position, p.position),
            season_text(p, season)]
    if pd.notna(p.gp):
        bits.append(f"{int(p.gp)} GP")
    if pd.notna(p.min_5v5_1y):
        bits.append(f"{p.min_5v5_1y:,.0f} min at 5v5")
    if pd.notna(p.get("tier_WAR", np.nan)):
        bits.append(f"Impact: {p.tier_WAR}")
    st.caption("  ·  ".join(str(b) for b in bits))
    save_slot = st.empty()
    TILES = [("WAR, this season", num(p.war_1y, 1)), ("WAR, 3-year", num(p.war_3y, 1)),
             ("Individual Impact (percentile)", "–" if pd.isna(p[f"impact_{v}_pct"]) else f"{int(p[f'impact_{v}_pct'])}"),
             ("Impact, goals per 60", num(p[f"impact_{v}"], 2))]

    c1, c2, c3, c4 = st.columns(4)
    c1.metric("WAR, this season", num(p.war_1y, 1), help="Wins above replacement from this season's ratings and minutes.")
    c2.metric("WAR, 3-year", num(p.war_3y, 1), help="A typical season at his 3-year level: 3-year ratings and his average minutes.")
    imp = p[f"impact_{v}_pct"]
    c3.metric("Individual Impact", "–" if pd.isna(imp) else f"{int(imp)}",
              help=f"Percentile among {group}. Goals above average per 60 minutes, all components combined.")
    c4.metric("Impact, goals per 60", num(p[f"impact_{v}"], 2), help="Goals above average per 60 minutes of ice time.")
    if not bool(p.get("active", True)):
        st.info("Under the minutes cutoff this season, so he is not ranked in the 3-year percentiles.")

    ROWS = [("WAR", f"war_{v}_pct", "Wins above replacement: total value, so it rewards playing more."),
            ("Individual Impact", f"impact_{v}_pct", "Goals above average per 60 minutes, all components combined."),
            ("EV Offense", f"off_{v}_pct", "Isolated effect on his team's expected goals at 5v5."),
            ("EV Defense", f"def_{v}_pct", "Isolated effect on expected goals against at 5v5."),
            ("Power Play", f"pp_off_{v}_pct", "Isolated effect on expected goals with the man advantage."),
            ("Penalty Kill", f"pk_def_{v}_pct", "Isolated effect on expected goals against while shorthanded."),
            ("Finishing", f"fin_{v}_pct", "Goals above expected and goal scoring."),
            ("Playmaking", f"play_{v}_pct", "Primary and secondary assists, weighted by situation."),
            ("Penalties", f"pen_{v}_pct", "Minor penalties drawn minus taken."),
            ("Competition", f"competition_{v}_pct", "Context, not skill: quality of the opponents he faces at 5v5."),
            ("Teammates", f"teammates_{v}_pct", "Context, not skill: quality of the linemates he plays with at 5v5.")]
    st.subheader("Percentiles among " + group)
    percentile_bars(p, ROWS)

    PARTS = [("EV Offense", "war_off", "Expected goals he adds for his team at 5v5."),
             ("EV Defense", "war_def", "Expected goals he takes away from opponents at 5v5."),
             ("Power Play", "war_pp", "Expected goals he adds with the man advantage."),
             ("Penalty Kill", "war_pk", "Expected goals he takes away while shorthanded."),
             ("Shooting results", "war_shoot", "Goals above expected on his own shots."),
             ("Setup results", "war_setup", "Goals above expected on teammates' shots while he is on the ice."),
             ("Penalties", "war_pen", "Minor penalties drawn minus taken, at the goal value of a minor.")]
    if skaters.get(f"war_setup_{v}", pd.Series([0.0])).abs().max() < 1e-9:       # setup is switched off in the ratings
        PARTS = [x for x in PARTS if x[1] != "war_setup"]
    parts_table = war_breakdown(p, PARTS, v, "his")
    seasons_table = impact_by_season(r, p)

    ZROWS = [("On-ice xG for per 60 (raw)", f"xgf60_{v}", f"off_{v}_pct", 1),
             ("On-ice xG against per 60 (raw)", f"xga60_{v}", f"def_{v}_pct", -1),
             ("Finishing", f"fin_{v}", f"fin_{v}_pct", 1),
             ("Playmaking", f"play_{v}", f"play_{v}_pct", 1),
             ("Power play", f"pp_off_{v}", f"pp_off_{v}_pct", 1),
             ("Penalty kill", f"pk_def_{v}", f"pk_def_{v}_pct", 1),
             ("Quality of competition", f"competition_{v}", f"competition_{v}_pct", 1),
             ("Quality of teammates", f"teammates_{v}", f"teammates_{v}_pct", 1)]
    zs = z_table(p, peers, ZROWS)
    st.write("")
    st.subheader("Against the league average")
    z_bars(zs, group)
    overall = zs.loc[~zs.category.str.startswith("Quality"), "z"].mean()
    st.write("")
    st.metric("Average z-score across the performance categories",
              "–" if pd.isna(overall) else f"{overall:+.2f}",
              help="The mean of the z-scores above that he is ranked on, leaving out quality of competition and teammates.")
    z_note = "Average z-score, performance categories"

# =================================================================== goalies
else:
    peers = r[(r.season == season) & (r[f"qualified_{v}"] == True)]
    group = "goalies"

    st.title(p.player)
    bits = [p.team, "Goalie", season_text(p, season)]
    if pd.notna(p.games) and p.games > 0:
        bits.append(f"{int(p.games)} GP")
    bits.append(f"{p.shots:,.0f} unblocked shots faced")
    if pd.notna(p.get("tier_WAR", np.nan)):
        bits.append(f"Impact: {p.tier_WAR}")
    st.caption("  ·  ".join(str(b) for b in bits))
    save_slot = st.empty()
    TILES = [("WAR, this season", num(p.war_1y, 1)), ("WAR, 3-year", num(p.war_3y, 1)),
             ("Individual Impact (percentile)", "–" if pd.isna(p[f"impact_{v}_pct"]) else f"{int(p[f'impact_{v}_pct'])}"),
             ("GSAx, this season", num(p.gsax, 1))]

    c1, c2, c3, c4 = st.columns(4)
    c1.metric("WAR, this season", num(p.war_1y, 1), help="Wins above replacement, built on goals saved above expected (GSAx), not raw saves.")
    c2.metric("WAR, 3-year", num(p.war_3y, 1), help="A typical season at his 3-year level: 3-year rating and his average workload.")
    imp = p[f"impact_{v}_pct"]
    c3.metric("Individual Impact", "–" if pd.isna(imp) else f"{int(imp)}",
              help="Percentile among goalies. Goals saved above average per game's worth of shots.")
    c4.metric("GSAx, this season", num(p.gsax, 1), help="Goals saved above expected on every shot he faced this season, before any adjustment.")
    if pd.isna(p.get(f"war_{v}_pct", np.nan)):
        st.info("Too few shots faced to be ranked in these percentiles.")

    ROWS = [("WAR", f"war_{v}_pct", "Wins above replacement: total value, so it rewards playing more."),
            ("Individual Impact", f"impact_{v}_pct", "Goals saved above average per game's worth of shots: the rate behind WAR."),
            ("Team-adjusted GSAA", f"adj100_{v}_pct", "Goals saved above average per 100 shots after allowing for the team in front of him and the shooters he faced."),
            ("GSAx, all shots", f"gsax100_{v}_pct", "Goals saved above expected per 100 shots, before any adjustment."),
            ("5v5 GSAx", f"gsax5v5_100_{v}_pct", "Goals saved above expected per 100 shots at 5v5."),
            ("GSAx vs power play", f"gsaxpk_100_{v}_pct", "Goals saved above expected per 100 shots while his team is shorthanded."),
            ("Low danger", f"gsaxlow_100_{v}_pct", "Shots with under a 5% chance of scoring."),
            ("Medium danger", f"gsaxmed_100_{v}_pct", "Shots with a 5 to 15% chance of scoring."),
            ("High danger", f"gsaxhigh_100_{v}_pct", "Shots with over a 15% chance of scoring."),
            ("Rebound control", f"reb100_{v}_pct", "Rebound shots allowed against what his saves would be expected to produce."),
            ("Win probability added", f"wpa1000_{v}_pct", "Goals saved, weighted by how much each one mattered to the result.")]
    st.subheader("Percentiles among goalies")
    percentile_bars(p, ROWS)

    PARTS = [("5v5 GSAx", "war_5v5", "Goals saved above expected at 5v5, against a replacement goalie."),
             ("GSAx vs power play", "war_pk", "Goals saved above expected while his team is shorthanded, against a replacement goalie."),
             ("GSAx, other situations", "war_other", "4-on-4, 3-on-3, his own team's power play and anything else."),
             ("Shooter and team adjustment", "war_adj", "Credit for the finishing skill of the shooters he faced; in the 3-year version also the team in front of him and the pull toward average for small samples."),
             ("Rebound control", "war_reb", "Goals saved by giving up fewer rebound shots than expected.")]
    if goalies.get(f"war_reb_{v}", pd.Series([0.0])).abs().max() < 1e-9:         # rebounds are left out of WAR in the ratings
        PARTS = [x for x in PARTS if x[1] != "war_reb"]
    parts_table = war_breakdown(p, PARTS, v, "his")
    seasons_table = impact_by_season(r, p)

    ZROWS = [("Team-adjusted GSAA per 100 shots", f"adj100_{v}", f"adj100_{v}_pct", 1),
             ("GSAx per 100 shots", f"gsax100_{v}", f"gsax100_{v}_pct", 1),
             ("5v5 GSAx per 100", f"gsax5v5_100_{v}", f"gsax5v5_100_{v}_pct", 1),
             ("GSAx vs power play per 100", f"gsaxpk_100_{v}", f"gsaxpk_100_{v}_pct", 1),
             ("Low danger per 100", f"gsaxlow_100_{v}", f"gsaxlow_100_{v}_pct", 1),
             ("Medium danger per 100", f"gsaxmed_100_{v}", f"gsaxmed_100_{v}_pct", 1),
             ("High danger per 100", f"gsaxhigh_100_{v}", f"gsaxhigh_100_{v}_pct", 1),
             ("Rebound control per 100 saves", f"reb100_{v}", f"reb100_{v}_pct", 1),
             ("Win probability added per 1,000 shots", f"wpa1000_{v}", f"wpa1000_{v}_pct", 1)]
    zs = z_table(p, peers, ZROWS)
    st.write("")
    st.subheader("Against the league average")
    z_bars(zs, group, 270)
    overall = zs.z.mean()
    st.write("")
    st.metric("Average z-score across the categories", "–" if pd.isna(overall) else f"{overall:+.2f}",
              help="The mean of the z-scores above that he is ranked on.")
    z_note = "Average z-score"

# the whole page as one square image
png = render_card(card_payload(card_theme, p.player, "  ·  ".join(str(b) for b in bits), v, TILES, group, p, ROWS, PARTS,
                               zs, overall, z_note))
save_slot.download_button("Save as PNG", png, file_name=f"{p.player}_{season_label(season)}_{v}.png".replace(" ", "_"),
                          mime="image/png")

with st.expander("Show the numbers"):
    t1 = pd.DataFrame([(lab, p.get(col, np.nan)) for lab, col, _ in ROWS], columns=["row", "percentile"])
    st.dataframe(t1, hide_index=True)
    if parts_table is not None:
        st.dataframe(parts_table, hide_index=True)
    if seasons_table is not None:
        st.dataframe(seasons_table, hide_index=True)
    st.dataframe(zs.round(3), hide_index=True)
    if kind == "Goalies":
        totals = [("GSAx, all shots", "gsax"), ("GSAx at 5v5", "gsax_5v5"), ("GSAx vs the power play", "gsax_pk"),
                  ("GSAx, low danger", "gsax_low"), ("GSAx, medium danger", "gsax_med"), ("GSAx, high danger", "gsax_high"),
                  ("Rebound control, goals saved", "reb_saved"), ("Win probability added, wins", "wpa")]
        st.caption("This season's totals, measured from the league average")
        st.dataframe(pd.DataFrame([(lab, round(float(p.get(col, np.nan)), 2)) for lab, col in totals],
                                  columns=["this season", "total"]), hide_index=True)
st.caption("pasttheeyetest.com")

In [ ]:
!pip -q install streamlit
import subprocess, time
from google.colab import output

APP = f'{DATA_DIR}/pte_dashboard.py'
subprocess.run(['pkill', '-f', 'streamlit run'])                 # stop a copy started earlier
subprocess.Popen(['streamlit', 'run', APP, '--server.port', '8501', '--server.headless', 'true',
                  '--server.enableCORS', 'false', '--server.enableXsrfProtection', 'false',
                  '--browser.gatherUsageStats', 'false', '--theme.base', 'dark'],
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(8)
output.serve_kernel_port_as_window(8501, anchor_text='Open the player dashboard')

In [ ]:
# Only if the link above does not load. Prints a temporary public address for the same app.
import re, subprocess
from google.colab import output   # (this cell is for Colab)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:8501'],
                          stderr=subprocess.PIPE, text=True)
for line in tunnel.stderr:
    found = re.search('https://[-a-z0-9]+[.]trycloudflare[.]com', line)
    if found:
        print('Dashboard:', found.group(0))
        break